# ATML PA3 · Task 3 — Test-Time Compute: Sampling, Selection and Adaptive Inference
**CS-6304 / EE-5102 · Danish Rasool (28100233)** · self-contained notebook (all code is written to disk by the cells below)

### Before running
1. **Add Input → the `ATML-PA3-Data` dataset** (expected at `/kaggle/input/atml-pa3-data`; any folder under
   `/kaggle/input` that contains the two model folders is found automatically).
2. Always run **from the top** (Run All, or Restart & Run All after a session restart): later cells use the
   SETTINGS cell's variables.
3. **Settings → Accelerator: GPU T4 ×2** (only GPU0 is used, for the 50-question live run), **Internet: On**
   (Qwen2.5-3B-Instruct download; MathVista metadata from the Hub only if the supplied files lack it).
4. **Save Version → Save & Run All (Commit)**.

### What happens (`AUTO`)
data join + alignment checks → dev/held-out splits (seed 6304) + 10 candidate orders per question → **live generation
starts in the background on GPU0** (single 3,072 / parallel 3×1,024 / refinement 1,024+2,048; FP16, batch 1,
pinned to its own CPU cores) → in parallel on the other cores: answer grouping + grading (math-verify / MathVista
rules, frozen with a manifest) → signals (log-prob, self-certainty, visual engagement, prefix-calibrated VisConf) →
all selection rules at N ∈ {1,2,4,8,16,32} → signal analysis → adaptive stopping (dev-tuned, frozen, held-out) →
compute allocation → live evaluation → qualitative cases → figures/tables. The last cell zips the outputs into
`/kaggle/working/results_task3.zip` and prints a **download link** (also when something failed part-way).

If `status.json` shows `live_generation: paused`, make this version's output a dataset, attach it and run again:
finished stages and live calls are kept and only the remaining calls are generated.


In [ ]:
# ============================ SETTINGS ============================
import os, time, json
from pathlib import Path
os.environ.setdefault("PA3_SESSION_START", str(time.time()))   # the session budget starts now
BUDGET_HOURS = 11.25                          # stop cleanly before Kaggle's 12 h limit
DATA_ROOT = "/kaggle/input/atml-pa3-data"     # searched recursively; /kaggle/input is the fallback
DATA_URL = ""        # only if the data is NOT attached: a Google Drive / direct .zip link to ATML-PA3-Data
DATA_HF_REPO = ""    # ... or a Hugging Face dataset repo id holding it
RUN_LIVE = True                               # Step 4 live generation on GPU0
RUN_CPU_TESTS = False                         # run the CPU test suite first (~1 min)
CODE_DIR = "/kaggle/working/ATML-PA3"   # fixed: the %%writefile cells below write here
OUTPUT_ROOT = "/kaggle/working/pa3_task3_outputs"
SETUP_ERROR = None

# %%writefile cannot create folders. Create them all now, and make %%writefile create its parent folder itself,
# so the code cells work even if a cell is run on its own or after a session restart (which empties /kaggle/working).
for _d in ["\\kaggle\\working\\ATML-PA3\\common\\pa3common", "\\kaggle\\working\\ATML-PA3\\task_3", "\\kaggle\\working\\ATML-PA3\\task_3\\configs", "\\kaggle\\working\\ATML-PA3\\task_3\\scripts", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\adaptive", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\analysis", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\answers", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\data", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\live", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\offline", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\report", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\select", "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\signals", "\\kaggle\\working\\ATML-PA3\\task_3\\tests"]:
    os.makedirs(_d, exist_ok=True)
try:
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def writefile(line, cell):
        import shlex
        args = shlex.split(line)
        append = "-a" in args or "--append" in args
        fn = os.path.expanduser([a for a in args if not a.startswith("-")][0])
        os.makedirs(os.path.dirname(fn) or ".", exist_ok=True)
        existed = os.path.exists(fn)
        with open(fn, "a" if append else "w", encoding="utf-8") as fh:
            fh.write(cell)
        print(("Appending to " if append else "Overwriting " if existed else "Writing ") + fn)
except Exception as _ex:  # noqa: BLE001  (not running under IPython)
    print("note: %%writefile not patched:", _ex)

In [ ]:
# ============================ WRITE THE CODE ============================
import os
for d in [
 "\\kaggle\\working\\ATML-PA3\\common\\pa3common",
 "\\kaggle\\working\\ATML-PA3\\task_3",
 "\\kaggle\\working\\ATML-PA3\\task_3\\configs",
 "\\kaggle\\working\\ATML-PA3\\task_3\\scripts",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\adaptive",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\analysis",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\answers",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\data",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\live",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\offline",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\report",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\select",
 "\\kaggle\\working\\ATML-PA3\\task_3\\src\\pa3ttc\\signals",
 "\\kaggle\\working\\ATML-PA3\\task_3\\tests"
]:
    os.makedirs(d, exist_ok=True)
print('writing 48 files under /kaggle/working/ATML-PA3')

In [ ]:
%%writefile /kaggle/working/ATML-PA3/common/pa3common/__init__.py
"""Shared, task-agnostic helpers for ATML PA3 (utils, Kaggle orchestration)."""


In [ ]:
%%writefile /kaggle/working/ATML-PA3/common/pa3common/kaggle.py
"""Task-agnostic Kaggle helpers: subprocess streaming, GPU job queue, session deadline, disk/scratch,
prior-session discovery and retries.  Used by task orchestrators (e.g. task_1/src/pa3/orchestrate.py)."""
from __future__ import annotations

import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Callable

NOISE = ("Loading weights", "it/s]", "s/it]", "Fetching", "Downloading", "Map:", "Generating ",
         "Requesting API", "Filter:", "Resolving data files")


# ------------------------------------------------------------------ session deadline
class Deadline:
    """Wall-clock budget for one Kaggle session (hard limit 12 h; outputs are saved only if the
    notebook finishes, so every step must stop before the deadline)."""

    def __init__(self, budget_hours: float, start: float | None = None):
        self.start = start or float(os.environ.get("PA3_SESSION_START", time.time()))
        self.end = self.start + budget_hours * 3600
        os.environ["PA3_SESSION_START"] = str(self.start)
        os.environ["PA3_DEADLINE_UNIX"] = str(self.end)

    def remaining(self) -> float:
        return self.end - time.time()

    def fits(self, seconds: float, margin: float = 600) -> bool:
        return self.remaining() > seconds + margin

    def __str__(self) -> str:
        return f"{self.remaining() / 3600:.2f} h left of session budget"


class StagePaused(RuntimeError):
    """Raised when the session budget is exhausted; rerun the notebook to continue."""


# ------------------------------------------------------------------ processes
def sh(cmd: list, log: Path, env: dict | None = None, timeout: float | None = None, echo: bool = True,
       cwd: str | Path | None = None) -> int:
    log = Path(log)
    log.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n$ {' '.join(map(str, cmd))}\n  (log: {log})", flush=True)
    t0 = time.time()
    with open(log, "a") as fh:
        fh.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} $ {' '.join(map(str, cmd))}\n")
        p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             env={**os.environ, **(env or {})}, cwd=str(cwd) if cwd else None, bufsize=1)
        try:
            for line in p.stdout:
                fh.write(line)
                fh.flush()
                if echo and not any(s in line for s in NOISE):
                    print(line, end="", flush=True)
                if timeout and time.time() - t0 > timeout:
                    p.kill()
                    print(f"!! timeout after {timeout:.0f}s", flush=True)
                    return -9
            return p.wait()
        finally:
            if p.poll() is None:
                p.kill()


def n_gpus() -> int:
    try:
        import torch

        return torch.cuda.device_count()
    except Exception:
        return 0


def run_on_gpus(jobs: list[tuple[str, list, Path]], env_extra: dict | None = None,
                poll: float = 10.0, kill_at: float | None = None) -> dict[str, int]:
    """Run independent single-GPU jobs concurrently, one per visible GPU (queue). Returns rc per job.
    Jobs still running at unix time `kill_at` are killed (rc -9) so the session can end cleanly."""
    slots = list(range(max(1, n_gpus())))
    todo = list(jobs)
    running: dict[int, tuple[str, subprocess.Popen, object, Path]] = {}
    rcs: dict[str, int] = {}
    while todo or running:
        while todo and len(running) < len(slots):
            gpu = next(s for s in slots if s not in running)
            name, cmd, log = todo.pop(0)
            log = Path(log)
            log.parent.mkdir(parents=True, exist_ok=True)
            fh = open(log, "a")
            fh.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} GPU{gpu} $ {' '.join(map(str, cmd))}\n")
            fh.flush()
            print(f"[gpu-queue] {name} -> GPU{gpu} (log {log})", flush=True)
            p = subprocess.Popen(list(map(str, cmd)), stdout=fh, stderr=subprocess.STDOUT,
                                 env={**os.environ, **(env_extra or {}), "CUDA_VISIBLE_DEVICES": str(gpu)})
            running[gpu] = (name, p, fh, log)
        time.sleep(poll)
        if kill_at is not None and time.time() > kill_at:
            for gpu, (name, p, fh, log) in list(running.items()):
                print(f"[gpu-queue] {name}: session deadline reached -> killing", flush=True)
                p.kill()
            todo = []
        for gpu in list(running):
            name, p, fh, log = running[gpu]
            if p.poll() is not None:
                fh.close()
                rcs[name] = p.returncode
                tail = [l for l in log.read_text(errors="ignore").splitlines() if l.startswith("[")][-5:]
                print(f"[gpu-queue] {name} finished rc={p.returncode}\n  " + "\n  ".join(tail), flush=True)
                del running[gpu]
    return rcs


def log_has(log: Path, *needles: str) -> bool:
    t = Path(log).read_text(errors="ignore") if Path(log).exists() else ""
    return any(n in t for n in needles)


def is_oom(log: Path) -> bool:
    return log_has(log, "OutOfMemoryError", "CUDA out of memory")


def is_host_oom(log: Path) -> bool:
    """All ranks SIGKILLed without a Python traceback = the kernel's OOM killer (host RAM)."""
    return log_has(log, "Signal 9 (SIGKILL)", "exitcode: -9") and not is_oom(log)


def retry(fn: Callable, tries: int = 4, wait: float = 20.0, what: str = ""):
    for i in range(tries):
        try:
            return fn()
        except Exception as ex:  # noqa: BLE001
            if i == tries - 1:
                raise
            print(f"[retry] {what} failed ({type(ex).__name__}: {str(ex)[:200]}); retrying in {wait:.0f}s",
                  flush=True)
            time.sleep(wait * (i + 1))


def pip_install(pkgs: list[str], tries: int = 3) -> None:
    def go():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *pkgs],
                       check=True)
    retry(go, tries, 15, "pip install")


# ------------------------------------------------------------------ disk / paths
def scratch_root(name: str = "pa3_scratch") -> Path:
    """Large, non-persisted disk for rolling recovery checkpoints (keeps /kaggle/working < 20 GB)."""
    env = os.environ.get("PA3_SCRATCH")
    cands = [Path(env)] if env else [Path("/kaggle/temp") / name, Path("/tmp") / name]
    for c in cands:
        if _is_tmpfs(c):
            print(f"[scratch] {c} is RAM-backed (tmpfs); skipping", flush=True)
            continue
        try:
            c.mkdir(parents=True, exist_ok=True)
            probe = c / ".w"
            probe.write_text("ok")
            probe.unlink()
            os.environ["PA3_SCRATCH"] = str(c)
            return c
        except OSError:
            continue
    raise RuntimeError("no writable scratch directory")


def _is_tmpfs(path: Path) -> bool:
    """True if `path` would live on a RAM-backed filesystem (writing GB-sized checkpoints there eats RAM)."""
    try:
        best, fstype = "", ""
        target = str(Path(path).resolve())
        for line in open("/proc/mounts"):
            parts = line.split()
            mnt, fs = parts[1], parts[2]
            if (target == mnt or target.startswith(mnt.rstrip("/") + "/")) and len(mnt) > len(best):
                best, fstype = mnt, fs
        return fstype in ("tmpfs", "ramfs")
    except OSError:
        return False


def free_gib(path: str | Path) -> float:
    return shutil.disk_usage(str(path)).free / 2**30


def tree_gib(root: str | Path) -> float:
    root = Path(root)
    if not root.exists():
        return 0.0
    return sum(f.stat().st_size for f in root.rglob("*") if f.is_file()) / 2**30


def disk_report(working: str | Path = "/kaggle/working", limit_gib: float = 19.0) -> float:
    w = Path(working) if Path(working).exists() else Path.cwd()
    used = tree_gib(w)
    msg = f"[disk] {w}: {used:.2f} GiB used (persisted-output limit ~20 GiB)"
    if os.environ.get("PA3_SCRATCH"):
        s = os.environ["PA3_SCRATCH"]
        msg += f" | scratch {s}: {tree_gib(s):.2f} GiB used, {free_gib(s):.0f} GiB free"
    print(msg, flush=True)
    if used > limit_gib:
        print("!! WARNING: /kaggle/working is close to the 20 GiB limit", flush=True)
    return used


def discover_prior_roots(dirname: str, base: str = "/kaggle/input", max_depth: int = 6) -> list[str]:
    """Earlier sessions' outputs attached as inputs: every directory called `dirname` under /kaggle/input."""
    b = Path(base)
    if not b.exists():
        return []
    return [str(p) for p in sorted(b.glob(f"**/{dirname}")) if p.is_dir() and len(p.relative_to(b).parts) <= max_depth]


def copytree_merge(src: Path, dst: Path, skip: tuple[str, ...] = ()) -> None:
    for f in Path(src).rglob("*"):
        rel = f.relative_to(src)
        if any(part in skip for part in rel.parts):
            continue
        t = Path(dst) / rel
        if f.is_dir():
            t.mkdir(parents=True, exist_ok=True)
        elif not t.exists():
            t.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, t)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/configs/task3.yaml
# PA3 Task 3 — test-time compute (approved design: see the design summary given in chat, 2026-10-31)
seed: 6304
data_root: /kaggle/input/atml-pa3-data      # searched recursively (symlinks followed); then data_fallback_root
data_fallback_root: /kaggle/input
data_hf_repo: null                          # optional: HF dataset repo id holding ATML-PA3-Data (downloaded if not found)
data_url: null                              # optional: download link (Google Drive / http .zip / .tar.gz) if not attached
n_rollouts: 32
n_questions_expected: 1000
n_dev: 200
n_live: 50
budgets: [1, 2, 4, 8, 16, 32]
n_orders: 10
n_boot: 1000
mv_timeout_s: 5                 # math-verify parse/verify timeout (seconds)
max_misaligned_frac: 0.01       # >1% rollouts with misaligned token lists -> data stage fails
# Optional column-name overrides if auto-detection picks the wrong column, e.g.
# columns: {math: {text: response, extracted: pred}, mathvista: {attn_image: image_attn}}
columns: {}
mathvista_metadata_hf: AI4Math/MathVista   # used only if the supplied files lack the metadata columns

adaptive:
  checks: [4, 8, 16]
  taus: [0.5, 0.625, 0.75, 0.875, 1.0]
  gammas: [0.0, 0.1, 0.25, 0.5]
  tolerance_pp: 1.0

live:
  model: Qwen/Qwen2.5-3B-Instruct
  revision: main
  dtype: float16
  system: "You are a helpful assistant."
  instruction: "Think step by step inside <think> </think> tags, then give your final answer inside \\boxed{}."
  review: "Review the solution above. Identify and correct any errors; if it is correct, retain its answer. Give your final answer inside \\boxed{}."
  caps: {single: 3072, parallel: 1024, revision: 2048}
  n_parallel: 3
  temperature: 0.7
  top_p: 0.8
  top_k: 20
  repetition_penalty: 1.0
  stop_margin_min: 20          # the worker pauses when less session time than this remains


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/__init__.py
"""PA3 Task 3 — test-time compute: sampling, selection, refinement and adaptive inference."""


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/adaptive/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/adaptive/controllers.py
"""Step 5: adaptive sampling simulated on the supplied pools (stopping between completed rollouts).

Rollouts are revealed in the saved orders; stopping is checked after n in {4, 8, 16}, otherwise all 32 are used.
Both controllers return the current self-consistency answer.
  agreement:     unique, non-NO_ANSWER leading answer with count / n >= tau
  quality-aware: agreement AND SC winner == RWC winner AND RWC margin >= gamma, where RWC uses self-certainty
                 (MATH) or VisConf recalibrated on the n revealed rollouts (MathVista); margin = (W1 - W2) / sum W
Features are precomputed per (question, order, n); a policy is then a vectorised first-true search.
"""
from __future__ import annotations

import numpy as np

from ..config import NO_ANSWER
from ..select.rules import rwc, vote, vote_margin

CHECKS_ALL = (4, 8, 16, 32)


def features(group, correct, T, qscore_fn, orders, checks=CHECKS_ALL) -> dict[str, np.ndarray]:
    """qscore_fn(q, k, o, n) -> quality scores of the first n candidates (candidate order)."""
    Q, K, R = orders.shape
    nC = len(checks)
    f = {k: np.zeros((Q, K, nC), t) for k, t in [("lead_frac", float), ("lead_unique", bool), ("lead_na", bool),
                                                   ("agree_rwc", bool), ("margin", float), ("sc_correct", bool),
                                                   ("cost", np.int64), ("sc_group", np.int32)]}
    for q in range(Q):
        for k in range(K):
            o = orders[q, k]
            g_o, c_o, T_o = group[q, o], correct[q, o], T[q, o]
            for ci, n in enumerate(checks):
                g = g_o[:n]
                win, idx, tot = vote(g)
                counts = sorted(tot.values(), reverse=True)
                rg, _, rtot = rwc(g, qscore_fn(q, k, o, n))
                f["lead_frac"][q, k, ci] = counts[0] / n
                f["lead_unique"][q, k, ci] = len(counts) == 1 or counts[0] > counts[1]
                f["lead_na"][q, k, ci] = win == NO_ANSWER
                f["agree_rwc"][q, k, ci] = rg == win
                f["margin"][q, k, ci] = vote_margin(rtot)
                f["sc_correct"][q, k, ci] = c_o[idx]
                f["sc_group"][q, k, ci] = win
                f["cost"][q, k, ci] = int(T_o[:n].sum())
    return f


def stop_index(f: dict, tau: float | None, gamma: float | None, family: str, n_checks: int = 3) -> np.ndarray:
    """Index into CHECKS_ALL where each (q, k) stops (n_checks = last index = 32 if nothing triggers).
    tau=None -> the always-32 fallback policy."""
    Q, K, _ = f["lead_frac"].shape
    if tau is None:
        return np.full((Q, K), n_checks, dtype=np.int64)
    cond = f["lead_unique"] & ~f["lead_na"] & (f["lead_frac"] >= tau - 1e-12)
    if family == "quality":
        cond = cond & f["agree_rwc"] & (f["margin"] >= gamma - 1e-12)
    cond = cond[:, :, :n_checks]
    any_ = cond.any(axis=2)
    first = cond.argmax(axis=2)
    return np.where(any_, first, n_checks)


def evaluate(f: dict, stop: np.ndarray, qmask: np.ndarray | None = None) -> dict:
    """Mean over orders within a question, then over questions."""
    take = lambda a: np.take_along_axis(a, stop[:, :, None], axis=2)[:, :, 0]  # noqa: E731
    acc_q = take(f["sc_correct"]).mean(1)
    cost_q = take(f["cost"]).mean(1)
    n_q = np.array(CHECKS_ALL)[stop].mean(1)
    if qmask is not None:
        acc_q, cost_q, n_q = acc_q[qmask], cost_q[qmask], n_q[qmask]
    return {"acc": float(acc_q.mean()), "cost": float(cost_q.mean()), "rollouts": float(n_q.mean()),
            "acc_q": acc_q, "cost_q": cost_q, "n_q": n_q}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/adaptive/tune.py
"""Development tuning, freezing and held-out evaluation of the stopping controllers.

Per controller family the candidates are its grid plus the always-32 fallback.  Selected = lowest mean dev token
cost among settings whose dev accuracy >= fixed-32 SC dev accuracy - 1 pp (averaged over the ten orders); ties ->
higher accuracy, then smaller tau, then smaller gamma.  The choice is written to frozen.json BEFORE the held-out
questions are evaluated; the held-out evaluation reads that file.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..config import odir
from ..io import df_to_md, read_json, write_json
from ..offline.bootstrap import boot_idx, paired
from ..offline.runner import load_inputs
from .controllers import CHECKS_ALL, evaluate, features, stop_index


def _qscore_fn(ds, sc, vc):
    if ds == "math":
        K = sc["K"]
        return lambda q, k, o, n: K[q, o[:n]]
    return lambda q, k, o, n: vc[f"Z_{n}"][q, k]


def grid(cfg) -> list[dict]:
    a = cfg["adaptive"]
    g = [{"family": "agreement", "tau": t, "gamma": None} for t in a["taus"]]
    g += [{"family": "agreement", "tau": None, "gamma": None}]
    g += [{"family": "quality", "tau": t, "gamma": gm} for t in a["taus"] for gm in a["gammas"]]
    g += [{"family": "quality", "tau": None, "gamma": None}]
    return g


def _key(r):
    inf = float("inf")
    return (round(r["cost"], 9), -round(r["acc"], 12), inf if r["tau"] is None else r["tau"],
            inf if r["gamma"] is None else r["gamma"])


def select_setting(rows: list[dict], target: float) -> dict:
    ok = [r for r in rows if r["acc"] >= target - 1e-12]
    return sorted(ok, key=_key)[0]


def run_dataset(ds: str, cfg: dict) -> dict:
    pool, group, correct, sc, vc, orders, split = load_inputs(ds)
    qpos = {q: i for i, q in enumerate(pool.qids)}
    dev = np.zeros(pool.Q, bool)
    dev[[qpos[q] for q in split["dev"]]] = True
    ev = np.zeros(pool.Q, bool)
    ev[[qpos[q] for q in split["eval"]]] = True
    f = features(group, correct, pool.T, _qscore_fn(ds, sc, vc), orders)
    d = odir("adaptive", ds)
    np.savez(d / "features.npz", **f)
    tol = cfg["adaptive"]["tolerance_pp"] / 100.0
    sc32_dev = evaluate(f, stop_index(f, None, None, "agreement"), dev)["acc"]
    rows = []
    for s in grid(cfg):
        r = evaluate(f, stop_index(f, s["tau"], s["gamma"], s["family"]), dev)
        rows.append({**s, "acc": r["acc"], "cost": r["cost"], "rollouts": r["rollouts"]})
    pd.DataFrame(rows).to_csv(d / "dev_grid.csv", index=False)
    frozen = {"sc32_dev_acc": sc32_dev, "target_acc": sc32_dev - tol, "rule": __doc__.strip()}
    for fam in ("agreement", "quality"):
        frozen[fam] = select_setting([r for r in rows if r["family"] == fam], sc32_dev - tol)
    write_json(d / "frozen.json", frozen)
    return heldout(ds, cfg, f, ev, pool)


def heldout(ds: str, cfg: dict, f: dict, ev: np.ndarray, pool) -> dict:
    d = odir("adaptive", ds)
    frozen = read_json(d / "frozen.json")            # evaluation uses only the frozen file
    idx = boot_idx(int(ev.sum()), cfg["n_boot"], cfg["seed"])
    pqo = pd.read_parquet(odir("offline", ds) / "per_question.parquet")
    pqo = pqo[(pqo["method"] == "sc") & (pqo["split"] == "eval")]
    qorder = [pool.qids[i] for i in np.where(ev)[0]]
    fixed = {}
    for N in cfg["budgets"]:
        x = pqo[pqo["N"] == N].set_index("qid").loc[qorder]
        fixed[N] = {"acc": float(x["acc"].mean()), "cost": float(x["cost"].mean()),
                    "acc_q": x["acc"].to_numpy(), "cost_q": x["cost"].to_numpy()}
    res = {"frozen": {k: frozen[k] for k in ("agreement", "quality")}, "fixed_sc": {
        N: {"acc": v["acc"], "cost": v["cost"]} for N, v in fixed.items()}}
    full_stop = np.full(f["lead_frac"].shape[:2], 3)
    sc32_q = evaluate(f, full_stop, ev)["acc_q"]
    sc32_correct = f["sc_correct"][:, :, 3]
    p_correct = None
    rows_tbl = [{"policy": f"fixed SC N={N}", "acc": v["acc"], "cost": v["cost"], "rollouts": N} for N, v in fixed.items()]
    cases = []
    for fam in ("agreement", "quality"):
        s = frozen[fam]
        stop = stop_index(f, s["tau"], s["gamma"], fam)
        r = evaluate(f, stop, ev)
        st = np.array(CHECKS_ALL)[stop][ev]                              # [Qe, K]
        dist = {int(n): float((st == n).mean()) for n in CHECKS_ALL}
        wrong = ~np.take_along_axis(f["sc_correct"], stop[:, :, None], 2)[:, :, 0]
        early = stop < 3
        ie = (early & wrong)[ev]
        rec = (early & wrong & sc32_correct)[ev]
        # nearest-cost fixed budget and linear interpolation of the fixed-N SC curve at the adaptive cost
        costs = np.array([fixed[N]["cost"] for N in cfg["budgets"]])
        accs = np.array([fixed[N]["acc"] for N in cfg["budgets"]])
        Nn = cfg["budgets"][int(np.argmin(np.abs(costs - r["cost"])))]
        interp = float(np.interp(r["cost"], costs, accs))
        res[fam] = {
            "setting": s, "acc": r["acc"], "cost": r["cost"], "rollouts": r["rollouts"], "stop_distribution": dist,
            "incorrect_early_stop_rate": float(ie.mean()), "recoverable_by_fixed32_rate": float(rec.mean()),
            "frac_early_stops": float(early[ev].mean()),
            "acc_of_early_stops": float((~wrong & early)[ev].sum() / max(1, early[ev].sum())),
            "nearest_cost_fixed_N": Nn, "fixed_sc_acc_interpolated_at_same_cost": interp,
            "vs_fixed32": paired(r["acc_q"], sc32_q, idx),
            "vs_nearest_cost_fixed": paired(r["acc_q"], fixed[Nn]["acc_q"], idx),
            "cost_saving_vs_fixed32": 1 - r["cost"] / fixed[32]["cost"],
        }
        rows_tbl.append({"policy": f"{fam} (tau={s['tau']}, gamma={s['gamma']})", "acc": r["acc"], "cost": r["cost"],
                         "rollouts": r["rollouts"]})
        # per-question retrospective table: stopping vs pool correct-fraction and agreement
        if p_correct is None:
            lab = np.load(odir("answers", ds) / "labels.npz")
            p_correct = lab["correct"].mean(1)
        evq = np.where(ev)[0]
        tab = pd.DataFrame({"qid": [pool.qids[q] for q in evq], "p_correct_pool": p_correct[evq],
                            "mean_stop_n": r["n_q"], "acc": r["acc_q"], "acc_sc32": sc32_q,
                            "lead_frac_at_4": f["lead_frac"][evq, :, 0].mean(1),
                            "incorrect_early_frac": ie.mean(1), "recoverable_frac": rec.mean(1)})
        tab.to_csv(d / f"heldout_per_question_{fam}.csv", index=False)
        bins = pd.cut(tab["p_correct_pool"], [-0.001, 0.0, 0.25, 0.5, 0.75, 0.999, 1.0],
                      labels=["0", "(0,.25]", "(.25,.5]", "(.5,.75]", "(.75,1)", "1"])
        tab.groupby(bins, observed=False)[["mean_stop_n", "acc", "acc_sc32", "incorrect_early_frac",
                                           "recoverable_frac"]].agg(["mean"]).assign(
            n=tab.groupby(bins, observed=False).size()).to_csv(d / f"by_pool_correct_fraction_{fam}.csv")
        # example cases: incorrect early stops that fixed-32 would recover (order 0 first)
        for qi_local, q in enumerate(evq):
            for k in range(f["lead_frac"].shape[1]):
                if early[q, k] and wrong[q, k] and sc32_correct[q, k]:
                    cases.append({"family": fam, "qid": pool.qids[q], "order": k,
                                  "stop_n": int(CHECKS_ALL[stop[q, k]]),
                                  "stopped_group": int(f["sc_group"][q, k, stop[q, k]]),
                                  "fixed32_group": int(f["sc_group"][q, k, 3]),
                                  "p_correct_pool": float(p_correct[q])})
    pd.DataFrame(cases).to_csv(d / "recoverable_early_stops.csv", index=False)
    tbl = pd.DataFrame(rows_tbl)
    (d / "heldout.md").write_text(f"## {ds}: adaptive stopping on held-out questions (frozen dev settings)\n\n"
                                  + df_to_md(tbl) + "\n")
    tbl.to_csv(d / "heldout_table.csv", index=False)
    write_json(d / "heldout.json", res)
    return res


def run(cfg: dict) -> dict:
    out = {}
    for ds in ("math", "mathvista"):
        out[ds] = run_dataset(ds, cfg)
        print(f"[adaptive] {ds}: frozen {out[ds]['frozen']}", flush=True)
    return out


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/analysis/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/analysis/compute_alloc.py
"""Step 6: compute allocation — does the n-th sample add a new answer, the first correct answer, or a repeat?
Held-out questions, averaged over the 10 orders then over questions.  Also the coverage-vs-selection gap per
method and budget (oracle coverage minus achieved accuracy)."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..config import NO_ANSWER, odir
from ..io import df_to_md
from ..offline.runner import LABEL, load_inputs


def marginal(ds: str) -> pd.DataFrame:
    pool, group, correct, sc, vc, orders, split = load_inputs(ds)
    qpos = {q: i for i, q in enumerate(pool.qids)}
    ev = [qpos[q] for q in split["eval"]]
    Q, K, R = orders.shape
    new = np.zeros((len(ev), K, R))
    new_ans = np.zeros_like(new)
    first_corr = np.zeros_like(new)
    for i, q in enumerate(ev):
        for k in range(K):
            seen, have_c = set(), False
            for n, r in enumerate(orders[q, k]):
                g = int(group[q, r])
                if g not in seen:
                    new[i, k, n] = 1
                    new_ans[i, k, n] = g != NO_ANSWER
                    seen.add(g)
                if correct[q, r] and not have_c:
                    first_corr[i, k, n] = 1
                    have_c = True
    df = pd.DataFrame({"n": np.arange(1, R + 1), "p_new_group": new.mean(1).mean(0),
                       "p_new_answered_group": new_ans.mean(1).mean(0),
                       "p_first_correct_appears": first_corr.mean(1).mean(0)})
    df["p_repeat"] = 1 - df["p_new_group"]
    df["coverage_cum"] = df["p_first_correct_appears"].cumsum()
    return df


def gap_table(ds: str) -> pd.DataFrame:
    s = pd.read_csv(odir("offline", ds) / "summary_eval.csv")
    s["gap"] = s["oracle"] - s["acc"]
    t = s.pivot_table(index="method", columns="N", values="gap")
    t = t.loc[[m for m in LABEL if m in t.index]]
    t.index = [LABEL[m] for m in t.index]
    t.columns = [f"N={c}" for c in t.columns]
    return t


def run(cfg: dict) -> dict:
    out = {}
    for ds in ("math", "mathvista"):
        d = odir("compute", ds)
        m = marginal(ds)
        m.to_csv(d / "marginal_samples.csv", index=False)
        g = gap_table(ds)
        g.to_csv(d / "selection_gap.csv")
        (d / "compute.md").write_text(
            f"## {ds}: what the n-th sample adds (held-out)\n\n"
            + df_to_md(m[m["n"].isin([1, 2, 3, 4, 8, 16, 32])]) + "\n## oracle coverage - accuracy (selection gap)\n\n"
            + df_to_md(g, index=True))
        out[ds] = {"p_new_group_at_32": float(m["p_new_group"].iloc[-1])}
    return out


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/analysis/conf_vision.py
"""Step 3b (MathVista): confidence-vision relationship and the tokenwise vs trajectory-level fusion check.

Token level, per rollout: Pearson(c_t, v_t) with c_t calibrated on the full 32-rollout pool of its question, and
Pearson(raw KL_t, v_t); summarised per question (mean over rollouts) and across held-out questions.
Rollout level: Spearman(C_i, V_i) across the 32 rollouts of a question.
Identity check: Z_tok - Z = Cov_t(c_t, v_t) (population covariance) for rollouts where v_t is always defined.
Illustrative trajectories: fixed rule — held-out questions bucketed by (VisConf+RWC vs self-consistency at N=32,
order 0); 2 per bucket drawn with default_rng(6304) from the sorted question IDs; per question the highest- and
lowest-Z answered rollouts; rolling means (window 16) of c_t and v_t over generation position.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..config import odir
from ..io import write_json
from ..offline.runner import load_inputs
from ..signals.visconf import token_c
from .signal_assoc import spearman


def _pearson(x, y) -> float:
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def _summ(x: np.ndarray) -> dict:
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {"n": 0}
    return {"n": int(x.size), "mean": float(x.mean()), "median": float(np.median(x)),
            "q25": float(np.percentile(x, 25)), "q75": float(np.percentile(x, 75)), "frac_positive": float((x > 0).mean())}


def run(cfg: dict) -> dict:
    ds = "mathvista"
    pool, group, correct, sc, vc, orders, split = load_inputs(ds)
    R = pool.R
    qpos = {q: i for i, q in enumerate(pool.qids)}
    ev = [qpos[q] for q in split["eval"]]
    zpos = np.full((pool.Q, R), np.nan)
    ztpos = zpos.copy()
    for q in range(pool.Q):
        zpos[q, orders[q, 0]] = vc["Z_32"][q, 0]
        ztpos[q, orders[q, 0]] = vc["Ztok_32"][q, 0]
    rows, ident = [], []
    for q in ev:
        a, d = vc["a_32"][q, 0], vc["d_32"][q, 0]
        for r in range(R):
            s = pool.seg(q, r)
            k, v = pool.kl[s].astype(np.float64), pool.v[s].astype(np.float64)
            ok = np.isfinite(v)
            c = token_c(k, a, d)
            rho_cv = _pearson(c[ok], v[ok])
            rho_kv = _pearson(k[ok], v[ok])
            cov = float(np.mean(c * v) - c.mean() * v.mean()) if ok.all() and len(c) else np.nan
            if ok.all() and len(c) and np.isfinite(ztpos[q, r]):
                ident.append(abs((ztpos[q, r] - zpos[q, r]) - cov))
            rows.append({"qid": pool.qids[q], "rpos": r, "T": len(k), "correct": bool(correct[q, r]),
                         "rho_c_v": rho_cv, "rho_kl_v": rho_kv, "cov_c_v": cov, "C": np.nan, "V": sc["V"][q, r],
                         "Z": zpos[q, r], "Ztok": ztpos[q, r]})
    df = pd.DataFrame(rows)
    df["C"] = df["Z"] / df["V"]
    out = odir("analysis", ds)
    df.to_parquet(out / "conf_vision_per_rollout.parquet", index=False)
    perq = df.groupby("qid")[["rho_c_v", "rho_kl_v", "cov_c_v"]].mean()
    roll_level = np.array([spearman(sc["V"][q], df[df["qid"] == pool.qids[q]]["C"].to_numpy()) for q in ev])
    summary = {
        "token_level_rho_c_v_per_question": _summ(perq["rho_c_v"].to_numpy()),
        "token_level_rho_kl_v_per_question": _summ(perq["rho_kl_v"].to_numpy()),
        "token_level_rho_c_v_correct_rollouts": _summ(df[df["correct"]]["rho_c_v"].to_numpy()),
        "token_level_rho_c_v_incorrect_rollouts": _summ(df[~df["correct"]]["rho_c_v"].to_numpy()),
        "cov_c_v_per_question": _summ(perq["cov_c_v"].to_numpy()),
        "rollout_level_spearman_C_V": _summ(roll_level),
        "identity_check": {"n_rollouts": len(ident), "max_abs_error": float(max(ident)) if ident else None},
        "mean_Ztok_minus_Z": float(np.nanmean(df["Ztok"] - df["Z"])),
    }
    write_json(out / "conf_vision_summary.json", summary)
    summary["trajectories"] = trajectories(pool, correct, sc, vc, orders, split, cfg, zpos)
    return summary


def trajectories(pool, correct, sc, vc, orders, split, cfg, zpos) -> dict:
    po = pd.read_parquet(odir("offline", "mathvista") / "per_order.parquet",
                         filters=[("N", "==", 32), ("order", "==", 0)])
    po = po[po["method"].isin(["sc", "visconf_rwc"]) & (po["split"] == "eval")]
    w = po.pivot_table(index="qid", columns="method", values=["correct", "group"], observed=True)
    buckets = {
        "agree_both_correct": w[(w[("group", "sc")] == w[("group", "visconf_rwc")]) & (w[("correct", "sc")] == 1)],
        "disagree_visconf_correct": w[(w[("correct", "visconf_rwc")] == 1) & (w[("correct", "sc")] == 0)],
        "disagree_sc_correct": w[(w[("correct", "visconf_rwc")] == 0) & (w[("correct", "sc")] == 1)],
        "agree_both_wrong": w[(w[("group", "sc")] == w[("group", "visconf_rwc")]) & (w[("correct", "sc")] == 0)],
    }
    rng = np.random.default_rng(cfg["seed"])
    qpos = {q: i for i, q in enumerate(pool.qids)}
    picked: dict = {}
    series = []
    for name, b in buckets.items():
        ids = sorted(b.index.tolist())
        sel = rng.choice(ids, size=min(2, len(ids)), replace=False).tolist() if ids else []
        picked[name] = {"n_in_bucket": len(ids), "selected": sel}
        for qid in sel:
            q = qpos[qid]
            z = zpos[q].copy()
            z[pool.T[q] == 0] = np.nan
            if not np.isfinite(z).any():
                continue
            a, d = vc["a_32"][q, 0], vc["d_32"][q, 0]
            for tag, r in (("highest_Z", int(np.nanargmax(z))), ("lowest_Z", int(np.nanargmin(z)))):
                s = pool.seg(q, r)
                c = token_c(pool.kl[s], a, d)
                v = pool.v[s].astype(np.float64)
                win = 16
                cr = pd.Series(c).rolling(win, min_periods=1).mean().to_numpy()
                vr = pd.Series(v).rolling(win, min_periods=1).mean().to_numpy()
                step = max(1, len(c) // 200)
                series.append({"bucket": name, "qid": qid, "which": tag, "rpos": r,
                               "rollout_id": pool.rollout_ids[q, r], "correct": bool(correct[q, r]),
                               "Z": float(z[r]), "T": len(c), "c_roll": cr[::step].tolist(),
                               "v_roll": vr[::step].tolist(), "step": step})
    write_json(odir("analysis", "mathvista") / "trajectories.json", {"rule": __doc__.split("Illustrative")[1].strip(),
                                                                      "picked": picked, "series": series})
    return picked


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/analysis/signal_assoc.py
"""Step 3a: within-question associations at N = 32 between rollout length, scores and correctness.

For each held-out question: Spearman(signal, correct), Spearman(signal, length); correctness in the signal's top
quartile (8 best rollouts) minus its bottom quartile (8 worst; ties broken by rollout position, NaN = -inf).
Averages are reported over mixed-correctness questions (main, approved decision 9) and over all questions.
`neg_length` (shorter = better) is the length-only control.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import rankdata

from ..config import odir
from ..io import df_to_md
from ..offline.runner import load_inputs


def spearman(x: np.ndarray, y: np.ndarray) -> float:
    x = np.where(np.isnan(x), -np.inf, x)
    if np.all(x == x[0]) or np.all(y == y[0]):
        return np.nan
    rx, ry = rankdata(x), rankdata(y)
    return float(np.corrcoef(rx, ry)[0, 1])


def quartile_diff(score: np.ndarray, correct: np.ndarray) -> float:
    s = np.where(np.isnan(score), -np.inf, score)
    R = len(s)
    q = R // 4
    order = np.lexsort((np.arange(R), -s))        # best first; ties -> lower rollout position first
    return float(correct[order[:q]].mean() - correct[order[-q:]].mean())


def signals_at_32(ds, sc, vc, orders) -> dict[str, np.ndarray]:
    sig = {"neg_length": -sc["T"], "logp_L": sc["L"], "selfcert_K": sc["K"]}
    if ds == "mathvista":
        sig["vision_V"] = sc["V"]
        Q = orders.shape[0]
        for key, name in (("Z", "visconf_Z"), ("Ztok", "tokfuse_Ztok")):
            out = np.full_like(sc["V"], np.nan)
            for q in range(Q):
                out[q, orders[q, 0]] = vc[f"{key}_32"][q, 0]
            sig[name] = out
    return sig


def run_dataset(ds: str, cfg: dict) -> dict:
    pool, group, correct, sc, vc, orders, split = load_inputs(ds)
    qpos = {q: i for i, q in enumerate(pool.qids)}
    ev = np.array([qpos[q] for q in split["eval"]])
    corr = correct.astype(float)
    T = sc["T"]
    sig = signals_at_32(ds, sc, vc, orders)
    mixed = np.array([0 < corr[q].mean() < 1 for q in ev])
    rows = []
    for name, s in sig.items():
        sp_c = np.array([spearman(s[q], corr[q]) for q in ev])
        sp_t = np.array([spearman(s[q], T[q]) for q in ev])
        qd = np.array([quartile_diff(s[q], corr[q]) for q in ev])
        rows.append({"signal": name, "spearman_vs_correct_mixed": np.nanmean(sp_c[mixed]),
                     "n_mixed": int(mixed.sum()), "spearman_vs_length": np.nanmean(sp_t),
                     "top_minus_bottom_quartile_mixed": qd[mixed].mean(),
                     "top_minus_bottom_quartile_all": qd.mean(), "n_all": len(ev)})
    assoc = pd.DataFrame(rows)
    # length of correct vs incorrect rollouts within mixed questions
    lc, li = [], []
    for q in ev[mixed]:
        lc.append(T[q][correct[q]].mean())
        li.append(T[q][~correct[q]].mean())
    extra = {"mean_len_correct_mixed": float(np.mean(lc)) if lc else np.nan,
             "mean_len_incorrect_mixed": float(np.mean(li)) if li else np.nan,
             "frac_questions_mixed": float(mixed.mean()),
             "frac_all_correct": float(np.mean([corr[q].mean() == 1 for q in ev])),
             "frac_all_wrong": float(np.mean([corr[q].mean() == 0 for q in ev]))}
    d = odir("analysis", ds)
    assoc.to_csv(d / "associations.csv", index=False)
    (d / "associations.md").write_text(f"## {ds}: within-question associations at N=32 (held-out)\n\n"
                                       + df_to_md(assoc) + "\n" + "\n".join(f"- {k}: {v:.4f}" for k, v in extra.items()) + "\n")
    return {"associations": rows, **extra}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/answers/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/answers/extract.py
"""Last \\boxed{...} answer with balanced braces (the Hendrycks-MATH `last_boxed_only_string` rule, plus the
`\\boxed 5` space form).  An unclosed last box (e.g. cut off by the token cap) gives None."""
from __future__ import annotations


def last_boxed(text: str | None) -> str | None:
    if not text:
        return None
    idx = text.rfind("\\boxed")
    cmd = "\\boxed"
    if idx < 0:
        idx = text.rfind("\\fbox")
        cmd = "\\fbox"
        if idx < 0:
            return None
    i = idx + len(cmd)
    while i < len(text) and text[i] in " \t":
        i += 1
    if i < len(text) and text[i] == "{":
        depth = 0
        for j in range(i, len(text)):
            c = text[j]
            if c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    ans = text[i + 1:j].strip()
                    return ans if ans else None
        return None
    if i > idx + len(cmd):   # "\boxed 5" form: up to whitespace or '$'
        j = i
        while j < len(text) and text[j] not in " \t\n$":
            j += 1
        ans = text[i:j].strip()
        return ans if ans else None
    return None


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/answers/math_equiv.py
"""MATH answer groups and grades with math-verify (gold-free grouping; grading uses the same rule).

Grouping, per question, rollouts in rollout-ID order: an empty/missing extraction -> NO_ANSWER (-1).  Identical
normalised strings share a group without calling math-verify.  Otherwise the answer is parsed (as `$s$`, then
`\\boxed{s}`, then raw); it joins the first earlier group whose representative r satisfies verify(r, answer),
else it starts a new group.  A non-empty answer that math-verify cannot parse forms a string group keyed by its
lower-cased normalised string (approved decision 4).  Grade of a group = verify(gold, representative) (decision
3); every rollout is also graded on its own so the disagreement rate can be logged.
"""
from __future__ import annotations

import re

NO_ANSWER = -1


def norm_str(s) -> str | None:
    if s is None:
        return None
    if isinstance(s, float) and s != s:
        return None
    s = str(s).strip()
    while len(s) >= 2 and s[0] == "$" and s[-1] == "$":
        s = s[1:-1].strip()
    s = re.sub(r"\s+", " ", s)
    if not s or s.lower() in ("none", "nan", "null"):
        return None
    return s


def parse_ans(s: str, timeout: int = 5) -> list:
    from math_verify import parse

    for cand in (f"${s}$", f"\\boxed{{{s}}}", s):
        try:
            p = parse(cand, parsing_timeout=timeout)
        except Exception:  # noqa: BLE001
            p = []
        if p:
            return p
    return []


def _verify(gold, target, timeout: int = 5) -> bool:
    from math_verify import verify

    if not gold or not target:
        return False
    try:
        return bool(verify(gold, target, timeout_seconds=timeout))
    except Exception:  # noqa: BLE001
        return False


def group_and_grade(answers: list, gold, timeout: int = 5) -> dict:
    """answers: extracted strings in rollout order.  Returns per-rollout group ids, per-group representative /
    kind / grade, and per-rollout individual grades."""
    gold_n = norm_str(gold)
    gold_p = parse_ans(gold_n, timeout) if gold_n else []
    groups: list[int] = []
    reps: list[dict] = []          # {"answer", "kind", "parsed"|"key"}
    by_str: dict[str, int] = {}
    parsed_of: dict[str, list] = {}
    for a in answers:
        n = norm_str(a)
        if n is None:
            groups.append(NO_ANSWER)
            continue
        if n in by_str:
            groups.append(by_str[n])
            continue
        p = parse_ans(n, timeout)
        parsed_of[n] = p
        g = None
        if p:
            for gi, rep in enumerate(reps):
                if rep["kind"] == "expr" and _verify(rep["parsed"], p, timeout):
                    g = gi
                    break
            if g is None:
                reps.append({"answer": n, "kind": "expr", "parsed": p})
                g = len(reps) - 1
        else:
            key = "STR:" + n.lower()
            g = next((gi for gi, rep in enumerate(reps) if rep["kind"] == "str" and rep["key"] == key), None)
            if g is None:
                reps.append({"answer": n, "kind": "str", "key": key})
                g = len(reps) - 1
        by_str[n] = g
        groups.append(g)

    def grade(kind, parsed=None, s=None) -> bool:
        if gold_n is None:
            return False
        if kind == "expr":
            return _verify(gold_p, parsed, timeout) if gold_p else False
        return s is not None and s.lower() == gold_n.lower()

    rep_correct = [grade(r["kind"], r.get("parsed"), r["answer"]) for r in reps]
    indiv = []
    for a, g in zip(answers, groups):
        if g == NO_ANSWER:
            indiv.append(False)
            continue
        n = norm_str(a)
        p = parsed_of.get(n)
        if p is None:
            p = parse_ans(n, timeout)
            parsed_of[n] = p
        indiv.append(grade("expr", p) if p else grade("str", s=n))
    return {"groups": groups, "group_answer": [r["answer"] for r in reps], "group_kind": [r["kind"] for r in reps],
            "group_correct": rep_correct, "indiv_correct": indiv, "gold_parsed": bool(gold_p)}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/answers/mathvista.py
"""MathVista answer normalisation (official `normalize_extracted_answer` rules) and grouping.

multi_choice: "(X)" -> X; a letter within range -> that option; otherwise the closest choice by Levenshtein
  distance (official `get_most_similar`, approved decision 5).  Answers are represented as option letters.
integer: str(int(float(x)));  float: str(round(float(x), precision));  list: str(x) without spaces;
text: stripped.  Missing extraction -> NO_ANSWER (never fuzzy-mapped).  A non-empty answer that fails the
numeric conversion forms a string group graded by string equality with the gold (decision 4).
Normalised-value equality is transitive, so group grade == individual grade.
"""
from __future__ import annotations

import json
import re

import numpy as np

NO_ANSWER = -1


def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def as_list(x) -> list[str]:
    if x is None:
        return []
    if isinstance(x, float) and np.isnan(x):
        return []
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            v = json.loads(s)
            if isinstance(v, list):
                return [str(c) for c in v]
        except json.JSONDecodeError:
            pass
        return [s]
    return [str(c) for c in x]


def _prec(p) -> int | None:
    try:
        p = float(p)
        return None if np.isnan(p) else int(p)
    except (TypeError, ValueError):
        return None


def normalize(extraction, meta: dict) -> tuple[str | None, str]:
    """-> (group key or None for NO_ANSWER, kind)."""
    if extraction is None or (isinstance(extraction, float) and np.isnan(extraction)):
        return None, "na"
    e = str(extraction).strip()
    if not e or e.lower() in ("none", "nan", "null"):
        return None, "na"
    qt, at = str(meta.get("question_type") or ""), str(meta.get("answer_type") or "")
    if qt == "multi_choice":
        choices = as_list(meta.get("choices"))
        letter = re.findall(r"\(([a-zA-Z])\)", e)
        if letter:
            e = letter[0].upper()
        letters = [chr(ord("A") + i) for i in range(len(choices))]
        if e in letters:
            return "MC:" + e, "mc_letter"
        if not choices:
            return "STR:" + e.lower(), "str"
        d = [levenshtein(e, c) for c in choices]
        return "MC:" + letters[int(np.argmin(d))], ("mc_exact" if min(d) == 0 else "mc_fuzzy")
    try:
        if at == "integer":
            return "NUM:" + str(int(float(e))), "num"
        if at == "float":
            p = _prec(meta.get("precision"))
            return "NUM:" + str(round(float(e), p if p is not None else 2)), "num"
    except (ValueError, OverflowError):
        return "STR:" + e.lower(), "str"
    if at == "list":
        return "LIST:" + re.sub(r"\s+", "", e), "list"
    return "STR:" + e.lower(), "str"


def gold_key(gold, meta: dict) -> str | None:
    if str(meta.get("question_type") or "") == "multi_choice":
        choices = as_list(meta.get("choices"))
        g = str(gold).strip() if gold is not None else ""
        if g in choices:
            return "MC:" + chr(ord("A") + choices.index(g))
        return normalize(g, meta)[0]
    return normalize(gold, meta)[0]


def group_and_grade(answers: list, gold, meta: dict) -> dict:
    gk = gold_key(gold, meta)
    groups, keys, kinds, rep_answer = [], [], [], []
    seen: dict[str, int] = {}
    rkinds = []
    for a in answers:
        k, kind = normalize(a, meta)
        rkinds.append(kind)
        if k is None:
            groups.append(NO_ANSWER)
            continue
        if k not in seen:
            seen[k] = len(keys)
            keys.append(k)
            rep_answer.append(str(a).strip())
            kinds.append(kind)
        groups.append(seen[k])
    corr = [gk is not None and k == gk for k in keys]
    indiv = [False if g == NO_ANSWER else corr[g] for g in groups]
    return {"groups": groups, "group_answer": [k.split(":", 1)[1] for k in keys], "group_kind": kinds,
            "group_correct": corr, "indiv_correct": indiv, "rollout_kind": rkinds, "gold_key": gk}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/answers/scorer.py
"""Scorer stage: answer groups + correctness labels for every stored rollout, frozen with a manifest.

Writes answers/<ds>/labels.npz (group [Q,R], correct [Q,R] group grade, indiv [Q,R]), rollout_groups.parquet,
groups.parquet and answers/scorer_manifest.json (math-verify version, source hashes, counts, extractor check).
Later stages only read these files ("fix the scorer before final evaluation").
"""
from __future__ import annotations

import hashlib
import importlib.metadata as md
import multiprocessing as mp
import os
from pathlib import Path

import numpy as np
import pandas as pd

from ..config import NO_ANSWER, odir
from ..data.load import Pool
from ..io import write_json
from . import math_equiv, mathvista
from .extract import last_boxed

_TIMEOUT = 5


def _math_q(args):
    answers, gold = args
    return math_equiv.group_and_grade(answers, gold, _TIMEOUT)


def _init(timeout):
    global _TIMEOUT
    _TIMEOUT = timeout


def _src_hash() -> str:
    h = hashlib.sha256()
    for f in sorted(Path(__file__).parent.glob("*.py")):
        h.update(f.read_bytes())
    return h.hexdigest()[:16]


def _version(pkg: str) -> str | None:
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None


def mathvista_meta(pool: Pool, cfg: dict) -> tuple[list[dict], str]:
    q = pool.questions
    need = ["question_type", "answer_type", "choices", "precision"]
    if all(c in q.columns for c in need[:2]):
        return [{c: r.get(c) for c in need} for r in q.to_dict("records")], "supplied files"
    from datasets import load_dataset  # noqa: PLC0415

    from pa3common.kaggle import retry  # noqa: PLC0415

    d = retry(lambda: load_dataset(cfg["mathvista_metadata_hf"], split="testmini"), 4, 20, "MathVista metadata")
    keep = [c for c in ("pid", "question_type", "answer_type", "choices", "precision", "answer") if c in d.column_names]
    d = d.select_columns(keep).to_pandas()
    d["pid"] = d["pid"].astype(str)
    by = d.set_index("pid").to_dict("index")
    miss = [x for x in pool.qids if x not in by]
    if miss:
        raise KeyError(f"{len(miss)} MathVista question IDs not found as HF pid, e.g. {miss[:5]}")
    return [{c: by[x].get(c) for c in need} for x in pool.qids], f"HF {cfg['mathvista_metadata_hf']} testmini"


def score_dataset(ds: str, cfg: dict, workers: int) -> dict:
    pool = Pool(ds)
    Q, R = pool.Q, pool.R
    ext = pool.meta["extracted"].to_numpy().reshape(Q, R)
    golds = pool.questions["gold"].tolist()
    if ds == "math":
        jobs = [(list(ext[q]), golds[q]) for q in range(Q)]
        if workers > 1:
            with mp.get_context("spawn").Pool(workers, initializer=_init, initargs=(cfg["mv_timeout_s"],)) as p:
                res = p.map(_math_q, jobs, chunksize=4)
        else:
            _init(cfg["mv_timeout_s"])
            res = [_math_q(j) for j in jobs]
        meta_src = None
    else:
        metas, meta_src = mathvista_meta(pool, cfg)
        res = [mathvista.group_and_grade(list(ext[q]), golds[q], metas[q]) for q in range(Q)]
    group = np.array([r["groups"] for r in res], dtype=np.int32)
    gc = [np.array(r["group_correct"], bool) for r in res]
    correct = np.zeros((Q, R), bool)
    for q in range(Q):
        m = group[q] != NO_ANSWER
        correct[q, m] = gc[q][group[q, m]]
    indiv = np.array([r["indiv_correct"] for r in res], dtype=bool)
    d = odir("answers", ds)
    np.savez(d / "labels.npz", group=group, correct=correct, indiv=indiv)
    rows = pd.DataFrame({"qi": np.repeat(np.arange(Q), R), "qid": np.repeat(pool.qids, R),
                         "rpos": np.tile(np.arange(R), Q), "rollout_id": pool.rollout_ids.ravel(),
                         "extracted": ext.ravel(), "group": group.ravel(), "correct": correct.ravel(),
                         "indiv_correct": indiv.ravel()})
    if ds == "mathvista":
        rows["kind"] = [k for r in res for k in r["rollout_kind"]]
    rows.to_parquet(d / "rollout_groups.parquet", index=False)
    g = [{"qid": pool.qids[q], "group": gi, "answer": a, "kind": k, "correct": bool(c),
          "size": int((group[q] == gi).sum())}
         for q, r in enumerate(res)
         for gi, (a, k, c) in enumerate(zip(r["group_answer"], r["group_kind"], r["group_correct"]))]
    g += [{"qid": pool.qids[q], "group": NO_ANSWER, "answer": "NO_ANSWER", "kind": "na", "correct": False,
           "size": int((group[q] == NO_ANSWER).sum())} for q in range(Q) if (group[q] == NO_ANSWER).any()]
    pd.DataFrame(g).to_parquet(d / "groups.parquet", index=False)
    n_groups = np.array([len(set(row.tolist())) for row in group])
    stats = {"questions": Q, "rollouts": int(Q * R), "no_answer_rollouts": int((group == NO_ANSWER).sum()),
             "rollout_accuracy": float(correct.mean()), "oracle_at_32": float(correct.any(1).mean()),
             "mean_distinct_groups_at_32": float(n_groups.mean()),
             "indiv_vs_group_grade_disagreements": int((indiv != correct).sum()),
             "questions_with_disagreement": int((indiv != correct).any(1).sum())}
    if ds == "math":
        kinds = [k for r in res for k in r["group_kind"]]
        stats["string_groups"] = int(sum(k == "str" for k in kinds))
        stats["gold_unparsed_questions"] = int(sum(not r["gold_parsed"] for r in res))
    else:
        rk = pd.Series([k for r in res for k in r["rollout_kind"]]).value_counts().to_dict()
        stats["rollout_kinds"] = {str(k): int(v) for k, v in rk.items()}
        stats["gold_unmatched_questions"] = int(sum(r["gold_key"] is None for r in res))
        stats["metadata_source"] = meta_src
    return stats


def extractor_check(ds: str) -> dict:
    """Compare our balanced-brace extractor on the stored texts with the supplied extractions."""
    pool = Pool(ds)
    t = pool.texts()
    ours = [math_equiv.norm_str(last_boxed(x)) for x in t["text"].tolist()]
    theirs = [math_equiv.norm_str(x) for x in pool.meta["extracted"].tolist()]
    same = np.array([a == b for a, b in zip(ours, theirs)])
    ex = [{"rollout": f"{q}/{r}", "ours": a, "supplied": b}
          for q, r, a, b, s in zip(t["qid"], t["rollout_id"], ours, theirs, same) if not s][:15]
    return {"agreement": float(same.mean()), "n": int(len(same)), "examples_of_mismatch": ex,
            "both_missing": int(sum(a is None and b is None for a, b in zip(ours, theirs)))}


def run(cfg: dict) -> dict:
    workers = max(1, len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else 1)
    man = {"math_verify": _version("math-verify"), "latex2sympy2_extended": _version("latex2sympy2_extended"),
           "scorer_source_sha256_16": _src_hash(), "mv_timeout_s": cfg["mv_timeout_s"],
           "rules": {"math": math_equiv.__doc__.strip(), "mathvista": mathvista.__doc__.strip()}}
    for ds in ("math", "mathvista"):
        man[ds] = score_dataset(ds, cfg, workers)
        try:
            man[ds]["extractor_check"] = extractor_check(ds)
        except Exception as ex:  # noqa: BLE001
            man[ds]["extractor_check"] = {"error": repr(ex)}
        print(f"[scorer] {ds}: {man[ds]}", flush=True)
    write_json(odir("answers") / "scorer_manifest.json", man)
    return man


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/config.py
"""Configuration and output paths.

The config is `configs/task3.yaml`; `PA3_CONFIG` points elsewhere and `PA3_CONFIG_OVERRIDES` (JSON) patches
keys (used by the tests).  Outputs live under `PA3_OUTPUT_ROOT`; large caches under `PA3_SCRATCH`.
"""
from __future__ import annotations

import copy
import json
import os
from pathlib import Path

import yaml

TASK = Path(__file__).resolve().parents[2]
DATASETS = ("math", "mathvista")
D_INDEX = {"math": 0, "mathvista": 1}
NO_ANSWER = -1          # group id of missing extractions (votes, always graded incorrect)


def _merge(a: dict, b: dict) -> dict:
    out = copy.deepcopy(a)
    for k, v in b.items():
        out[k] = _merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out


def load_config() -> dict:
    path = Path(os.environ.get("PA3_CONFIG", TASK / "configs" / "task3.yaml"))
    cfg = yaml.safe_load(path.read_text())
    ov = os.environ.get("PA3_CONFIG_OVERRIDES")
    if ov:
        cfg = _merge(cfg, json.loads(ov))
    return cfg


def out_root() -> Path:
    p = Path(os.environ.get("PA3_OUTPUT_ROOT", "pa3_task3_outputs"))
    p.mkdir(parents=True, exist_ok=True)
    return p


def scratch() -> Path:
    p = Path(os.environ.get("PA3_SCRATCH", str(out_root() / "_scratch")))
    p.mkdir(parents=True, exist_ok=True)
    return p


def cache_dir(ds: str) -> Path:
    p = scratch() / "cache" / ds
    p.mkdir(parents=True, exist_ok=True)
    return p


def odir(*parts: str) -> Path:
    p = out_root().joinpath(*parts)
    p.mkdir(parents=True, exist_ok=True)
    return p


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/data/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/data/discover.py
"""Find the supplied ATML-PA3-Data tables, tolerant to how the dataset was uploaded.

Accepted layouts (searched with symlinks followed, under data_root first, then all of /kaggle/input):
  * <dir>/generations.parquet + probability_metrics.parquet [+ attention_metrics.parquet]   (as in the task text)
  * any file or shard folder whose name contains "generation" / "prob" / "atten|attn", e.g.
    generations-00000-of-00002.parquet, generations/ (folder of parquet shards), mathvista_generations.parquet
  * both models' tables in one folder, told apart by "vista"/"vl" in the file names
  * the same inside a .zip / .tar(.gz) archive (extracted to scratch disk)
  * optionally a Hugging Face dataset repo (config `data_hf_repo`), downloaded to scratch disk
  * optionally a download link (config `data_url`: Google Drive file/folder link, or any http(s)/file URL to a
    .zip/.tar(.gz)/.parquet), downloaded to scratch disk and unpacked
The VLM table set is the one with an attention table.  A listing of the searched folders is always written to
data/input_listing.txt so a failure can be diagnosed from the results zip.
"""
from __future__ import annotations

import os
import re
import tarfile
import zipfile
from pathlib import Path

ROLE_RE = {"attn": re.compile(r"atten|attn", re.I), "prob": re.compile(r"prob|logp", re.I),
           "gen": re.compile(r"generation|rollout|completion", re.I)}
PARQ = (".parquet", ".pq")
ARCH = (".zip", ".tar", ".tar.gz", ".tgz")


def _walk(root: Path, max_depth: int = 8):
    """os.walk with symlinks followed and cycle protection; yields (dirpath, dirnames, filenames)."""
    seen = set()
    root = Path(root)
    base = len(root.parts)
    for dp, dns, fns in os.walk(root, followlinks=True):
        rp = os.path.realpath(dp)
        if rp in seen or len(Path(dp).parts) - base > max_depth:
            dns[:] = []
            continue
        seen.add(rp)
        dns.sort()
        yield Path(dp), dns, sorted(fns)


def listing(roots: list[Path], max_lines: int = 600) -> str:
    out = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            out.append(f"{root}  (does not exist)")
            continue
        for dp, dns, fns in _walk(root, max_depth=6):
            depth = len(dp.parts) - len(root.parts)
            link = " -> " + os.path.realpath(dp) if dp.is_symlink() else ""
            out.append("  " * depth + f"{dp.name or str(dp)}/{link}")
            for f in fns[:40]:
                try:
                    sz = (dp / f).stat().st_size
                except OSError:
                    sz = -1
                out.append("  " * (depth + 1) + f"{f}  ({sz / 2**20:.1f} MiB)")
            if len(fns) > 40:
                out.append("  " * (depth + 1) + f"... {len(fns) - 40} more files")
            if len(out) > max_lines:
                out.append("... (listing truncated)")
                return "\n".join(out)
    return "\n".join(out)


def _role(name: str) -> str | None:
    stem = name.lower()
    for r in ("attn", "prob", "gen"):          # order matters: "probability" never matches gen
        if ROLE_RE[r].search(stem):
            return r
    return None


def _hint(name: str) -> str:
    n = name.lower()
    if "vista" in n or re.search(r"(^|[^a-z])vl([^a-z]|$)", n):
        return "mathvista"
    if "math" in n:
        return "math"
    return ""


def _is_shard_dir(d: Path) -> bool:
    try:
        return any(f.suffix.lower() in PARQ for f in d.iterdir() if f.is_file())
    except OSError:
        return False


def scan(root: Path) -> dict[tuple[str, str], dict[str, list[str]]]:
    """-> {(folder, dataset-hint): {role: [parquet paths]}}"""
    groups: dict[tuple[str, str], dict[str, list[str]]] = {}
    for dp, dns, fns in _walk(root):
        for f in fns:
            if not f.lower().endswith(PARQ):
                continue
            r = _role(f)
            if r is None:
                # shard file inside a folder named after the role (generations/part-0.parquet)
                r = _role(dp.name)
                if r is None:
                    continue
                key = (str(dp.parent), _hint(dp.name) or _hint(dp.parent.name))
            else:
                key = (str(dp), _hint(f))
            groups.setdefault(key, {}).setdefault(r, []).append(str(dp / f))
    return groups


def _resolve_groups(groups) -> dict[str, dict[str, list[str]]]:
    found: dict[str, dict] = {}
    for (folder, hint), roles in sorted(groups.items()):
        if "gen" not in roles or "prob" not in roles:
            continue
        kind = "mathvista" if "attn" in roles else "math"
        if hint and hint != kind:
            continue
        if kind in found:
            raise ValueError(f"two candidate {kind} table sets: {found[kind]['folder']} and {folder}")
        found[kind] = {"folder": folder, **{r: sorted(v) for r, v in roles.items()}}
    return found


def _extract_archives(root: Path, dest: Path) -> list[Path]:
    out = []
    for dp, dns, fns in _walk(root, max_depth=4):
        for f in fns:
            p = dp / f
            low = f.lower()
            if not low.endswith(ARCH):
                continue
            tgt = dest / re.sub(r"[^A-Za-z0-9_.-]", "_", f)
            if not tgt.exists():
                print(f"[data] extracting {p} -> {tgt}", flush=True)
                tgt.mkdir(parents=True, exist_ok=True)
                if low.endswith(".zip"):
                    with zipfile.ZipFile(p) as z:
                        z.extractall(tgt)
                else:
                    with tarfile.open(p) as t:
                        t.extractall(tgt, filter="data")
            out.append(tgt)
    return out


def download(url: str, dest: Path) -> Path:
    """Fetch `url` into dest/ (Google Drive via gdown, otherwise urllib); unpack archives.  Returns the folder."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    if "drive.google.com" in url or "docs.google.com" in url:
        import gdown  # noqa: PLC0415  (preinstalled on Kaggle; pip install gdown otherwise)

        if "/folders/" in url:
            gdown.download_folder(url, output=str(dest), quiet=False)
        else:
            gdown.download(url, output=str(dest) + "/", quiet=False, fuzzy=True)
    else:
        import urllib.request  # noqa: PLC0415

        name = re.sub(r"[^A-Za-z0-9_.-]", "_", url.rstrip("/").split("/")[-1].split("?")[0]) or "download"
        print(f"[data] downloading {url} -> {dest / name}", flush=True)
        urllib.request.urlretrieve(url, dest / name)
    _extract_archives(dest, dest / "unpacked")
    return dest


def find_tables(preferred: str | Path, fallback: str | Path = "/kaggle/input", scratch: Path | None = None,
                hf_repo: str | None = None, listing_path: Path | None = None,
                url: str | None = None) -> dict[str, dict]:
    roots = [Path(preferred)] + ([Path(fallback)] if Path(fallback) != Path(preferred) else [])
    if listing_path is not None:
        listing_path.parent.mkdir(parents=True, exist_ok=True)
        listing_path.write_text(listing(roots))
    tried = []
    for root in roots:
        if not root.exists():
            continue
        tried.append(str(root))
        found = _resolve_groups(scan(root))
        if {"math", "mathvista"} <= set(found):
            return found
    if scratch is not None:
        for root in roots:
            if root.exists():
                for ex in _extract_archives(root, Path(scratch) / "extracted"):
                    found = _resolve_groups(scan(ex))
                    if {"math", "mathvista"} <= set(found):
                        return found
        if hf_repo:
            from huggingface_hub import snapshot_download  # noqa: PLC0415

            local = Path(scratch) / "hf_data"
            print(f"[data] downloading {hf_repo} from the Hugging Face Hub -> {local}", flush=True)
            snapshot_download(repo_id=hf_repo, repo_type="dataset", local_dir=str(local))
            tried.append(f"hf:{hf_repo}")
            found = _resolve_groups(scan(local))
            if {"math", "mathvista"} <= set(found):
                return found
        if url:
            local = Path(scratch) / "url_data"
            if not local.exists() or not _resolve_groups(scan(local)):
                download(url, local)
            tried.append(f"url:{url}")
            found = _resolve_groups(scan(local))
            if {"math", "mathvista"} <= set(found):
                return found
    raise FileNotFoundError(
        f"ATML-PA3-Data not found (searched {tried or [str(r) for r in roots]}). Need, for each model, a "
        "generations table and a probability_metrics table (parquet), plus an attention_metrics table for the VLM. "
        "Is the dataset attached (Add Input)? Otherwise set DATA_URL (download link) in the SETTINGS cell. What was found is listed in data/input_listing.txt:\n"
        + (listing(roots, max_lines=80)))


def find_readme(tables: dict[str, dict]) -> Path | None:
    seen = set()
    for t in tables.values():
        d = Path(t["folder"])
        for base in (d, d.parent, d.parent.parent):
            if base in seen:
                continue
            seen.add(base)
            for p in sorted(base.glob("*")):
                if p.is_file() and p.name.lower().startswith("readme"):
                    return p
    return None


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/data/load.py
"""Load, join and verify the supplied rollout exports and build a compact per-dataset cache.

Each model folder holds generations.parquet + probability_metrics.parquet (+ attention_metrics.parquet for the
VLM).  Tables are joined on (dataset, sample_id, rollout_id).  Token measurements are read as ragged arrays
(flat values + per-rollout lengths) straight from Arrow list columns; a long format (one row per token with a
position column) is also accepted.  Column names are auto-detected from alias lists and can be overridden with
`columns.<ds>.<role>` in the config; the resolution is written to data/<ds>_columns.json.

Cache (scratch disk, rebuilt if missing):  meta.parquet (one row per rollout, sorted by question ID string then
rollout ID), tokens.npz (offsets, logp, kl[, v]), questions.parquet, texts.parquet.
"""
from __future__ import annotations

import re

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from ..config import cache_dir, odir
from ..io import write_json

KEYS = ["dataset", "sample_id", "rollout_id"]
ALIASES = {
    "text": ["response", "generation", "generated_text", "output_text", "completion", "output", "text",
             "response_text", "decoded_text"],
    "token_ids": ["token_ids", "generated_token_ids", "output_token_ids", "completion_token_ids", "gen_token_ids",
                  "tokens", "output_ids"],
    "n_tokens": ["num_tokens", "n_tokens", "num_generated_tokens", "n_generated_tokens", "generated_length",
                 "output_length", "length", "token_count", "num_output_tokens", "retained_tokens", "n_retained"],
    "extracted": ["extracted_answer", "pred_answer", "predicted_answer", "extracted", "answer_extracted",
                  "parsed_answer", "prediction", "pred", "model_answer"],
    "gold": ["gold_answer", "gold", "ground_truth", "reference_answer", "target_answer", "target", "solution_answer",
             "answer"],
    "problem": ["problem", "question", "query", "question_text", "problem_text", "original_problem", "input",
                "prompt_text", "prompt"],
    "logp": ["logp", "token_logp", "sampled_logp", "logprobs", "log_probs", "token_logprobs"],
    "kl": ["kl_u_p", "kl_up", "kl_uniform", "self_certainty", "kl"],
    "attn_image": ["image", "attn_image", "image_attention", "image_attn", "attention_image", "image_total",
                   "a_image", "image_mass"],
    "attn_prompt": ["prompt", "prompt_text", "attn_prompt", "attn_prompt_text", "prompt_attention",
                    "prompt_text_attention", "text_prompt", "attention_prompt", "prompt_total", "a_prompt"],
    "attn_generated": ["generated", "generated_text", "attn_generated", "attn_generated_text", "generated_attention",
                       "generated_text_attention", "attention_generated", "generated_total", "a_generated", "gen"],
}
FUZZY = {"attn_image": r"image|img|vis", "attn_generated": r"gener|output|response|completion",
         "attn_prompt": r"prompt|question|instruction|text"}
POS_ALIASES = ["token_index", "token_idx", "token_position", "position", "pos", "t", "step", "index"]
MV_META = ["question_type", "answer_type", "choices", "precision", "unit"]


# ----------------------------------------------------------------------------- column resolution
def resolve(cols: list[str], role: str, override: str | None = None, required: bool = True,
            exclude: tuple = ()) -> str | None:
    if override:
        if override not in cols:
            raise KeyError(f"configured column {override!r} for {role!r} not in {cols}")
        return override
    lower = {c.lower(): c for c in cols if c not in exclude}
    for a in ALIASES.get(role, []):
        if a.lower() in lower:
            return lower[a.lower()]
    if role in FUZZY:
        m = [c for c in cols if c not in exclude and c not in KEYS and re.search(FUZZY[role], c, re.I)]
        if len(m) == 1:
            return m[0]
    if required:
        raise KeyError(f"cannot identify the column for {role!r} among {cols}; "
                       f"set columns.<dataset>.{role} in configs/task3.yaml")
    return None


def _is_list(t: pa.DataType) -> bool:
    return pa.types.is_list(t) or pa.types.is_large_list(t) or pa.types.is_fixed_size_list(t)


def segment_sum(vals: np.ndarray, lens: np.ndarray) -> np.ndarray:
    out = np.full(len(lens), np.nan)
    nz = lens > 0
    if nz.any():
        starts = np.concatenate([[0], np.cumsum(lens)[:-1]])
        out[nz] = np.add.reduceat(vals.astype(np.float64), starts[nz])
    return out


def list_column(arr: pa.Array | pa.ChunkedArray, lengths_only: bool = False):
    """Arrow list column -> (flat float values, per-row lengths).  Nested lists (token x something) are summed
    over the inner dimension (e.g. per-layer attention totals)."""
    if isinstance(arr, pa.ChunkedArray):
        arr = arr.combine_chunks() if arr.num_chunks else pa.array([], arr.type)
    lens = pc.list_value_length(arr).fill_null(0).to_numpy(zero_copy_only=False).astype(np.int64)
    if lengths_only:
        return None, lens
    inner = arr.flatten()
    if _is_list(inner.type):
        ilens = pc.list_value_length(inner).fill_null(0).to_numpy(zero_copy_only=False).astype(np.int64)
        vals = segment_sum(inner.flatten().to_numpy(zero_copy_only=False), ilens)
    else:
        vals = inner.to_numpy(zero_copy_only=False)
    vals = np.asarray(vals, dtype=np.float32)
    assert lens.sum() == len(vals), "list offsets inconsistent"
    return vals, lens


def gather(vals: np.ndarray, lens: np.ndarray, rows: np.ndarray, take: np.ndarray | None = None) -> np.ndarray:
    """Concatenate segments `rows` of a ragged array (optionally only the first `take[i]` items of each)."""
    offs = np.concatenate([[0], np.cumsum(lens)])
    starts = offs[rows]
    L = lens[rows] if take is None else np.minimum(lens[rows], take)
    new = np.concatenate([[0], np.cumsum(L)])
    idx = np.repeat(starts - new[:-1], L) + np.arange(new[-1])
    return vals[idx]


def _files(src) -> list[str]:
    if isinstance(src, (list, tuple)):
        return [str(x) for x in src]
    p = Path(src)
    return sorted(str(f) for f in p.glob("*.parquet")) if p.is_dir() else [str(p)]


def read_schema(src) -> pa.Schema:
    """src: a parquet file, a folder of shards, or a list of shard files."""
    return pq.read_schema(_files(src)[0])


def read_table(src, columns: list[str]) -> pa.Table:
    """Memory-mapped, single-threaded read, shard by shard (avoids the batch-buffer copy of a dataset scan)."""
    parts = [pq.read_table(f, columns=columns, memory_map=True, use_threads=False) for f in _files(src)]
    return parts[0] if len(parts) == 1 else pa.concat_tables(parts)


def _keys_str(df: pd.DataFrame, ds: str) -> pd.DataFrame:
    out = pd.DataFrame({k: df[k].astype(str) if k in df else ds for k in KEYS})
    return out


def read_token_table(path, roles: dict[str, str], ds: str) -> tuple[pd.DataFrame, dict]:
    """-> (keys frame with column _row, {role: (vals, lens)}) in the table's row order (or key order for the
    long format)."""
    schema = read_schema(path)
    cols = [c for c in roles.values()]
    key_cols = [k for k in KEYS if k in schema.names]
    tbl = read_table(path, columns=key_cols + cols)
    if all(_is_list(schema.field(c).type) for c in cols):
        keys = _keys_str(tbl.select(key_cols).to_pandas(), ds)
        data = {r: list_column(tbl.column(c)) for r, c in roles.items()}
        layout = "list"
    else:  # long format: one row per token
        pos = resolve(schema.names, "pos", required=False) or next((p for p in POS_ALIASES if p in schema.names), None)
        if pos is None:
            raise KeyError(f"{path}: scalar token columns {cols} but no token-position column")
        tbl = read_table(path, columns=key_cols + [pos] + cols)
        df = tbl.to_pandas()
        k = _keys_str(df, ds)
        df = pd.concat([k, df[[pos] + cols].reset_index(drop=True)], axis=1).sort_values(KEYS + [pos], kind="stable")
        g = df.groupby(KEYS, sort=False).size()
        keys = g.index.to_frame(index=False)
        lens = np.append(g.to_numpy().astype(np.int64), 0)      # extra empty segment for rollouts with no rows
        data = {r: (df[c].to_numpy(np.float32), lens) for r, c in roles.items()}
        layout = f"long(pos={pos})"
    keys["_row"] = np.arange(len(keys))
    return keys, {"_layout": layout, **data}


# ----------------------------------------------------------------------------- main loader
def _scalar_str(v) -> str | None:
    if v is None:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    if isinstance(v, (list, tuple, np.ndarray)):
        v = list(v)
        return None if not v else str(v[-1])
    s = str(v)
    return s if s.strip() and s.strip().lower() not in ("none", "nan", "null") else None


def load_dataset(src: dict, ds: str, cfg: dict) -> dict:
    """Join + verify one model's tables and write the cache.  `src` = {"folder", "gen": [files], "prob": [files],
    "attn": [files]} from data.discover.find_tables.  Returns the alignment report."""
    ov = (cfg.get("columns") or {}).get(ds, {}) or {}
    n_roll = int(cfg["n_rollouts"])
    gpath, ppath, apath = src["gen"], src["prob"], src.get("attn")
    gschema = read_schema(gpath)
    gcols = gschema.names
    colmap = {"generations": {}, "probability_metrics": {}, "attention_metrics": {}}
    gold = resolve(gcols, "gold", ov.get("gold"))
    extracted = resolve(gcols, "extracted", ov.get("extracted"), exclude=(gold,))
    text = resolve(gcols, "text", ov.get("text"), required=False, exclude=(gold, extracted))
    tok = resolve(gcols, "token_ids", ov.get("token_ids"), required=False)
    ntok = resolve(gcols, "n_tokens", ov.get("n_tokens"), required=False)
    problem = resolve(gcols, "problem", ov.get("problem"), required=False, exclude=(gold, extracted, text))
    colmap["generations"] = {"gold": gold, "extracted": extracted, "text": text, "token_ids": tok,
                             "n_tokens": ntok, "problem": problem}
    meta_cols = [c for c in MV_META if c in gcols] if ds == "mathvista" else []
    scal = [c for c in {gold, extracted, text, ntok, problem, *meta_cols} if c]
    key_cols = [k for k in KEYS if k in gcols]
    gtbl = read_table(gpath, columns=key_cols + scal + ([tok] if tok else []))
    gdf = pd.concat([_keys_str(gtbl.select(key_cols).to_pandas(), ds),
                     gtbl.select(scal).to_pandas()], axis=1)
    gdf["_n_token_ids"] = list_column(gtbl.column(tok), lengths_only=True)[1] if tok and _is_list(
        gschema.field(tok).type) else -1
    del gtbl
    gdf["_grow"] = np.arange(len(gdf))

    pschema = read_schema(ppath).names
    proles = {"logp": resolve(pschema, "logp", ov.get("logp")), "kl": resolve(pschema, "kl", ov.get("kl"))}
    colmap["probability_metrics"] = dict(proles)
    pkeys, pdat = read_token_table(ppath, proles, ds)
    tables = {"probability_metrics": (pkeys, pdat)}
    if ds == "mathvista":
        aschema = read_schema(apath).names
        img = resolve(aschema, "attn_image", ov.get("attn_image"))
        gen = resolve(aschema, "attn_generated", ov.get("attn_generated"), exclude=(img,))
        prm = resolve(aschema, "attn_prompt", ov.get("attn_prompt"), exclude=(img, gen))
        aroles = {"img": img, "prm": prm, "gen": gen}
        colmap["attention_metrics"] = dict(aroles)
        tables["attention_metrics"] = read_token_table(apath, aroles, ds)

    # ---- join on (dataset, sample_id, rollout_id)
    report: dict = {"dataset": ds, "source_files": src, "columns": colmap, "rows": {"generations": len(gdf)}}
    m = gdf
    for name, (keys, dat) in tables.items():
        report["rows"][name] = len(keys)
        report.setdefault("layout", {})[name] = dat["_layout"]
        keys = keys.rename(columns={"_row": f"_row_{name}"})
        both = m.merge(keys, on=KEYS, how="outer", indicator=True)
        report.setdefault("join_unmatched", {})[name] = {
            "only_in_generations": int((both["_merge"] == "left_only").sum()),
            f"only_in_{name}": int((both["_merge"] == "right_only").sum())}
        if dat["_layout"].startswith("long"):
            # long format has no rows for a zero-length rollout: keep it with an empty token list
            lo = both["_merge"] == "left_only"
            both.loc[lo, f"_row_{name}"] = len(dat[next(r for r in dat if r != "_layout")][1]) - 1
            both.loc[lo, "_merge"] = "both"
            report.setdefault("zero_length_filled_from_long_format", {})[name] = int(lo.sum())
        m = both[both["_merge"] == "both"].drop(columns="_merge")
        m[f"_row_{name}"] = m[f"_row_{name}"].astype(np.int64)
        if m.duplicated(KEYS).any():
            raise ValueError(f"{ds}: duplicate (dataset, sample_id, rollout_id) keys after joining {name}")
    m["qid"] = m["sample_id"].astype(str)
    rnum = pd.to_numeric(m["rollout_id"], errors="coerce")
    m["_rsort"] = rnum if rnum.notna().all() else m["rollout_id"]
    m = m.sort_values(["qid", "_rsort"], kind="stable").reset_index(drop=True)
    per_q = m.groupby("qid", sort=False).size()
    bad_q = per_q[per_q != n_roll]
    report["n_questions"] = int(len(per_q))
    report["rollouts_per_question"] = {str(k): int(v) for k, v in per_q.value_counts().items()}
    if len(bad_q):
        raise ValueError(f"{ds}: {len(bad_q)} questions do not have {n_roll} joined rollouts, e.g. "
                         f"{bad_q.head(5).to_dict()} (see join_unmatched {report['join_unmatched']})")
    report["dataset_values"] = sorted(map(str, m["dataset"].unique()))

    # ---- token alignment
    prow = m["_row_probability_metrics"].to_numpy(np.int64)
    lp_v, lp_l = pdat["logp"]
    kl_v, kl_l = pdat["kl"]
    lens = {"logp": lp_l[prow], "kl": kl_l[prow]}
    if ds == "mathvista":
        akeys, adat = tables["attention_metrics"]
        arow = m["_row_attention_metrics"].to_numpy(np.int64)
        for r in ("img", "prm", "gen"):
            lens[f"attn_{r}"] = adat[r][1][arow]
    T = np.min(np.stack(list(lens.values())), axis=0)
    mis = np.zeros(len(m), bool)
    for v in lens.values():
        mis |= v != T
    n_tok_col = m[ntok].to_numpy() if ntok else None
    ali = {"n_rollouts": int(len(m)), "misaligned_measurements": int(mis.sum()),
           "zero_length_rollouts": int((T == 0).sum()),
           "T_mean": float(T.mean()), "T_max": int(T.max()), "T_min": int(T.min())}
    if n_tok_col is not None:
        d = pd.to_numeric(pd.Series(n_tok_col), errors="coerce").to_numpy() - T
        ali["n_tokens_column_minus_T"] = {str(int(k)) if np.isfinite(k) else "nan": int(v)
                                          for k, v in pd.Series(d).value_counts().head(8).items()}
    if tok:
        d = m["_n_token_ids"].to_numpy() - T
        ali["token_ids_len_minus_T"] = {str(int(k)): int(v) for k, v in pd.Series(d).value_counts().head(8).items()}
    for name, v in lens.items():
        ali[f"mismatch_{name}_vs_T"] = int((v != T).sum())
    frac = mis.mean()
    if frac > float(cfg.get("max_misaligned_frac", 0.01)):
        write_json(odir("data") / f"{ds}_alignment.json", {**report, "alignment": ali})
        raise ValueError(f"{ds}: {mis.sum()} rollouts ({frac:.2%}) have token lists of different lengths: {ali}")
    report["alignment"] = ali

    logp = gather(lp_v, lp_l, prow, T).astype(np.float32)
    kl = gather(kl_v, kl_l, prow, T).astype(np.float32)
    arrays = {"offsets": np.concatenate([[0], np.cumsum(T)]).astype(np.int64), "logp": logp, "kl": kl}
    ali["nan_logp"] = int(np.isnan(logp).sum())
    ali["nan_kl"] = int(np.isnan(kl).sum())
    if ds == "mathvista":
        a = {r: gather(adat[r][0], adat[r][1], arow, T) for r in ("img", "prm", "gen")}
        den = a["img"] + a["prm"] + a["gen"]
        ok = np.isfinite(den) & (den > 0) & np.isfinite(a["img"])
        v = np.full(len(den), np.nan)
        v[ok] = a["img"][ok] / den[ok]
        arrays["v"] = v.astype(np.float32)
        ali["attention_tokens_undefined_v"] = int((~ok).sum())
        ali["v_outside_0_1"] = int(((v < -1e-6) | (v > 1 + 1e-6)).sum())

    # ---- per-rollout meta, per-question table, texts
    qids = list(per_q.index)
    qi_of = {q: i for i, q in enumerate(qids)}
    meta = pd.DataFrame({
        "qi": m["qid"].map(qi_of).to_numpy(np.int32), "qid": m["qid"],
        "rpos": m.groupby("qid", sort=False).cumcount().to_numpy(np.int32),
        "rollout_id": m["rollout_id"].astype(str), "T": T.astype(np.int64), "misaligned": mis,
        "extracted": [_scalar_str(x) for x in m[extracted].tolist()],
        "n_tokens_col": pd.to_numeric(m[ntok], errors="coerce") if ntok else np.nan,
    })
    gold_by_q = m.groupby("qid", sort=False)[gold].agg(lambda s: [_scalar_str(x) for x in s])
    ali["questions_with_inconsistent_gold"] = int(sum(len(set(v)) > 1 for v in gold_by_q))
    qdf = pd.DataFrame({"qi": np.arange(len(qids)), "qid": qids,
                        "gold": [v[0] for v in gold_by_q.loc[qids]]})
    first = m.groupby("qid", sort=False).head(1).set_index("qid").loc[qids]
    qdf["problem"] = first[problem].astype(str).to_numpy() if problem else None
    for c in meta_cols:
        qdf[c] = [_jsonable(x) for x in first[c].tolist()]
    report["mathvista_meta_columns_in_files"] = meta_cols
    cd = cache_dir(ds)
    meta.to_parquet(cd / "meta.parquet", index=False)
    qdf.to_parquet(cd / "questions.parquet", index=False)
    pd.DataFrame({"qid": m["qid"], "rollout_id": m["rollout_id"].astype(str),
                  "text": m[text].astype(str) if text else ""}).to_parquet(cd / "texts.parquet", index=False)
    np.savez(cd / "tokens.npz", **arrays)
    write_json(odir("data") / f"{ds}_alignment.json", report)
    write_json(odir("data") / f"{ds}_columns.json", {"generations": gcols, "probability_metrics": pschema,
                                                      "attention_metrics": read_schema(apath).names
                                                      if ds == "mathvista" else None, "resolved": colmap})
    return report


def _jsonable(x):
    if isinstance(x, np.ndarray):
        return [_jsonable(v) for v in x.tolist()]
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return None if np.isnan(x) else float(x)
    return x


# ----------------------------------------------------------------------------- cache access
class Pool:
    """In-memory view of one dataset's cache: [Q, R] matrices + flat token arrays."""

    def __init__(self, ds: str):
        cd = cache_dir(ds)
        self.ds = ds
        self.meta = pd.read_parquet(cd / "meta.parquet")
        self.questions = pd.read_parquet(cd / "questions.parquet")
        z = np.load(cd / "tokens.npz")
        self.offsets = z["offsets"]
        self.logp, self.kl = z["logp"], z["kl"]
        self.v = z["v"] if "v" in z.files else None
        self.qids = self.questions["qid"].tolist()
        self.Q = len(self.qids)
        self.R = int(len(self.meta) // self.Q)
        assert self.Q * self.R == len(self.meta)
        self.T = self.meta["T"].to_numpy().reshape(self.Q, self.R)
        self.rollout_ids = self.meta["rollout_id"].to_numpy().reshape(self.Q, self.R)

    def seg(self, q: int, r: int) -> slice:
        i = q * self.R + r
        return slice(int(self.offsets[i]), int(self.offsets[i + 1]))

    def texts(self) -> pd.DataFrame:
        return pd.read_parquet(cache_dir(self.ds) / "texts.parquet")


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/data/splits.py
"""Development / held-out splits, the live-generation subset, and candidate orders.

Splits (per dataset, fresh generator): sort question IDs as strings, permute with default_rng(6304); first 200
= dev, remaining 800 = held-out eval; the live subset is the first 50 MATH eval IDs in permutation order.
Orders: per question (index q in the sorted list) default_rng([6304, d, q]) draws 10 permutations of the 32
rollout positions (d = 0 MATH, 1 MathVista).  Stateless, so any subset reproduces the same orders.
"""
from __future__ import annotations

import numpy as np


def make_split(qids: list[str], seed: int, n_dev: int) -> dict:
    s = np.array(sorted(str(q) for q in qids))
    perm = np.random.default_rng(seed).permutation(len(s))
    p = s[perm]
    return {"seed": seed, "rule": "sorted(str ids) -> default_rng(seed).permutation; dev=first n_dev",
            "sorted": s.tolist(), "permutation": perm.tolist(), "permuted": p.tolist(),
            "dev": p[:n_dev].tolist(), "eval": p[n_dev:].tolist()}


def make_orders(n_q: int, d_index: int, seed: int, n_orders: int, n_roll: int) -> np.ndarray:
    out = np.empty((n_q, n_orders, n_roll), dtype=np.int16)
    for q in range(n_q):
        rng = np.random.default_rng([seed, d_index, q])
        for k in range(n_orders):
            out[q, k] = rng.permutation(n_roll)
    return out


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/io.py
"""Small torch-free I/O helpers (the offline stages never import torch)."""
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import numpy as np


def _default(o: Any) -> Any:
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (set, tuple)):
        return list(o)
    if isinstance(o, Path):
        return str(o)
    return str(o)


def _clean(o: Any) -> Any:
    if isinstance(o, float) and not np.isfinite(o):
        return None
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    return o


def write_json(path: str | Path, obj: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(_clean(obj), indent=2, default=_default))
    os.replace(tmp, path)


def read_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text())


def append_jsonl(path: str | Path, row: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a") as fh:
        fh.write(json.dumps(_clean(row), default=_default) + "\n")
        fh.flush()
        os.fsync(fh.fileno())


def read_jsonl(path: str | Path) -> list[dict]:
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if line:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:   # a line cut off by a killed process
                pass
    return rows


def pin_cpus(var: str = "PA3_CPUS") -> list[int] | None:
    """Restrict this process to the CPU cores listed in env `var` (e.g. "2,3")."""
    spec = os.environ.get(var)
    if not spec or not hasattr(os, "sched_setaffinity"):
        return None
    cores = {int(c) for c in spec.split(",") if c.strip()}
    try:
        os.sched_setaffinity(0, cores)
    except OSError:
        return None
    return sorted(os.sched_getaffinity(0))


def df_to_md(df, floatfmt: str = ".4f", index: bool = False) -> str:
    """Markdown table without the optional `tabulate` dependency."""
    cols = ([df.index.name or ""] if index else []) + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for row in df.itertuples(index=True):
        idx, values = row[0], row[1:]
        vals = [str(idx)] if index else []
        for v in values:
            if isinstance(v, (float, np.floating)):
                vals.append("" if not np.isfinite(v) else format(v, floatfmt))
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines) + "\n"


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/live/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/live/backend.py
"""Generation backends.  HFBackend: Qwen2.5-3B-Instruct, FP16, no quantisation, batch size 1, KV cache.
FakeBackend: deterministic stand-in used by the CPU tests (same accounting code path).

Sampling: temperature 0.7, top-p 0.8, top-k 20, repetition penalty 1.0.  The model's own generation_config
(which ships repetition_penalty 1.05) is overwritten as well as passing explicit kwargs, so no hub default leaks
in; the effective config is recorded.  Each call is seeded (torch + CUDA) right before generation.  Timing is
end-to-end per call (chat templating, tokenisation, generation, decoding), with CUDA synchronisation; model
loading and one short warm-up call are excluded.
"""
from __future__ import annotations

import time


class HFBackend:
    def __init__(self, model_id: str, revision: str, live_cfg: dict, device: str = "cuda"):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.device = torch.device(device if torch.cuda.is_available() or device == "cpu" else "cpu")
        dtype = getattr(torch, live_cfg.get("dtype", "float16"))
        self.tok = AutoTokenizer.from_pretrained(model_id, revision=revision)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, dtype=dtype).to(self.device)
        self.model.eval()
        self.sampling = {k: live_cfg[k] for k in ("temperature", "top_p", "top_k", "repetition_penalty")}
        gc = self.model.generation_config
        for k, v in self.sampling.items():
            setattr(gc, k, v)
        gc.do_sample = True
        eos = gc.eos_token_id
        eos = [eos] if isinstance(eos, int) else list(eos or [])
        if self.tok.eos_token_id is not None:
            eos.append(self.tok.eos_token_id)
        im_end = self.tok.convert_tokens_to_ids("<|im_end|>")
        if isinstance(im_end, int) and im_end >= 0 and im_end != self.tok.unk_token_id:
            eos.append(im_end)
        self.eos_ids = sorted(set(eos))
        gc.eos_token_id = self.eos_ids
        self.pad_id = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.eos_ids[0]
        self.info = {"model": model_id, "revision": revision,
                     "commit": getattr(self.model.config, "_commit_hash", None), "dtype": str(dtype),
                     "device": str(self.device), "eos_ids": self.eos_ids,
                     "gpu": torch.cuda.get_device_name(self.device) if self.device.type == "cuda" else "cpu",
                     "generation_config": gc.to_dict()}

    def render(self, messages: list[dict]) -> str:
        return self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def warmup(self) -> None:
        self.generate([{"role": "user", "content": "Hello"}], 8, seed=0)

    def generate(self, messages: list[dict], max_new_tokens: int, seed: int) -> dict:
        torch = self.torch
        cuda = self.device.type == "cuda"
        if cuda:
            torch.cuda.synchronize(self.device)
            torch.cuda.reset_peak_memory_stats(self.device)
        t0 = time.perf_counter()
        prompt = self.render(messages)
        enc = self.tok(prompt, return_tensors="pt", add_special_tokens=False).to(self.device)
        n_in = int(enc["input_ids"].shape[1])
        torch.manual_seed(seed)
        if cuda:
            torch.cuda.manual_seed_all(seed)
        err = None
        try:
            with torch.inference_mode():
                out = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                                          eos_token_id=self.eos_ids, pad_token_id=self.pad_id, **self.sampling)
            gen = out[0, n_in:].tolist()
        except RuntimeError as ex:      # e.g. FP16 inf/nan in the sampling distribution: recorded, not hidden
            if "inf" not in str(ex) and "nan" not in str(ex):
                raise
            gen, err = [], f"{type(ex).__name__}: {str(ex)[:300]}"
        stop = next((i for i, t in enumerate(gen) if t in self.eos_ids), None)
        retained = gen[:stop] if stop is not None else gen
        text = self.tok.decode(retained, skip_special_tokens=True)
        if cuda:
            torch.cuda.synchronize(self.device)
        t1 = time.perf_counter()
        return {"prompt_text": prompt, "input_tokens": n_in, "output_tokens": len(gen),
                "output_tokens_retained": len(retained), "terminated": stop is not None,
                "truncated": stop is None and len(gen) >= max_new_tokens, "response": text,
                "seconds": t1 - t0, "error": err,
                "peak_gpu_mem_gib": torch.cuda.max_memory_allocated(self.device) / 2**30 if cuda else None}


class FakeBackend:
    """Deterministic stand-in for tests: the answer depends on (seed, prompt) so seeds matter; truncates when the
    'length' drawn from the seed exceeds the cap."""

    def __init__(self, live_cfg: dict, gold: dict | None = None):
        self.gold = gold or {}
        self.info = {"model": "fake", "revision": None, "sampling": {k: live_cfg[k] for k in
                                                                  ("temperature", "top_p", "top_k", "repetition_penalty")}}

    def render(self, messages):
        return "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages) + "<|im_start|>assistant\n"

    def warmup(self):
        pass

    def generate(self, messages, max_new_tokens, seed):
        import numpy as np

        prompt = self.render(messages)
        rng = np.random.default_rng(seed)
        length = int(rng.integers(50, 1500))
        q = next((k for k in self.gold if k in messages[1]["content"]), None)
        is_review = len(messages) > 2
        ans = self.gold.get(q, "0") if (rng.random() < (0.7 if is_review else 0.5)) else str(int(rng.integers(1, 9)))
        n_out = min(length, max_new_tokens)
        trunc = length > max_new_tokens
        text = "<think>work</think> " + ("partial" if trunc else f"\\boxed{{{ans}}}")
        return {"prompt_text": prompt, "input_tokens": len(prompt.split()), "output_tokens": n_out + (0 if trunc else 1),
                "output_tokens_retained": n_out, "terminated": not trunc, "truncated": trunc, "response": text,
                "seconds": 0.001 * n_out, "error": None, "peak_gpu_mem_gib": None}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/live/evaluate.py
"""Step 4 evaluation of the live calls (no gold answers are used to choose between outputs).

Strategies (costs are actual tokens/time of the calls they consume):
  single          slot 0
  parallel        slots 1-3, self-consistency over the three answers (tie -> earliest, i.e. slot 1)
  refine          slots 1 + 4 (shared initial response charged), returns the revised answer; NO_ANSWER if the
                  revision has no \\boxed{} (approved decision 6, main row)
  refine_fallback same calls; keeps the initial answer when the revision has none (comparative extra)
  initial_only    slot 1 alone (reference)
Answers are grouped/graded with the same math-verify rule as the offline scorer.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..answers.math_equiv import NO_ANSWER, group_and_grade
from ..config import odir
from ..io import df_to_md, read_json, read_jsonl, write_json
from ..offline.bootstrap import boot_idx, paired
from ..select.rules import vote

CALLS = {"single": [0], "parallel": [1, 2, 3], "refine": [1, 4], "refine_fallback": [1, 4], "initial_only": [1]}


def run(cfg: dict) -> dict:
    out = odir("live")
    qs = read_json(out / "questions.json")
    calls = {(r["qid"], r["slot"]): r for r in read_jsonl(out / "responses.jsonl")}
    rows, incomplete = [], []
    for item in qs:
        qid, gold = item["qid"], item["gold"]
        if not all((qid, s) in calls for s in range(5)):
            incomplete.append(qid)
            continue
        a = {s: calls[(qid, s)]["extracted"] for s in range(5)}
        r = {"qid": qid, "live_idx": item["live_idx"], "gold": gold}
        g = group_and_grade([a[0]], gold, cfg["mv_timeout_s"])
        r["single_answer"], r["single_correct"] = a[0], g["groups"][0] != NO_ANSWER and g["group_correct"][g["groups"][0]]
        g = group_and_grade([a[1], a[2], a[3]], gold, cfg["mv_timeout_s"])
        win, idx, tot = vote(g["groups"])
        r["parallel_answer"] = a[[1, 2, 3][idx]]
        r["parallel_correct"] = win != NO_ANSWER and g["group_correct"][win]
        r["parallel_n_groups"] = len(tot)
        r["parallel_all_distinct_tie"] = len(tot) == 3
        g = group_and_grade([a[1], a[4]], gold, cfg["mv_timeout_s"])
        gi, gr = g["groups"]
        ok = lambda x: x != NO_ANSWER and g["group_correct"][x]  # noqa: E731
        r["initial_answer"], r["initial_correct"] = a[1], ok(gi)
        r["refine_answer"], r["refine_correct"] = a[4], ok(gr)
        r["refine_fallback_answer"] = a[4] if gr != NO_ANSWER else a[1]
        r["refine_fallback_correct"] = ok(gr) if gr != NO_ANSWER else ok(gi)
        r["revision_changed_answer"] = gi != gr
        r["initial_only_answer"], r["initial_only_correct"] = a[1], ok(gi)
        for s, ss in CALLS.items():
            cs = [calls[(qid, x)] for x in ss]
            r[f"{s}_input_tokens"] = sum(c["input_tokens"] for c in cs)
            r[f"{s}_output_tokens"] = sum(c["output_tokens"] for c in cs)
            r[f"{s}_output_tokens_retained"] = sum(c["output_tokens_retained"] for c in cs)
            r[f"{s}_seconds"] = sum(c["seconds"] for c in cs)
            r[f"{s}_truncated_calls"] = sum(bool(c["truncated"]) for c in cs)
            r[f"{s}_n_calls"] = len(cs)
            r[f"{s}_errors"] = sum(c.get("error") is not None for c in cs)
            pk = [c["peak_gpu_mem_gib"] for c in cs if c.get("peak_gpu_mem_gib") is not None]
            r[f"{s}_peak_gpu_mem_gib"] = max(pk) if pk else np.nan
            r[f"{s}_extracted"] = r[f"{s}_answer"] is not None
        r["refine_final_call_truncated"] = bool(calls[(qid, 4)]["truncated"])
        r["single_final_call_truncated"] = bool(calls[(qid, 0)]["truncated"])
        rows.append(r)
    df = pd.DataFrame(rows)
    df.to_csv(out / "per_question.csv", index=False)
    if df.empty:
        write_json(out / "summary.json", {"incomplete": incomplete, "n": 0})
        return {"n": 0}
    idx = boot_idx(len(df), cfg["n_boot"], cfg["seed"])
    summ = []
    for s in CALLS:
        summ.append({"strategy": s, "accuracy": df[f"{s}_correct"].mean(),
                     "extraction_rate": df[f"{s}_extracted"].mean(),
                     "truncation_rate_calls": df[f"{s}_truncated_calls"].sum() / df[f"{s}_n_calls"].sum(),
                     "mean_input_tokens": df[f"{s}_input_tokens"].mean(),
                     "mean_output_tokens": df[f"{s}_output_tokens"].mean(),
                     "mean_output_tokens_retained": df[f"{s}_output_tokens_retained"].mean(),
                     "mean_seconds": df[f"{s}_seconds"].mean(), "total_seconds": df[f"{s}_seconds"].sum(),
                     "max_peak_gpu_mem_gib": df[f"{s}_peak_gpu_mem_gib"].max(),
                     "errors": int(df[f"{s}_errors"].sum()), "output_allowance": {
                         "single": 3072, "parallel": 3072, "refine": 3072, "refine_fallback": 3072,
                         "initial_only": 1024}[s]})
    sdf = pd.DataFrame(summ)
    ic, rc = df["initial_correct"].astype(bool), df["refine_correct"].astype(bool)
    trans = {"wrong_to_correct": int((~ic & rc).sum()), "correct_to_wrong": int((ic & ~rc).sum()),
             "correct_to_correct": int((ic & rc).sum()), "wrong_to_wrong": int((~ic & ~rc).sum()),
             "answer_changed": int(df["revision_changed_answer"].sum()),
             "revision_no_answer": int((~df["refine_extracted"].astype(bool)).sum()),
             "revision_truncated": int(df["refine_final_call_truncated"].sum())}
    cmp = {f"{a}_minus_{b}": paired(df[f"{a}_correct"].astype(float).to_numpy(),
                                     df[f"{b}_correct"].astype(float).to_numpy(), idx)
           for a, b in [("parallel", "single"), ("refine", "single"), ("refine", "parallel"),
                        ("refine", "initial_only"), ("refine_fallback", "refine")]}
    summary = {"n_questions": len(df), "incomplete": incomplete, "transitions_vs_initial": trans,
               "bootstrap": cmp, "parallel_three_way_ties": int(df["parallel_all_distinct_tie"].sum()),
               "backend": read_json(out / "backend.json") if (out / "backend.json").exists() else None}
    sdf.to_csv(out / "summary.csv", index=False)
    write_json(out / "summary.json", {**summary, "table": summ})
    (out / "summary.md").write_text(
        f"## Live strategies on {len(df)} MATH held-out questions (Qwen2.5-3B-Instruct, FP16, batch 1)\n\n"
        + df_to_md(sdf, floatfmt=".3f") + "\n### Revision transitions vs the shared initial answer\n\n"
        + "\n".join(f"- {k}: {v}" for k, v in trans.items()) + "\n\n### Paired bootstrap (accuracy differences)\n\n"
        + "\n".join(f"- {k}: {v['diff']:+.3f} [{v['lo']:+.3f}, {v['hi']:+.3f}]" for k, v in cmp.items()) + "\n")
    print(f"[live-eval] {summary['transitions_vs_initial']}", flush=True)
    return summary


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/live/run.py
"""Step 4 live-generation worker (runs in the background on GPU0, pinned to its own CPU cores).

Per question, five independently seeded calls in the fixed order 1, 2, 3, 4, 0:
  slots 1-3  parallel responses (cap 1,024); slot 1 is also the shared initial response for refinement
  slot 4     critique-and-revision (cap 2,048): system, user, assistant = slot-1 response, user = review instruction
  slot 0     single longer response (cap 3,072)
Seed of a call = SeedSequence([6304, live question index, slot]).  Every call is appended to responses.jsonl
(prompt, response, seed, token counts, truncation flag, time, peak memory); a rerun skips finished calls, so the
worker resumes after a pause.  Exit code 3 = paused because the session budget is nearly used.
"""
from __future__ import annotations

import os
import sys
import time

import numpy as np

from ..answers.extract import last_boxed
from ..config import load_config, odir
from ..io import append_jsonl, pin_cpus, read_json, read_jsonl, write_json

SLOT_ORDER = (1, 2, 3, 4, 0)


def call_seed(seed: int, live_idx: int, slot: int) -> int:
    return int(np.random.SeedSequence([seed, live_idx, slot]).generate_state(1)[0])


def messages_initial(problem: str, lc: dict) -> list[dict]:
    return [{"role": "system", "content": lc["system"]},
            {"role": "user", "content": f"{problem}\n\n{lc['instruction']}"}]


def messages_review(problem: str, initial_response: str, lc: dict) -> list[dict]:
    return messages_initial(problem, lc) + [{"role": "assistant", "content": initial_response},
                                            {"role": "user", "content": lc["review"]}]


def cap_of(slot: int, lc: dict) -> int:
    return lc["caps"]["single"] if slot == 0 else lc["caps"]["revision"] if slot == 4 else lc["caps"]["parallel"]


def make_backend(cfg: dict, questions: list[dict]):
    lc = cfg["live"]
    if os.environ.get("PA3_LIVE_BACKEND") == "fake":
        from .backend import FakeBackend

        return FakeBackend(lc, {q["problem"]: q["gold"] for q in questions})
    from .backend import HFBackend

    return HFBackend(lc["model"], lc["revision"], lc)


def main() -> int:
    cfg = load_config()
    lc = cfg["live"]
    cores = pin_cpus()
    out = odir("live")
    qs = read_json(out / "questions.json")
    path = out / "responses.jsonl"
    rows = read_jsonl(path)
    have = {(r["qid"], r["slot"]): r for r in rows}
    if len(have) == len(qs) * len(SLOT_ORDER):
        print("[live] all calls already done", flush=True)
        return 0
    deadline = float(os.environ.get("PA3_DEADLINE_UNIX", "inf"))
    margin = 60 * float(lc.get("stop_margin_min", 20))
    t_load = time.time()
    backend = make_backend(cfg, qs)
    backend.warmup()
    write_json(out / "backend.json", {**backend.info, "cpu_cores": cores, "load_and_warmup_seconds": time.time() - t_load})
    print(f"[live] backend ready ({time.time() - t_load:.0f}s), cores={cores}", flush=True)
    for item in qs:
        for slot in SLOT_ORDER:
            if (item["qid"], slot) in have:
                continue
            if time.time() > deadline - margin:
                write_json(out / "status.json", {"state": "paused", "done_calls": len(have)})
                print("[live] session budget nearly used -> paused", flush=True)
                return 3
            if slot == 4:
                msgs = messages_review(item["problem"], have[(item["qid"], 1)]["response"], lc)
            else:
                msgs = messages_initial(item["problem"], lc)
            seed = call_seed(cfg["seed"], item["live_idx"], slot)
            res = backend.generate(msgs, cap_of(slot, lc), seed)
            row = {"qid": item["qid"], "live_idx": item["live_idx"], "slot": slot, "seed": seed,
                   "cap": cap_of(slot, lc), "messages": msgs, **res, "extracted": last_boxed(res["response"])}
            append_jsonl(path, row)
            have[(item["qid"], slot)] = row
        done_q = sum(all((q["qid"], s) in have for s in SLOT_ORDER) for q in qs)
        print(f"[live] {done_q}/{len(qs)} questions done", flush=True)
    write_json(out / "status.json", {"state": "done", "done_calls": len(have)})
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/offline/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/offline/bootstrap.py
"""Paired question-level bootstrap (resample questions, never rollouts) on per-question values that were already
averaged over the 10 candidate orders."""
from __future__ import annotations

import numpy as np


def boot_idx(n: int, n_boot: int, seed: int) -> np.ndarray:
    return np.random.default_rng(seed).integers(0, n, size=(n_boot, n))


def paired(a: np.ndarray, b: np.ndarray, idx: np.ndarray) -> dict:
    d = np.asarray(a, float) - np.asarray(b, float)
    bm = d[idx].mean(axis=1)
    return {"diff": float(d.mean()), "lo": float(np.percentile(bm, 2.5)), "hi": float(np.percentile(bm, 97.5)),
            "a": float(np.mean(a)), "b": float(np.mean(b)), "n": int(len(d))}


def mean_ci(a: np.ndarray, idx: np.ndarray) -> dict:
    a = np.asarray(a, float)
    bm = a[idx].mean(axis=1)
    return {"mean": float(a.mean()), "lo": float(np.percentile(bm, 2.5)), "hi": float(np.percentile(bm, 97.5))}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/offline/runner.py
"""Steps 1-2: every selection rule at every candidate budget, for every saved order.

per_order.parquet  one row per (question, order, N, method): selected rollout, answer group/answer, correctness,
                   generated-token cost (sum of T over consumed candidates; FirstAns consumes only its first),
                   oracle coverage and distinct answer groups among the first N candidates.
per_question.parquet  the same averaged over the 10 orders (the unit for all summaries and bootstraps).
summary_<split>.csv/md, decomposition, bootstrap.csv (eval split, paired over questions).
"""
from __future__ import annotations

import time

import numpy as np
import pandas as pd

from ..config import NO_ANSWER, odir
from ..data.load import Pool
from ..io import df_to_md, read_json, write_json
from ..select.rules import select
from .bootstrap import boot_idx, paired

BASE = [("first", None), ("sc", None), ("logp", "L"), ("logp_rwc", "L"), ("selfcert", "K"), ("selfcert_rwc", "K"),
        ("shortest", None)]
VLM = [("vision", "V"), ("vision_rwc", "V"), ("visconf", "Z"), ("visconf_rwc", "Z"), ("tokfuse", "Ztok"),
       ("tokfuse_rwc", "Ztok")]
EXTRA = {"logp_rwc": "extra ablation (approved 11)", "shortest": "length-only control",
         "tokfuse": "Step-3 ablation", "tokfuse_rwc": "Step-3 ablation"}
LABEL = {"first": "FirstAns", "sc": "Self-consistency", "logp": "Highest mean log-prob", "logp_rwc": "Log-prob + RWC",
         "selfcert": "Highest self-certainty", "selfcert_rwc": "Self-certainty + RWC", "shortest": "Shortest rollout",
         "vision": "Vision-only (highest V)", "vision_rwc": "Vision-only + RWC", "visconf": "VisConf (highest Z)",
         "visconf_rwc": "VisConf + RWC", "tokfuse": "Tokenwise fusion (highest)", "tokfuse_rwc": "Tokenwise fusion + RWC"}


def methods_for(ds: str) -> list[tuple[str, str | None]]:
    return BASE + (VLM if ds == "mathvista" else [])


def load_inputs(ds: str):
    pool = Pool(ds)
    lab = np.load(odir("answers", ds) / "labels.npz")
    sc = dict(np.load(odir("signals", ds) / "rollout_scores.npz"))
    vc = dict(np.load(odir("signals", ds) / "visconf.npz")) if ds == "mathvista" else {}
    orders = np.load(odir("splits") / f"{ds}_orders.npy")
    split = read_json(odir("splits") / f"{ds}.json")
    return pool, lab["group"], lab["correct"], sc, vc, orders, split


def prefix_scores(key, sc, vc, q, k, o, N):
    if key in ("Z", "Ztok"):
        return vc[f"{key}_{N}"][q, k] if N >= 2 else np.zeros(1)
    return sc[key][q, o[:N]]


def run_dataset(ds: str, cfg: dict) -> pd.DataFrame:
    pool, group, correct, sc, vc, orders, split = load_inputs(ds)
    meths = methods_for(ds)
    Q, K, R = orders.shape
    B = list(cfg["budgets"])
    n = Q * K * len(B) * len(meths)
    col = {c: np.zeros(n, t) for c, t in [("qi", np.int32), ("order", np.int8), ("N", np.int8), ("m", np.int8),
                                          ("rpos", np.int16), ("group", np.int32), ("correct", bool),
                                          ("cost", np.int64), ("oracle", bool), ("n_groups", np.int16),
                                          ("n_groups_answered", np.int16)]}
    T = pool.T
    i = 0
    for q in range(Q):
        for k in range(K):
            o = orders[q, k]
            g_o, c_o, T_o = group[q, o], correct[q, o], T[q, o]
            for N in B:
                gN = g_o[:N]
                oracle = bool(c_o[:N].any())
                ug = set(gN.tolist())
                ng, nga = len(ug), len(ug - {NO_ANSWER})
                cost_all = int(T_o[:N].sum())
                for mi, (m, key) in enumerate(meths):
                    s = prefix_scores(key, sc, vc, q, k, o, N) if key else None
                    j = select(m, gN, s, T_o[:N])
                    col["qi"][i], col["order"][i], col["N"][i], col["m"][i] = q, k, N, mi
                    col["rpos"][i], col["group"][i], col["correct"][i] = o[j], gN[j], c_o[j]
                    col["cost"][i] = int(T_o[0]) if m == "first" else cost_all
                    col["oracle"][i], col["n_groups"][i], col["n_groups_answered"][i] = oracle, ng, nga
                    i += 1
    df = pd.DataFrame(col)
    df["method"] = pd.Categorical.from_codes(df.pop("m"), [m for m, _ in meths])
    qids = np.array(pool.qids)
    df.insert(0, "qid", qids[df["qi"].to_numpy()])
    sp = {q: "dev" for q in split["dev"]} | {q: "eval" for q in split["eval"]}
    df.insert(1, "split", df["qid"].map(sp))
    df["rollout_id"] = pool.rollout_ids[df["qi"].to_numpy(), df["rpos"].to_numpy()]
    g = pd.read_parquet(odir("answers", ds) / "groups.parquet")
    amap = dict(zip(zip(g["qid"], g["group"]), g["answer"]))
    df["answer"] = [amap.get((a, b)) for a, b in zip(df["qid"].tolist(), df["group"].tolist())]
    return df


def per_question(df: pd.DataFrame) -> pd.DataFrame:
    pq_ = (df.groupby(["qid", "split", "N", "method"], observed=True)
           .agg(acc=("correct", "mean"), oracle=("oracle", "mean"), cost=("cost", "mean"),
                n_groups=("n_groups", "mean"), n_groups_answered=("n_groups_answered", "mean"))
           .reset_index())
    pq_["missed"] = pq_["oracle"] - pq_["acc"]          # correct candidate available but not selected
    pq_["no_correct"] = 1.0 - pq_["oracle"]             # sampling failure
    return pq_


def summarize(pq_: pd.DataFrame, split: str) -> pd.DataFrame:
    s = (pq_[pq_["split"] == split].groupby(["N", "method"], observed=True)
         [["acc", "oracle", "cost", "n_groups", "n_groups_answered", "missed", "no_correct"]].mean().reset_index())
    s["label"] = s["method"].map(LABEL)
    s["note"] = s["method"].map(EXTRA).fillna("")
    return s


def comparisons(ds: str) -> list[tuple[str, str]]:
    c = [("sc", "first")]
    scores = ["logp", "selfcert"] + (["vision", "visconf", "tokfuse"] if ds == "mathvista" else [])
    for s in scores:
        c += [(s, "sc"), (s + "_rwc", "sc"), (s + "_rwc", s)]
    c += [("shortest", "sc")]
    if ds == "mathvista":
        c += [("visconf", "vision"), ("visconf_rwc", "vision_rwc"), ("visconf", "selfcert"),
              ("visconf_rwc", "selfcert_rwc"), ("tokfuse", "visconf"), ("tokfuse_rwc", "visconf_rwc")]
    return c


def bootstrap_table(pq_: pd.DataFrame, ds: str, cfg: dict, split: str = "eval") -> pd.DataFrame:
    e = pq_[pq_["split"] == split]
    piv = e.pivot_table(index="qid", columns=["N", "method"], values="acc", observed=True)
    idx = boot_idx(len(piv), cfg["n_boot"], cfg["seed"])
    rows = []
    for N in [n for n in cfg["budgets"] if n >= 2]:
        for a, b in comparisons(ds):
            if (N, a) in piv and (N, b) in piv:
                rows.append({"dataset": ds, "split": split, "N": N, "A": a, "B": b,
                             **paired(piv[(N, a)].to_numpy(), piv[(N, b)].to_numpy(), idx)})
    return pd.DataFrame(rows)


def compact_table(s: pd.DataFrame, value: str = "acc") -> pd.DataFrame:
    t = s.pivot_table(index="method", columns="N", values=value, observed=True)
    order = [m for m in LABEL if m in t.index]
    t = t.loc[order]
    t.index = [LABEL[m] + (" *" if m in EXTRA else "") for m in t.index]
    t.columns = [f"N={c}" for c in t.columns]
    return t


def run(cfg: dict) -> dict:
    info = {}
    for ds in ("math", "mathvista"):
        t0 = time.time()
        df = run_dataset(ds, cfg)
        d = odir("offline", ds)
        df.to_parquet(d / "per_order.parquet", index=False)
        pq_ = per_question(df)
        pq_.to_parquet(d / "per_question.parquet", index=False)
        for split in ("eval", "dev"):
            s = summarize(pq_, split)
            s.to_csv(d / f"summary_{split}.csv", index=False)
            (d / f"summary_{split}.md").write_text(
                f"## {ds} — {split} split: accuracy by method and budget (* = extra/ablation)\n\n"
                + df_to_md(compact_table(s), index=True)
                + "\n### oracle coverage / distinct answer groups / mean token cost (all-candidate methods)\n\n"
                + df_to_md(s[s["method"] == "sc"][["N", "oracle", "n_groups", "n_groups_answered", "cost",
                                                     "missed", "no_correct"]].reset_index(drop=True)))
        bt = bootstrap_table(pq_, ds, cfg)
        bt.to_csv(d / "bootstrap.csv", index=False)
        orders = np.load(odir("splits") / f"{ds}_orders.npy")
        pool = Pool(ds)
        pd.DataFrame([{"qid": pool.qids[q], "order": k, "candidate_rollout_ids": pool.rollout_ids[q, orders[q, k]].tolist()}
                      for q in range(orders.shape[0]) for k in range(orders.shape[1])]).to_parquet(
            odir("splits") / f"{ds}_candidate_orders.parquet", index=False)
        info[ds] = {"rows": len(df), "seconds": round(time.time() - t0, 1)}
        print(f"[offline] {ds}: {info[ds]}", flush=True)
    write_json(odir("offline") / "info.json", info)
    return info


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/orchestrate.py
"""Kaggle pipeline for Task 3 (failure-tolerant, resumable).

data -> splits -> [live generation starts in the background on GPU0, pinned to its own CPU cores] ->
scorer -> signals -> offline (Steps 1-2) -> analysis (Step 3) -> adaptive (Step 5) -> compute (Step 6) ->
wait for live -> live_eval (Step 4) -> qualitative -> report.

Offline stages run as subprocesses pinned to the other cores (approved decision 7) so the generation timing is not
disturbed.  A stage is skipped when its _stages/<stage>.json marker exists (also from an earlier session attached
under /kaggle/input), except `data`, which is rebuilt whenever the scratch cache is missing.  A failed stage is
recorded in status.json and only the stages that depend on it are skipped.
"""
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

from pa3common.kaggle import Deadline, copytree_merge, discover_prior_roots, n_gpus, scratch_root, sh

from .io import write_json

TASK = Path(__file__).resolve().parents[2]
COMMON = TASK.parent / "common"
PY = sys.executable
OUT_NAME = "pa3_task3_outputs"
STATE: dict = {}
DEPS = {"data": [], "splits": ["data"], "scorer": ["data"], "signals": ["splits"], "offline": ["scorer", "signals"],
        "analysis": ["offline"], "adaptive": ["offline"], "compute": ["offline"], "live_eval": ["splits"],
        "qualitative": ["offline"], "report": ["data"]}


def _env(cores: list[int] | None = None) -> dict:
    e = {"PYTHONPATH": os.pathsep.join([str(TASK / "src"), str(COMMON)]), "TOKENIZERS_PARALLELISM": "false"}
    if cores:
        n = str(len(cores))
        e.update({"PA3_CPUS": ",".join(map(str, cores)), "OMP_NUM_THREADS": n, "MKL_NUM_THREADS": n,
                  "OPENBLAS_NUM_THREADS": n})
    return e


def setup_env(output_root: str, budget_hours: float = 11.25) -> dict:
    R = Path(output_root)
    R.mkdir(parents=True, exist_ok=True)
    os.environ["PA3_OUTPUT_ROOT"] = str(R)
    os.environ.update(_env())
    dl = Deadline(budget_hours)
    scratch = scratch_root("pa3_task3_scratch")
    prior = [p for p in discover_prior_roots(OUT_NAME) if Path(p).resolve() != R.resolve()]
    for p in prior:
        print(f"[setup] merging earlier-session outputs from {p}", flush=True)
        copytree_merge(Path(p), R, skip=("report_assets",))
    cores = sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else list(range(os.cpu_count() or 1))
    gpus = n_gpus()
    gen = cores[: max(1, len(cores) // 2)] if gpus and len(cores) >= 2 else []
    off = [c for c in cores if c not in gen] or cores
    STATE.update(deadline=dl, root=R, scratch=scratch, logs=R / "logs", steps={}, gen_cores=gen, off_cores=off,
                 gpus=gpus)
    info = {"output_root": str(R), "scratch": str(scratch), "gpus": gpus, "prior_roots": prior, "budget": str(dl),
            "generation_cores": gen, "offline_cores": off}
    write_json(R / "env.json", info)
    print(json.dumps(info, indent=1), flush=True)
    return info


def _status(step: str, state: str, **kw) -> None:
    STATE["steps"][step] = {"state": state, "time": time.strftime("%H:%M:%S"), **kw}
    write_json(STATE["root"] / "status.json", {"steps": STATE["steps"], "deadline": str(STATE["deadline"])})


def _marker(name: str) -> Path:
    return STATE["root"] / "_stages" / f"{name}.json"


def _cache_ok() -> bool:
    return all((STATE["scratch"] / "cache" / ds / "tokens.npz").exists() for ds in ("math", "mathvista"))


def _stage(name: str) -> bool:
    failed = [d for d in DEPS[name] if STATE["steps"].get(d, {}).get("state") in ("failed", "skipped")]
    if failed:
        _status(name, "skipped", reason=f"dependency failed or skipped: {failed}")
        return False
    if _marker(name).exists() and (name != "data" or _cache_ok()):
        _status(name, "done", note="already done")
        return True
    if not STATE["deadline"].fits(300, margin=120):
        _status(name, "skipped", reason="session budget")
        return False
    t0 = time.time()
    rc = sh([PY, "-m", "pa3ttc.stages", name], STATE["logs"] / f"{name}.log", env=_env(STATE["off_cores"]), cwd=TASK)
    ok = rc == 0 and _marker(name).exists()
    _status(name, "done" if ok else "failed", seconds=round(time.time() - t0), rc=rc)
    if not ok:
        print(f"!! stage {name} failed (rc={rc}); see {STATE['logs'] / (name + '.log')}", flush=True)
    return ok


def _launch_live():
    log = STATE["logs"] / "live_generation.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    fh = open(log, "a")
    env = {**os.environ, **_env(STATE["gen_cores"]), "CUDA_VISIBLE_DEVICES": "0"}
    p = subprocess.Popen([PY, "-m", "pa3ttc.live.run"], stdout=fh, stderr=subprocess.STDOUT, env=env, cwd=TASK)
    print(f"[live] generation started in the background on GPU0 (pid {p.pid}, cores {STATE['gen_cores']}, log {log})",
          flush=True)
    _status("live_generation", "running", pid=p.pid)
    return p, fh


def _wait_live(p, fh) -> int:
    kill_at = STATE["deadline"].end - 300
    last = 0.0
    while p.poll() is None:
        if time.time() > kill_at:
            p.kill()
            print("[live] session deadline reached -> killed (finished calls are kept; rerun to resume)", flush=True)
            break
        if time.time() - last > 600:
            tail = [l for l in Path(fh.name).read_text(errors="ignore").splitlines() if l.startswith("[live]")][-1:]
            print(f"[live] waiting... {tail}", flush=True)
            last = time.time()
        time.sleep(15)
    fh.close()
    rc = p.wait()
    state = {0: "done", 3: "paused"}.get(rc, "failed")
    _status("live_generation", state, rc=rc)
    return rc


def stage_auto(run_cpu_tests: bool = False, run_live: bool = True) -> dict:
    if run_cpu_tests:
        rc = sh([PY, "-m", "pytest", "-q", "-x", str(TASK / "tests")], STATE["logs"] / "cpu_tests.log", env=_env(),
                cwd=TASK)
        _status("cpu_tests", "done" if rc == 0 else "failed", rc=rc)
    if not _stage("data"):
        log = STATE["logs"] / "data.log"
        tail = log.read_text(errors="ignore").splitlines()[-60:] if log.exists() else []
        print("\n" + "!" * 100 + "\n!! DATA STAGE FAILED — nothing else can run. Last lines of logs/data.log:\n"
              + "\n".join(tail) + "\n" + "!" * 100, flush=True)
    _stage("splits")
    live = None
    live_done = (STATE["root"] / "live" / "status.json").exists() and json.loads(
        (STATE["root"] / "live" / "status.json").read_text()).get("state") == "done"
    if run_live and not live_done and STATE["steps"].get("splits", {}).get("state") == "done":
        if STATE["gpus"] >= 1:
            live = _launch_live()
        else:
            _status("live_generation", "skipped", reason="no GPU visible")
    for s in ("scorer", "signals", "offline", "analysis", "adaptive", "compute"):
        _stage(s)
    if live is not None:
        _wait_live(*live)
        if _marker("live_eval").exists():
            _marker("live_eval").unlink()       # re-evaluate with the new calls
    if (STATE["root"] / "live" / "responses.jsonl").exists():
        _stage("live_eval")
    for s in ("qualitative", "report"):
        if _marker(s).exists():
            _marker(s).unlink()                 # cheap; always rebuilt from the latest outputs
        _stage(s)
    states = {k: v["state"] for k, v in STATE["steps"].items()}
    failed = [k for k, v in states.items() if v == "failed"]
    state = "failed" if failed else ("done" if all(v == "done" for k, v in states.items() if k != "cpu_tests")
                                     else "incomplete")
    final = {"steps": states, "state": state, "failed": failed,
             "skipped": [k for k, v in states.items() if v == "skipped"]}
    _status("pipeline", state, failed=failed)
    return final


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/report/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/report/figures.py
"""Report figures (PNG) and the combined summary.md.  Every figure is built only from saved outputs, so this
step can be rerun on its own; a missing input skips that figure."""
from __future__ import annotations

import json
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

from ..config import odir
from ..io import df_to_md, read_json

SURF, INK, INK2, GRID, ORACLE = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df", "#8a8985"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
FIG_METHODS = {"math": ["first", "sc", "logp", "selfcert", "selfcert_rwc", "shortest"],
               "mathvista": ["first", "sc", "selfcert", "selfcert_rwc", "visconf", "visconf_rwc", "vision_rwc"]}
COLOR = {m: SERIES[i] for i, m in enumerate(["first", "sc", "selfcert", "selfcert_rwc", "visconf", "visconf_rwc",
                                              "vision_rwc", "logp"])}
COLOR["shortest"] = SERIES[4]
DSNAME = {"math": "MATH (Qwen2.5-3B-Instruct)", "mathvista": "MathVista (Qwen2.5-VL-3B-Instruct)"}


def _plt():
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "axes.edgecolor": GRID,
                         "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
                         "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.spines.top": False,
                         "axes.spines.right": False, "font.size": 10, "legend.frameon": False,
                         "lines.linewidth": 1.8, "lines.markersize": 5})
    return plt


def fig_dir() -> Path:
    return odir("report_assets", "figures")


def acc_curves(ds: str):
    from ..offline.runner import LABEL

    plt = _plt()
    s = pd.read_csv(odir("offline", ds) / "summary_eval.csv")
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
    for m in FIG_METHODS[ds]:
        x = s[s["method"] == m].sort_values("N")
        axes[0].plot(x["N"], x["acc"], marker="o", color=COLOR[m], label=LABEL[m])
        axes[1].plot(x["cost"], x["acc"], marker="o", color=COLOR[m], label=LABEL[m])
    o = s[s["method"] == "sc"].sort_values("N")
    axes[0].plot(o["N"], o["oracle"], ls="--", color=ORACLE, label="Oracle coverage (upper bound)")
    ad = odir("adaptive", ds) / "heldout.json"
    if ad.exists():
        h = read_json(ad)
        for fam, mk in (("agreement", "D"), ("quality", "s")):
            axes[1].plot(h[fam]["cost"], h[fam]["acc"], marker=mk, ls="none", ms=8, color=INK,
                         label=f"Adaptive ({fam})")
    axes[0].set_xscale("log", base=2)
    axes[0].set_xticks([1, 2, 4, 8, 16, 32], ["1", "2", "4", "8", "16", "32"])
    axes[0].set_xlabel("Candidate budget N")
    axes[1].set_xscale("log")
    axes[1].set_xlabel("Mean generated tokens per question (offline cost proxy)")
    for a in axes:
        a.set_ylabel("Accuracy (held-out)")
    h0, l0 = axes[0].get_legend_handles_labels()
    h1, l1 = axes[1].get_legend_handles_labels()
    extra = [(h, l) for h, l in zip(h1, l1) if l not in l0]
    fig.legend(h0 + [h for h, _ in extra], l0 + [l for _, l in extra], loc="lower center", ncol=4, fontsize=8)
    fig.suptitle(f"{DSNAME[ds]}: accuracy vs budget and vs token cost", color=INK)
    fig.tight_layout(rect=(0, 0.16, 1, 1))
    fig.savefig(fig_dir() / f"{ds}_accuracy_curves.png", dpi=150)
    plt.close(fig)


def decomposition(ds: str):
    plt = _plt()
    s = pd.read_csv(odir("offline", ds) / "summary_eval.csv")
    meths = ["sc", "selfcert_rwc"] + (["visconf_rwc"] if ds == "mathvista" else [])
    fig, axes = plt.subplots(1, len(meths), figsize=(4.2 * len(meths), 3.8), sharey=True)
    from ..offline.runner import LABEL

    for a, m in zip(np.atleast_1d(axes), meths):
        x = s[s["method"] == m].sort_values("N")
        lab = [str(n) for n in x["N"]]
        b = np.zeros(len(x))
        for col, c, name in (("acc", SERIES[2], "Correct"), ("missed", SERIES[3], "Correct available, missed"),
                             ("no_correct", ORACLE, "No correct candidate")):
            a.bar(lab, x[col], bottom=b, color=c, label=name, edgecolor=SURF, linewidth=1.5)
            b += x[col].to_numpy()
        a.set_title(LABEL[m], fontsize=10)
        a.set_xlabel("N")
    np.atleast_1d(axes)[0].set_ylabel("Fraction of held-out questions")
    np.atleast_1d(axes)[0].legend(fontsize=8, loc="lower left")
    fig.suptitle(f"{DSNAME[ds]}: sampling vs selection failures", color=INK)
    fig.tight_layout()
    fig.savefig(fig_dir() / f"{ds}_failure_decomposition.png", dpi=150)
    plt.close(fig)


def marginal():
    plt = _plt()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    for a, ds in zip(axes, ("math", "mathvista")):
        m = pd.read_csv(odir("compute", ds) / "marginal_samples.csv")
        a.plot(m["n"], m["p_new_group"], color=SERIES[0], label="adds a new answer group")
        a.plot(m["n"], m["p_new_answered_group"], color=SERIES[1], label="adds a new non-missing answer")
        a.plot(m["n"], m["p_first_correct_appears"], color=SERIES[2], label="first correct answer appears")
        a.set_title(DSNAME[ds], fontsize=10)
        a.set_xlabel("Sample index n (revealed order)")
    axes[0].set_ylabel("Probability (held-out, averaged over orders)")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(fig_dir() / "marginal_value_of_samples.png", dpi=150)
    plt.close(fig)


def stopping():
    plt = _plt()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
    for a, ds in zip(axes, ("math", "mathvista")):
        h = read_json(odir("adaptive", ds) / "heldout.json")
        ns = ["4", "8", "16", "32"]
        w = 0.38
        for i, fam in enumerate(("agreement", "quality")):
            d = h[fam]["stop_distribution"]
            a.bar(np.arange(4) + (i - 0.5) * w, [d.get(n, d.get(int(n), 0)) for n in ns], width=w - 0.04,
                  color=SERIES[i], label=f"{fam} (τ={h[fam]['setting']['tau']}, γ={h[fam]['setting']['gamma']})")
        a.set_xticks(np.arange(4), ns)
        a.set_xlabel("Rollouts consumed at stop")
        a.set_title(DSNAME[ds], fontsize=10)
        a.legend(fontsize=8)
    axes[0].set_ylabel("Fraction of (question, order) runs")
    fig.tight_layout()
    fig.savefig(fig_dir() / "adaptive_stopping_distribution.png", dpi=150)
    plt.close(fig)


def quartiles():
    plt = _plt()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for a, ds in zip(axes, ("math", "mathvista")):
        t = pd.read_csv(odir("analysis", ds) / "associations.csv")
        a.barh(t["signal"], t["top_minus_bottom_quartile_mixed"], color=SERIES[0])
        a.axvline(0, color=INK2, lw=0.8)
        a.set_title(DSNAME[ds], fontsize=10)
        a.set_xlabel("P(correct | top quartile) - P(correct | bottom quartile)\n(mixed held-out questions, N=32)")
    fig.tight_layout()
    fig.savefig(fig_dir() / "score_quartile_gaps.png", dpi=150)
    plt.close(fig)


def conf_vision():
    plt = _plt()
    df = pd.read_parquet(odir("analysis", "mathvista") / "conf_vision_per_rollout.parquet")
    perq = df.groupby("qid")["rho_c_v"].mean().dropna()
    fig = plt.figure(figsize=(11, 4.4))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.4])
    ax_h = fig.add_subplot(gs[:, 0])
    ax_c = fig.add_subplot(gs[0, 1])
    ax_v = fig.add_subplot(gs[1, 1], sharex=ax_c)
    ax_h.hist(perq, bins=40, color=SERIES[0], edgecolor=SURF)
    ax_h.axvline(0, color=INK2, lw=0.8)
    ax_h.set_xlabel("Within-rollout corr(c_t, v_t), mean per question")
    ax_h.set_ylabel("Held-out questions")
    tj = read_json(odir("analysis", "mathvista") / "trajectories.json")["series"][:2]
    for i, s in enumerate(tj):
        x = np.arange(len(s["c_roll"])) * s["step"]
        lab = f"{s['which'].replace('_', ' ')} rollout ({'correct' if s['correct'] else 'wrong'})"
        ax_c.plot(x, s["c_roll"], color=SERIES[i], label=lab)
        ax_v.plot(x, s["v_roll"], color=SERIES[i], label=lab)
    ax_c.set_ylabel("calibrated c_t")
    ax_v.set_ylabel("visual share v_t")
    ax_v.set_xlabel("Generated token position")
    if tj:
        ax_c.set_title(f"Illustrative question {tj[0]['qid']} ({tj[0]['bucket'].replace('_', ' ')}); "
                       "rolling mean, window 16", fontsize=9)
        ax_c.legend(fontsize=7, loc="lower right")
    fig.tight_layout()
    fig.savefig(fig_dir() / "mathvista_confidence_vision.png", dpi=150)
    plt.close(fig)


def live():
    plt = _plt()
    s = pd.read_csv(odir("live") / "summary.csv")
    s = s[s["strategy"].isin(["single", "parallel", "refine"])]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
    for i, (_, r) in enumerate(s.iterrows()):
        for a, col, lab in ((axes[0], "mean_output_tokens", "Mean output tokens per question"),
                            (axes[1], "mean_seconds", "Mean seconds per question (T4, batch 1)")):
            a.plot(r[col], r["accuracy"], marker="o", ms=9, ls="none", color=SERIES[i], label=r["strategy"])
            a.set_xlabel(lab)
    axes[0].set_ylabel("Accuracy (50 MATH questions)")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(fig_dir() / "live_strategies.png", dpi=150)
    plt.close(fig)


def summary_md() -> None:
    R = odir()
    parts = ["# PA3 Task 3 — results summary\n"]
    for rel in ["offline/math/summary_eval.md", "offline/mathvista/summary_eval.md", "analysis/math/associations.md",
                "analysis/mathvista/associations.md", "adaptive/math/heldout.md", "adaptive/mathvista/heldout.md",
                "compute/math/compute.md", "compute/mathvista/compute.md", "live/summary.md"]:
        p = R / rel
        if p.exists():
            parts.append(p.read_text())
    for rel in ["answers/scorer_manifest.json", "analysis/mathvista/conf_vision_summary.json"]:
        p = R / rel
        if p.exists():
            j = json.loads(p.read_text())
            j.pop("rules", None)
            parts.append(f"\n## {rel}\n\n```json\n{json.dumps(j, indent=1)[:6000]}\n```\n")
    for ds in ("math", "mathvista"):
        p = R / "offline" / ds / "bootstrap.csv"
        if p.exists():
            b = pd.read_csv(p)
            b = b[b["N"].isin([8, 32])][["N", "A", "B", "a", "b", "diff", "lo", "hi"]]
            parts.append(f"\n## {ds}: paired bootstrap (held-out, 1,000 question resamples), N=8 and N=32\n\n" + df_to_md(b))
    tdir = odir("report_assets", "tables")
    for rel in ["offline/math/summary_eval.md", "offline/mathvista/summary_eval.md", "adaptive/math/heldout.md",
                "adaptive/mathvista/heldout.md", "live/summary.md"]:
        p = R / rel
        if p.exists():
            (tdir / rel.replace("/", "_")).write_text(p.read_text())
    (odir("report_assets") / "summary.md").write_text("\n".join(parts))


def run(cfg: dict) -> dict:
    done, failed, skipped = [], {}, []
    jobs = [("acc_math", lambda: acc_curves("math")), ("acc_mathvista", lambda: acc_curves("mathvista")),
            ("decomp_math", lambda: decomposition("math")), ("decomp_mathvista", lambda: decomposition("mathvista")),
            ("marginal", marginal), ("stopping", stopping), ("quartiles", quartiles), ("conf_vision", conf_vision),
            ("live", live), ("summary", summary_md)]
    for name, fn in jobs:
        try:
            fn()
            done.append(name)
        except FileNotFoundError as ex:          # an upstream stage did not run: nothing to draw
            skipped.append(f"{name} (missing {Path(ex.filename or '').name})")
        except Exception as ex:  # noqa: BLE001
            failed[name] = f"{type(ex).__name__}: {ex}"
            traceback.print_exc()
    print(f"[report] done={done} skipped={skipped} failed={list(failed)}", flush=True)
    return {"done": done, "skipped": skipped, "failed": failed}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/report/qualitative.py
"""Representative cases with a stated, seeded selection rule (default_rng(6304) over sorted question IDs).

1. Selection disagreement (held-out, N = 32, order 0): self-consistency vs the quality-aware RWC rule
   (self-certainty + RWC for MATH, VisConf + RWC for MathVista); 2 questions where only the quality rule is
   right and 2 where only SC is right.
2. Revision (live): up to 3 wrong->correct and 3 correct->wrong transitions, lowest live index first.
3. Premature stopping (held-out): 2 incorrect early stops per dataset and controller that fixed-32 SC would have
   recovered (order-level cases).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..config import odir
from ..data.load import Pool
from ..io import read_jsonl, write_json

QRULE = {"math": "selfcert_rwc", "mathvista": "visconf_rwc"}


def _tail(s, n=700) -> str:
    s = "" if s is None else str(s)
    return s if len(s) <= n else "…" + s[-n:]


def _head(s, n=500) -> str:
    s = "" if s is None else str(s)
    return s if len(s) <= n else s[:n] + "…"


def _groups_line(g: pd.DataFrame, qid: str) -> str:
    x = g[g["qid"] == qid].sort_values("size", ascending=False)
    return ", ".join(f"`{a}`×{n}{' ✓' if c else ''}" for a, n, c in zip(x["answer"], x["size"], x["correct"]))


def run(cfg: dict) -> dict:
    rng = np.random.default_rng(cfg["seed"])
    md = [f"# Qualitative cases\n\nSelection rule:\n\n{__doc__.strip()}\n"]
    out: dict = {"selection": [], "revision": [], "stopping": []}
    for ds in ("math", "mathvista"):
        pool = Pool(ds)
        texts = pool.texts().set_index(["qid", "rollout_id"])["text"]
        g = pd.read_parquet(odir("answers", ds) / "groups.parquet")
        po = pd.read_parquet(odir("offline", ds) / "per_order.parquet", filters=[("N", "==", 32), ("order", "==", 0)])
        po = po[(po["split"] == "eval") & po["method"].isin(["sc", QRULE[ds]])]
        w = po.pivot_table(index="qid", columns="method", values="correct", observed=True)
        rows = po.set_index(["qid", "method"])
        qtext = pool.questions.set_index("qid")
        md.append(f"\n## {ds}: selection disagreement at N=32 (SC vs {QRULE[ds]})\n")
        for tag, mask in (("quality_right_sc_wrong", (w[QRULE[ds]] == 1) & (w["sc"] == 0)),
                          ("sc_right_quality_wrong", (w[QRULE[ds]] == 0) & (w["sc"] == 1))):
            ids = sorted(w.index[mask].tolist())
            pick = rng.choice(ids, size=min(2, len(ids)), replace=False).tolist() if ids else []
            md.append(f"\n### {tag} ({len(ids)} held-out questions in this bucket; showing {len(pick)})\n")
            for qid in pick:
                case = {"dataset": ds, "bucket": tag, "qid": qid, "gold": qtext.loc[qid, "gold"],
                        "groups": _groups_line(g, qid)}
                md.append(f"\n**{qid}** — gold `{case['gold']}`  \nProblem: {_head(qtext.loc[qid, 'problem'])}  \n"
                          f"Answer groups (order 0): {case['groups']}\n")
                for m in ("sc", QRULE[ds]):
                    r = rows.loc[(qid, m)]
                    case[m] = {"answer": r["answer"], "correct": bool(r["correct"]), "rollout_id": r["rollout_id"]}
                    txt = texts.get((qid, r["rollout_id"]), "")
                    md.append(f"- {m}: `{r['answer']}` ({'correct' if r['correct'] else 'wrong'}), rollout "
                              f"{r['rollout_id']} — tail:\n\n  > {_tail(txt, 400).replace(chr(10), ' ')}\n")
                out["selection"].append(case)
        for fam in ("agreement", "quality"):
            p = odir("adaptive", ds) / "recoverable_early_stops.csv"
            if not p.exists():
                continue
            rc = pd.read_csv(p, dtype={"qid": str})
            rc = rc[rc["family"] == fam]
            if rc.empty:
                continue
            pick = rc.iloc[sorted(rng.choice(len(rc), size=min(2, len(rc)), replace=False).tolist())]
            md.append(f"\n## {ds}: premature stops ({fam} controller) recovered by fixed-32 SC ({len(rc)} order-level cases)\n")
            for _, r in pick.iterrows():
                ga = g[(g["qid"] == r["qid"])].set_index("group")["answer"]
                case = {"dataset": ds, "family": fam, "qid": r["qid"], "order": int(r["order"]),
                        "stop_n": int(r["stop_n"]), "stopped_answer": ga.get(r["stopped_group"]),
                        "fixed32_answer": ga.get(r["fixed32_group"]), "p_correct_pool": r["p_correct_pool"],
                        "gold": qtext.loc[r["qid"], "gold"]}
                out["stopping"].append(case)
                md.append(f"- **{r['qid']}** order {case['order']}: stopped at n={case['stop_n']} with "
                          f"`{case['stopped_answer']}`; fixed-32 SC `{case['fixed32_answer']}` (gold `{case['gold']}`, "
                          f"pool correct fraction {case['p_correct_pool']:.2f}); 32-pool groups: {_groups_line(g, r['qid'])}\n")
    lp = odir("live") / "per_question.csv"
    if lp.exists():
        lq = pd.read_csv(lp, dtype={"qid": str})
        for c in ("initial_correct", "refine_correct"):
            lq[c] = lq[c].astype(str).str.lower().eq("true")
        calls = {(r["qid"], r["slot"]): r for r in read_jsonl(odir("live") / "responses.jsonl")}
        md.append("\n## Live: revision transitions\n")
        for tag, mask in (("wrong_to_correct", ~lq["initial_correct"] & lq["refine_correct"]),
                          ("correct_to_wrong", lq["initial_correct"] & ~lq["refine_correct"])):
            sel = lq[mask].sort_values("live_idx").head(3)
            md.append(f"\n### {tag} ({int(mask.sum())} questions; showing {len(sel)})\n")
            for _, r in sel.iterrows():
                c1, c4 = calls[(r["qid"], 1)], calls[(r["qid"], 4)]
                out["revision"].append({"qid": r["qid"], "tag": tag, "gold": r["gold"],
                                        "initial": r["initial_answer"], "revised": r["refine_answer"]})
                md.append(f"\n**{r['qid']}** gold `{r['gold']}`: initial `{r['initial_answer']}` → revised "
                          f"`{r['refine_answer']}` (initial truncated: {c1['truncated']}, revision truncated: "
                          f"{c4['truncated']})\n\n  > revision tail: {_tail(c4['response'], 500).replace(chr(10), ' ')}\n")
    d = odir("qualitative")
    (d / "cases.md").write_text("\n".join(md))
    write_json(d / "cases.json", out)
    return {k: len(v) for k, v in out.items()}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/select/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/select/rules.py
"""Selection rules over one candidate prefix (arrays in candidate order: index 0 = first revealed).

Ties: highest/lowest score -> earliest candidate; answer votes -> the answer whose earliest supporting
candidate comes first.  RWC: average ranks among the N candidates (rank 1 = best, ties averaged),
w_i = (N - r_i)/(N - 1), W(a) = sum of w over a's candidates.  At N = 1 every rule returns the only candidate.
NaN scores (e.g. T = 0) count as -inf.  NO_ANSWER (-1) is an ordinary answer group in every vote.
"""
from __future__ import annotations

import numpy as np

TIE_EPS = 1e-9


def clean(scores) -> np.ndarray:
    s = np.asarray(scores, dtype=np.float64)
    return np.where(np.isnan(s), -np.inf, s)


def argmax_earliest(scores) -> int:
    return int(np.argmax(clean(scores)))


def argmin_earliest(vals) -> int:
    v = np.asarray(vals, dtype=np.float64)
    return int(np.argmin(np.where(np.isnan(v), np.inf, v)))


def vote(groups, weights=None) -> tuple[int, int, dict]:
    """-> (winning group, index of its earliest supporter, totals in first-appearance order)."""
    tot: dict[int, float] = {}
    first: dict[int, int] = {}
    for i, g in enumerate(groups):
        g = int(g)
        if g not in tot:
            tot[g] = 0.0
            first[g] = i
        tot[g] += 1.0 if weights is None else float(weights[i])
    best = None
    for g in tot:                       # first-appearance order => ties go to the earliest supporter
        if best is None or tot[g] > tot[best] + TIE_EPS:
            best = g
    return best, first[best], tot


def avg_rank_desc(scores) -> np.ndarray:
    x = -clean(scores)
    n = len(x)
    order = np.argsort(x, kind="mergesort")
    xs = x[order]
    ranks = np.empty(n)
    i = 0
    while i < n:
        j = i
        while j + 1 < n and xs[j + 1] == xs[i]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2 + 1
        i = j + 1
    return ranks


def rwc_weights(scores) -> np.ndarray:
    n = len(scores)
    if n == 1:
        return np.ones(1)
    return (n - avg_rank_desc(scores)) / (n - 1)


def rwc(groups, scores) -> tuple[int, int, dict]:
    if len(groups) == 1:
        return int(groups[0]), 0, {int(groups[0]): 1.0}
    return vote(groups, rwc_weights(scores))


def vote_margin(tot: dict) -> float:
    """(largest - second largest) / total weight; runner-up 0 when only one answer group exists."""
    v = sorted(tot.values(), reverse=True)
    s = sum(v)
    if s <= 0:
        return 0.0
    return (v[0] - (v[1] if len(v) > 1 else 0.0)) / s


def select(method: str, groups, scores=None, lengths=None) -> int:
    """Index (in candidate order) of the rollout a method returns.  For vote rules this is the earliest
    supporter of the winning answer (only its answer matters)."""
    n = len(groups)
    if n == 1 or method == "first":
        return 0
    if method == "sc":
        return vote(groups)[1]
    if method == "shortest":
        return argmin_earliest(lengths)
    if method.endswith("_rwc"):
        return rwc(groups, scores)[1]
    return argmax_earliest(scores)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/signals/__init__.py



In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/signals/rollout.py
"""Order-independent rollout signals: T_i, L_i = mean log p(sampled token), K_i = mean KL(U || p),
V_i = mean visual engagement v_t = A_img / (A_img + A_prompt + A_generated) over tokens where it is defined.
T_i = 0 gives NaN scores, which selection treats as -inf (approved decision 10)."""
from __future__ import annotations

import numpy as np


def seg_mean(vals: np.ndarray, offsets: np.ndarray) -> np.ndarray:
    lens = np.diff(offsets)
    x = vals.astype(np.float64)
    ok = np.isfinite(x)
    P = np.concatenate([[0.0], np.cumsum(np.where(ok, x, 0.0))])
    N = np.concatenate([[0], np.cumsum(ok)])
    s = P[offsets[1:]] - P[offsets[:-1]]
    n = N[offsets[1:]] - N[offsets[:-1]]
    out = np.full(len(lens), np.nan)
    m = n > 0
    out[m] = s[m] / n[m]
    return out


def rollout_scores(pool) -> dict[str, np.ndarray]:
    Q, R = pool.Q, pool.R
    out = {"T": pool.T.astype(np.float64),
           "L": seg_mean(pool.logp, pool.offsets).reshape(Q, R),
           "K": seg_mean(pool.kl, pool.offsets).reshape(Q, R)}
    if pool.v is not None:
        out["V"] = seg_mean(pool.v, pool.offsets).reshape(Q, R)
    for k in ("L", "K", "V"):
        if k in out:
            out[k][pool.T == 0] = np.nan
    return out


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/signals/stage.py
"""Signals stage: per-rollout L, K, V, T for both datasets; prefix-calibrated VisConf / tokenwise fusion for
MathVista at every budget N >= 2 and every saved order (scores stored in candidate order)."""
from __future__ import annotations

import time

import numpy as np
import pandas as pd

from ..config import odir
from ..data.load import Pool
from ..io import write_json
from .rollout import rollout_scores
from .visconf import VisConfIndex


def run(cfg: dict) -> dict:
    info = {}
    for ds in ("math", "mathvista"):
        pool = Pool(ds)
        sc = rollout_scores(pool)
        d = odir("signals", ds)
        np.savez(d / "rollout_scores.npz", **sc)
        df = pd.DataFrame({"qid": np.repeat(pool.qids, pool.R), "rollout_id": pool.rollout_ids.ravel(),
                           **{k: v.ravel() for k, v in sc.items()}})
        df.to_parquet(d / "rollout_scores.parquet", index=False)
        info[ds] = {k: {"mean": float(np.nanmean(v)), "nan": int(np.isnan(v).sum())} for k, v in sc.items()}
        if ds != "mathvista":
            continue
        t0 = time.time()
        orders = np.load(odir("splits") / f"{ds}_orders.npy")
        idx = VisConfIndex(pool.offsets, pool.kl, pool.v)
        Q, K, R = orders.shape
        V = sc["V"]
        out = {}
        for N in [n for n in cfg["budgets"] if n >= 2]:
            Z = np.full((Q, K, N), np.nan, np.float64)
            C, Zt = Z.copy(), Z.copy()
            A, D = np.full((Q, K), np.nan), np.full((Q, K), np.nan)
            for q in range(Q):
                if N == R:   # same candidate set for every order: compute once, then permute
                    pos = np.arange(R)
                    c, z, zt, a, dd = idx.scores(q * R + pos, V[q, pos])
                    for k in range(K):
                        o = orders[q, k]
                        C[q, k], Z[q, k], Zt[q, k], A[q, k], D[q, k] = c[o], z[o], zt[o], a, dd
                    continue
                for k in range(K):
                    pos = orders[q, k, :N]
                    C[q, k], Z[q, k], Zt[q, k], A[q, k], D[q, k] = idx.scores(q * R + pos, V[q, pos])
            out.update({f"C_{N}": C, f"Z_{N}": Z, f"Ztok_{N}": Zt, f"a_{N}": A, f"d_{N}": D})
        np.savez(d / "visconf.npz", **out)
        info[ds]["visconf_seconds"] = round(time.time() - t0, 1)
    write_json(odir("signals") / "summary.json", info)
    print(f"[signals] {info}", flush=True)
    return info


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/signals/visconf.py
"""VisConf with candidate-set calibration, computed exactly but fast.

For a candidate set G: a = Q.05, b = Q.95 of the pooled token KL values of G (linear interpolation), d =
max(b - a, 1e-8), c_t = clip((k_t - a) / d, 0, 1), C_i = mean_t c_t, Z_i = V_i * C_i, and the tokenwise
ablation Z_tok_i = mean_t c_t v_t (over tokens with defined v_t).

Each rollout's KL values are stored sorted, with its own prefix sums of k, v, k*v and of the valid-v count.
c_t is 0 for k <= a, 1 for k >= a + d and (k - a)/d in between, so with i0 = #(k <= a) and i1 = #(k < a + d):
  sum_t c_t   = (S_k[i1] - S_k[i0] - a (i1 - i0)) / d + (T - i1)
  sum_t c_t v = (S_kv[i1] - S_kv[i0] - a (S_v[i1] - S_v[i0])) / d + (S_v[T] - S_v[i1])
Only the tokens of G enter the percentiles, so unseen rollouts can never change a score (tested).
"""
from __future__ import annotations

import numpy as np


class VisConfIndex:
    def __init__(self, offsets: np.ndarray, kl: np.ndarray, v: np.ndarray):
        self.off = offsets.astype(np.int64)
        n = len(self.off) - 1
        seg = np.repeat(np.arange(n), np.diff(self.off))
        k = kl.astype(np.float64)
        order = np.lexsort((k, seg))
        self.sk = k[order]
        vv = v.astype(np.float64)[order]
        ok = np.isfinite(vv)
        v0 = np.where(ok, vv, 0.0)
        k0 = np.where(np.isfinite(self.sk), self.sk, 0.0)
        # per-rollout prefix sums (each segment starts from its own 0, so a rollout's sums never depend on the
        # values of any other rollout — not even through floating-point rounding); segment r lives at
        # [off[r] + r, off[r + 1] + r]
        m = len(self.sk) + n
        self.Pk, self.Pv, self.Pkv = np.zeros(m), np.zeros(m), np.zeros(m)
        self.Pn = np.zeros(m, dtype=np.int64)
        for r in range(n):
            s, e = self.off[r], self.off[r + 1]
            b = s + r
            if e > s:
                self.Pk[b + 1:b + 1 + e - s] = np.cumsum(k0[s:e])
                self.Pv[b + 1:b + 1 + e - s] = np.cumsum(v0[s:e])
                self.Pkv[b + 1:b + 1 + e - s] = np.cumsum(k0[s:e] * v0[s:e])
                self.Pn[b + 1:b + 1 + e - s] = np.cumsum(ok[s:e])

    def calibration(self, rows) -> tuple[float, float]:
        parts = [self.sk[self.off[r]:self.off[r + 1]] for r in rows]
        pooled = np.concatenate(parts) if parts else np.zeros(0)
        pooled = pooled[np.isfinite(pooled)]
        if pooled.size == 0:
            return np.nan, np.nan
        a, b = np.percentile(pooled, [5, 95])
        return float(a), float(max(b - a, 1e-8))

    def scores(self, rows, V: np.ndarray | None = None):
        """rows: global rollout indices of the candidate set (any order).  V: their V_i (same order).
        -> (C, Z, Z_tok, a, d) with one entry per row."""
        rows = np.asarray(rows)
        a, d = self.calibration(rows)
        C = np.full(len(rows), np.nan)
        Zt = np.full(len(rows), np.nan)
        if np.isfinite(a):
            u = a + d
            for j, r in enumerate(rows):
                s, e = int(self.off[r]), int(self.off[r + 1])
                T = e - s
                if T == 0:
                    continue
                b = s + int(r)
                seg = self.sk[s:e]
                i0 = int(np.searchsorted(seg, a, "right"))
                i1 = int(np.searchsorted(seg, u, "left"))
                C[j] = ((self.Pk[b + i1] - self.Pk[b + i0] - a * (i1 - i0)) / d + (T - i1)) / T
                nv = self.Pn[b + T]
                if nv > 0:
                    num = ((self.Pkv[b + i1] - self.Pkv[b + i0] - a * (self.Pv[b + i1] - self.Pv[b + i0])) / d
                           + (self.Pv[b + T] - self.Pv[b + i1]))
                    Zt[j] = num / nv
        Z = C * V if V is not None else np.full(len(rows), np.nan)
        return C, Z, Zt, a, d


def naive_scores(segments_k: list[np.ndarray], segments_v: list[np.ndarray]):
    """Direct implementation of the formulas (tests only)."""
    pooled = np.concatenate([s for s in segments_k]) if segments_k else np.zeros(0)
    a, b = np.percentile(pooled, [5, 95])
    d = max(b - a, 1e-8)
    C, Z, Zt = [], [], []
    for k, v in zip(segments_k, segments_v):
        if len(k) == 0:
            C.append(np.nan), Z.append(np.nan), Zt.append(np.nan)
            continue
        c = np.clip((k - a) / d, 0, 1)
        ok = np.isfinite(v)
        Ci = c.mean()
        Vi = v[ok].mean() if ok.any() else np.nan
        C.append(Ci)
        Z.append(Ci * Vi)
        Zt.append((c[ok] * v[ok]).mean() if ok.any() else np.nan)
    return np.array(C), np.array(Z), np.array(Zt), a, d


def token_c(k: np.ndarray, a: float, d: float) -> np.ndarray:
    return np.clip((k.astype(np.float64) - a) / d, 0.0, 1.0)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/src/pa3ttc/stages.py
"""Command-line entry for one pipeline stage:  python -m pa3ttc.stages <stage>

Stages: data, splits, scorer, signals, offline, analysis, adaptive, compute, live_eval, qualitative, report.
Each stage pins itself to the cores in PA3_CPUS, reads earlier stages' files and writes _stages/<stage>.json
when it finishes.
"""
from __future__ import annotations

import os
import shutil
import sys
import time

import numpy as np

from .config import D_INDEX, DATASETS, cache_dir, load_config, odir
from .io import pin_cpus, read_json, write_json


def stage_data(cfg):
    from .config import scratch
    from .data.discover import find_readme, find_tables
    from .data.load import load_dataset

    tables = find_tables(cfg["data_root"], cfg.get("data_fallback_root", "/kaggle/input"), scratch=scratch(),
                         hf_repo=cfg.get("data_hf_repo"), listing_path=odir("data") / "input_listing.txt",
                         url=cfg.get("data_url"))
    for ds in DATASETS:
        print(f"[data] {ds}: {tables[ds]}", flush=True)
    rd = find_readme(tables)
    if rd:
        shutil.copy(rd, odir("data") / "README_supplied.md")
        print(f"[data] README found: {rd}", flush=True)
    out = {"tables": tables, "readme": str(rd) if rd else None}
    for ds in DATASETS:
        rep = load_dataset(tables[ds], ds, cfg)
        out[ds] = {"n_questions": rep["n_questions"], "alignment": rep["alignment"], "columns": rep["columns"]}
        print(f"[data] {ds}: {out[ds]}", flush=True)
    return out


def stage_splits(cfg):
    from .data.load import Pool
    from .data.splits import make_orders, make_split

    out = {}
    for ds in DATASETS:
        pool = Pool(ds)
        if pool.Q != cfg["n_questions_expected"]:
            print(f"!! {ds}: {pool.Q} questions (expected {cfg['n_questions_expected']})", flush=True)
        sp = make_split(pool.qids, cfg["seed"], cfg["n_dev"])
        write_json(odir("splits") / f"{ds}.json", sp)
        orders = make_orders(pool.Q, D_INDEX[ds], cfg["seed"], cfg["n_orders"], pool.R)
        np.save(odir("splits") / f"{ds}_orders.npy", orders)
        out[ds] = {"dev": len(sp["dev"]), "eval": len(sp["eval"]), "first_eval_ids": sp["eval"][:5]}
    pool = Pool("math")
    live_ids = read_json(odir("splits") / "math.json")["eval"][: cfg["n_live"]]
    q = pool.questions.set_index("qid")
    if "problem" not in q.columns or q["problem"].isna().all() or (q["problem"].astype(str) == "None").all():
        raise KeyError("no problem-text column found in the MATH generations; set columns.math.problem")
    qs = [{"live_idx": i, "qid": x, "problem": str(q.loc[x, "problem"]), "gold": q.loc[x, "gold"]}
          for i, x in enumerate(live_ids)]
    write_json(odir("live") / "questions.json", qs)
    write_json(odir("splits") / "live_ids.json", live_ids)
    out["live"] = {"n": len(qs), "first": live_ids[:3]}
    return out


def stage_scorer(cfg):
    from .answers.scorer import run
    return run(cfg)


def stage_signals(cfg):
    from .signals.stage import run
    return run(cfg)


def stage_offline(cfg):
    from .offline.runner import run
    return run(cfg)


def stage_analysis(cfg):
    from .analysis import conf_vision, signal_assoc

    out = {ds: signal_assoc.run_dataset(ds, cfg) for ds in DATASETS}
    out["conf_vision"] = conf_vision.run(cfg)
    return out


def stage_adaptive(cfg):
    from .adaptive.tune import run
    return run(cfg)


def stage_compute(cfg):
    from .analysis.compute_alloc import run
    return run(cfg)


def stage_live_eval(cfg):
    from .live.evaluate import run
    return run(cfg)


def stage_qualitative(cfg):
    from .report.qualitative import run
    return run(cfg)


def stage_report(cfg):
    from .report.figures import run
    return run(cfg)


STAGES = {n[6:]: f for n, f in dict(globals()).items() if n.startswith("stage_")}


def data_cached() -> bool:
    return all((cache_dir(ds) / "tokens.npz").exists() for ds in DATASETS)


def main(name: str) -> int:
    cores = pin_cpus()
    cfg = load_config()
    t0 = time.time()
    print(f"[stage {name}] cores={cores}", flush=True)
    if os.environ.get("PA3_TEST_FAIL_STAGE") == name:      # used only by tests/test_orchestrate.py
        raise RuntimeError(f"injected failure in stage {name}")
    res = STAGES[name](cfg)
    write_json(odir("_stages") / f"{name}.json", {"seconds": round(time.time() - t0, 1), "result": res})
    print(f"[stage {name}] done in {time.time() - t0:.0f}s", flush=True)
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv[1]))


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/scripts/build_notebook.py
"""Generate task_3/notebooks/pa3_task3_kaggle.ipynb — a self-contained Kaggle notebook.

Every source file of task_3/ (configs, src, scripts, tests, README) and the shared common/pa3common modules it uses
is embedded as a `%%writefile` cell, followed by an integrity check (sha256 of each written file), dependency
installation, the AUTO pipeline, a results view and the download cell.  Setup errors are caught so the last cell
(zip + download link) always runs.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

TASK = Path(__file__).resolve().parents[1]
REPO = TASK.parent
CODE_DIR = "/kaggle/working/ATML-PA3"
COMMON_FILES = ["__init__.py", "kaggle.py"]

INTRO = r"""# ATML PA3 · Task 3 — Test-Time Compute: Sampling, Selection and Adaptive Inference
**CS-6304 / EE-5102 · Danish Rasool (28100233)** · self-contained notebook (all code is written to disk by the cells below)

### Before running
1. **Add Input → the `ATML-PA3-Data` dataset** (expected at `/kaggle/input/atml-pa3-data`; any folder under
   `/kaggle/input` that contains the two model folders is found automatically).
2. Always run **from the top** (Run All, or Restart & Run All after a session restart): later cells use the
   SETTINGS cell's variables.
3. **Settings → Accelerator: GPU T4 ×2** (only GPU0 is used, for the 50-question live run), **Internet: On**
   (Qwen2.5-3B-Instruct download; MathVista metadata from the Hub only if the supplied files lack it).
4. **Save Version → Save & Run All (Commit)**.

### What happens (`AUTO`)
data join + alignment checks → dev/held-out splits (seed 6304) + 10 candidate orders per question → **live generation
starts in the background on GPU0** (single 3,072 / parallel 3×1,024 / refinement 1,024+2,048; FP16, batch 1,
pinned to its own CPU cores) → in parallel on the other cores: answer grouping + grading (math-verify / MathVista
rules, frozen with a manifest) → signals (log-prob, self-certainty, visual engagement, prefix-calibrated VisConf) →
all selection rules at N ∈ {1,2,4,8,16,32} → signal analysis → adaptive stopping (dev-tuned, frozen, held-out) →
compute allocation → live evaluation → qualitative cases → figures/tables. The last cell zips the outputs into
`/kaggle/working/results_task3.zip` and prints a **download link** (also when something failed part-way).

If `status.json` shows `live_generation: paused`, make this version's output a dataset, attach it and run again:
finished stages and live calls are kept and only the remaining calls are generated.
"""

SETTINGS = f'''# ============================ SETTINGS ============================
import os, time, json
from pathlib import Path
os.environ.setdefault("PA3_SESSION_START", str(time.time()))   # the session budget starts now
BUDGET_HOURS = 11.25                          # stop cleanly before Kaggle's 12 h limit
DATA_ROOT = "/kaggle/input/atml-pa3-data"     # searched recursively; /kaggle/input is the fallback
DATA_URL = ""        # only if the data is NOT attached: a Google Drive / direct .zip link to ATML-PA3-Data
DATA_HF_REPO = ""    # ... or a Hugging Face dataset repo id holding it
RUN_LIVE = True                               # Step 4 live generation on GPU0
RUN_CPU_TESTS = False                         # run the CPU test suite first (~1 min)
CODE_DIR = "{CODE_DIR}"   # fixed: the %%writefile cells below write here
OUTPUT_ROOT = "/kaggle/working/pa3_task3_outputs"
SETUP_ERROR = None

# %%writefile cannot create folders. Create them all now, and make %%writefile create its parent folder itself,
# so the code cells work even if a cell is run on its own or after a session restart (which empties /kaggle/working).
for _d in __DIRS__:
    os.makedirs(_d, exist_ok=True)
try:
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def writefile(line, cell):
        import shlex
        args = shlex.split(line)
        append = "-a" in args or "--append" in args
        fn = os.path.expanduser([a for a in args if not a.startswith("-")][0])
        os.makedirs(os.path.dirname(fn) or ".", exist_ok=True)
        existed = os.path.exists(fn)
        with open(fn, "a" if append else "w", encoding="utf-8") as fh:
            fh.write(cell)
        print(("Appending to " if append else "Overwriting " if existed else "Writing ") + fn)
except Exception as _ex:  # noqa: BLE001  (not running under IPython)
    print("note: %%writefile not patched:", _ex)'''

INSTALL = '''# ============================ DEPENDENCIES ============================
import sys, subprocess, traceback
sys.path[:0] = [f"{CODE_DIR}/common", f"{CODE_DIR}/task_3/src"]
try:
    from pa3common.kaggle import pip_install
    import torch
    torch_before = torch.__version__
    pip_install(["transformers==5.21.0", "math-verify[antlr4_13_2]==0.9.0", "matplotlib", "pytest", "pyyaml", "scipy"])
    out = subprocess.run([sys.executable, "-c", "import torch, transformers, math_verify, pyarrow; "
                          "import importlib.metadata as m; print(torch.__version__, transformers.__version__, "
                          "m.version('math-verify'), pyarrow.__version__)"], capture_output=True, text=True)
    print("kernel torch:", torch_before, "| subprocess torch/transformers/math-verify/pyarrow:", out.stdout.strip(),
          out.stderr[-400:])
    assert out.stdout.split()[0] == torch_before, "pip changed torch — restart the kernel and rerun"
    print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
          "| CPU cores:", len(os.sched_getaffinity(0)))
except Exception as ex:  # noqa: BLE001
    traceback.print_exc()
    SETUP_ERROR = ex'''

DATACHECK = '''# ============================ DATA CHECK ============================
# Shows what is attached under /kaggle/input and which tables the pipeline will use (fails fast if none).
if SETUP_ERROR is None:
    try:
        from pa3ttc.data.discover import find_tables, listing
        print(listing([Path("/kaggle/input")], max_lines=120))
        from pa3common.kaggle import scratch_root
        _t = find_tables(DATA_ROOT, "/kaggle/input", scratch=scratch_root("pa3_task3_scratch"),
                         url=DATA_URL or None, hf_repo=DATA_HF_REPO or None)
        for _k, _v in _t.items():
            print(f"\\n{_k}: folder {_v['folder']}")
            for _r in ("gen", "prob", "attn"):
                print(f"   {_r}: {_v.get(_r)}")
    except Exception as ex:  # noqa: BLE001
        print("!! DATA NOT FOUND:", ex)
        SETUP_ERROR = ex
        os.makedirs(f"{OUTPUT_ROOT}/data", exist_ok=True)       # so the results zip shows what was attached
        Path(f"{OUTPUT_ROOT}/data/input_listing.txt").write_text(
            listing([Path("/kaggle/input")], max_lines=3000) + "\\n\\n" + str(ex))'''

RUN = '''# ============================ RUN ============================
# Errors are caught so the RESULTS and DOWNLOAD cells still run (the error is re-raised at the very end).
import traceback
RUN_ERROR = SETUP_ERROR
if RUN_ERROR is None:
    os.chdir(f"{CODE_DIR}/task_3")
    os.environ["PA3_CONFIG_OVERRIDES"] = json.dumps({"data_root": DATA_ROOT, "data_url": DATA_URL or None,
                                                     "data_hf_repo": DATA_HF_REPO or None})
    try:
        from pa3ttc import orchestrate as O
        info = O.setup_env(OUTPUT_ROOT, budget_hours=BUDGET_HOURS)
        result = O.stage_auto(run_cpu_tests=RUN_CPU_TESTS, run_live=RUN_LIVE)
        print(json.dumps(result, indent=1))
        if result["failed"]:      # make a failed stage visible: the DOWNLOAD cell zips, then raises
            RUN_ERROR = RuntimeError(f"stages failed: {result['failed']} (skipped: {result['skipped']}) — "
                                     f"see {OUTPUT_ROOT}/logs/<stage>.log and data/input_listing.txt")
            print("!!", RUN_ERROR)
    except Exception as ex:  # noqa: BLE001
        traceback.print_exc()
        RUN_ERROR = ex
else:
    print("setup failed — skipping the pipeline:", repr(RUN_ERROR))'''

SHOW = '''# ============================ RESULTS ============================
from pathlib import Path
from IPython.display import Markdown, Image, display
R = Path(OUTPUT_ROOT)
if (R / "status.json").exists():
    print(json.dumps(json.loads((R / "status.json").read_text()), indent=1))
for name in ("data/math_alignment.json", "data/mathvista_alignment.json"):
    if (R / name).exists():
        print(name, json.dumps(json.loads((R / name).read_text()).get("alignment"), indent=1))
T, F = R / "report_assets/tables", R / "report_assets/figures"
for t in sorted(T.glob("*.md")) if T.exists() else []:
    display(Markdown(f"**{t.name}**\\n\\n" + t.read_text()))
for f in sorted(F.glob("*.png")) if F.exists() else []:
    display(Image(filename=str(f)))'''

DOWNLOAD = '''# ============================ DOWNLOAD RESULTS ============================
# Zips everything in the outputs folder and shows a download link (also after a failure).
import os, zipfile
from pathlib import Path
from IPython.display import FileLink, display
OUT = Path(globals().get("OUTPUT_ROOT", "/kaggle/working/pa3_task3_outputs"))
ZIP = Path("/kaggle/working/results_task3.zip")
n = 0
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.rglob("*")) if OUT.exists() else []:
        if f.is_file() and "_scratch" not in f.parts and f.suffix not in (".tmp", ".part", ".zip"):
            zf.write(f, f.relative_to(OUT.parent))
            n += 1
print(f"{ZIP}: {n} files, {ZIP.stat().st_size / 2**20:.1f} MiB")
os.chdir("/kaggle/working")
display(FileLink(ZIP.name))
if globals().get("RUN_ERROR") is not None:
    raise RuntimeError("the pipeline failed earlier (see the RUN cell); partial results are in the zip") from RUN_ERROR'''

_N = [0]


def _cell(kind: str, src: str) -> dict:
    _N[0] += 1
    c = {"cell_type": kind, "id": f"cell-{_N[0]:03d}", "metadata": {}, "source": src.splitlines(keepends=True)}
    if kind == "code":
        c.update({"execution_count": None, "outputs": []})
    return c


def files() -> list[tuple[str, Path]]:
    out = [(f"common/pa3common/{f}", REPO / "common" / "pa3common" / f) for f in COMMON_FILES]
    for sub in ("configs", "src", "scripts", "tests"):
        for p in sorted((TASK / sub).rglob("*")):
            if p.is_file() and "__pycache__" not in p.parts and p.suffix in (".py", ".yaml"):
                out.append((f"task_3/{p.relative_to(TASK).as_posix()}", p))
    out.append(("task_3/README.md", TASK / "README.md"))
    return out


def build() -> dict:
    _N[0] = 0
    fl = files()
    dirs = sorted({str(Path(CODE_DIR) / Path(rel).parent) for rel, _ in fl})
    mk = ("# ============================ WRITE THE CODE ============================\n"
          "import os\nfor d in " + json.dumps(dirs, indent=1) + ":\n    os.makedirs(d, exist_ok=True)\n"
          f"print('writing {len(fl)} files under {CODE_DIR}')")
    cells = [_cell("markdown", INTRO), _cell("code", SETTINGS.replace("__DIRS__", json.dumps(dirs))), _cell("code", mk)]
    sums = {}
    for rel, p in fl:
        txt = p.read_text()
        assert "\r" not in txt, rel
        if not txt.endswith("\n"):
            txt += "\n"
        sums[rel] = hashlib.sha256(txt.rstrip("\n").encode()).hexdigest()   # robust to trailing-newline handling
        cells.append(_cell("code", f"%%writefile {CODE_DIR}/{rel}\n{txt}"))
    check = ("# ============================ INTEGRITY CHECK ============================\n"
             "import hashlib, json\n"
             f"EXPECTED = json.loads({json.dumps(json.dumps(sums))})\n"
             "bad = [r for r, h in EXPECTED.items()\n"
             f"       if hashlib.sha256(open('{CODE_DIR}/' + r, 'rb').read().rstrip(b'\\n')).hexdigest() != h]\n"
             "if bad:\n"
             "    SETUP_ERROR = RuntimeError(f'files differ from the embedded sources: {bad}')\n"
             "    print(SETUP_ERROR)\n"
             "else:\n"
             "    print(f'{len(EXPECTED)} files written and verified')")
    cells += [_cell("code", check), _cell("code", INSTALL), _cell("code", DATACHECK), _cell("code", RUN), _cell("code", SHOW),
              _cell("code", DOWNLOAD)]
    return {"cells": cells,
            "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
                         "language_info": {"name": "python"},
                         "kaggle": {"accelerator": "nvidiaTeslaT4", "isInternetEnabled": True, "isGpuEnabled": True}},
            "nbformat": 4, "nbformat_minor": 5}


if __name__ == "__main__":
    out = TASK / "notebooks" / "pa3_task3_kaggle.ipynb"
    out.parent.mkdir(exist_ok=True)
    nb = build()
    out.write_text(json.dumps(nb, indent=1, ensure_ascii=False) + "\n")
    print(f"wrote {out} ({len(nb['cells'])} cells, {len(files())} embedded files)")


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/tests/conftest.py
"""Synthetic ATML-PA3-Data exports (same file layout and column names as described in the task) for CPU tests."""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

TASK = Path(__file__).resolve().parents[1]
sys.path[:0] = [str(TASK / "src"), str(TASK.parent / "common")]

MATH_GOLD = ["\\frac{1}{2}", "3", "2\\sqrt{3}", "x^2+1", "10", "\\text{Monday}", "7", "(1,2)"]
MATH_EQUIV = {"\\frac{1}{2}": ["0.5", "\\dfrac12", "1/2"], "3": ["3.0", "x=3"], "2\\sqrt{3}": ["\\sqrt{12}"],
              "x^2+1": ["1+x^2"], "10": ["10^\\circ"], "\\text{Monday}": ["Monday"], "7": ["7"], "(1,2)": ["(1,2)"]}


def _rollout_tokens(rng, T, n_vocab=50):
    return {"logp": (-rng.exponential(0.5, T)).astype(np.float32).tolist(),
            "kl": rng.uniform(2, 10, T).astype(np.float32).tolist(),
            "ids": rng.integers(0, n_vocab, T).tolist()}


def make_data(root: Path, Q: int = 16, R: int = 32, seed: int = 0, long_format: bool = False,
              t_range: tuple = (5, 60)) -> Path:
    rng = np.random.default_rng(seed)
    root = Path(root)
    # ---------------- MATH (LLM)
    d = root / "Qwen2.5-3B-Instruct" / "MATH"
    d.mkdir(parents=True, exist_ok=True)
    gen, prob = [], []
    for q in range(Q):
        sid = f"test/algebra/{q}.json"
        gold = MATH_GOLD[q % len(MATH_GOLD)]
        p_corr = [0.0, 1.0, 0.3, 0.6, 0.9][q % 5]
        for r in range(R):
            T = int(rng.integers(*t_range)) if not (q == 1 and r == 5) else 0
            u = rng.random()
            if q == 2 and r < 3:
                ext = None
            elif u < p_corr:
                ext = rng.choice([gold] + MATH_EQUIV[gold])
            else:
                ext = str(int(rng.integers(20, 24)))
            tok = _rollout_tokens(rng, T)
            if ext is not None and u < p_corr:     # make correct rollouts a bit more confident
                tok["kl"] = (np.array(tok["kl"]) + 1.0).tolist()
            text = f"<think>reasoning {q}-{r}</think> The answer is $\\boxed{{{ext}}}$." if ext else "<think>cut"
            gen.append({"dataset": "math", "sample_id": sid, "rollout_id": r, "response": text,
                        "token_ids": tok["ids"] + [151645], "num_tokens": T, "extracted_answer": ext,
                        "gold_answer": gold, "problem": f"Problem {q}: compute something."})
            prob.append({"dataset": "math", "sample_id": sid, "rollout_id": r, "logp": tok["logp"], "kl_u_p": tok["kl"]})
    pd.DataFrame(gen).to_parquet(d / "generations.parquet", index=False)
    pr = pd.DataFrame(prob).sample(frac=1.0, random_state=1)     # different row order than generations
    if long_format:
        rows = [{"dataset": x["dataset"], "sample_id": x["sample_id"], "rollout_id": x["rollout_id"], "token_index": t,
                 "logp": lp, "kl_u_p": kl} for x in pr.to_dict("records")
                for t, (lp, kl) in enumerate(zip(x["logp"], x["kl_u_p"]))]
        pd.DataFrame(rows).sample(frac=1.0, random_state=2).to_parquet(d / "probability_metrics.parquet", index=False)
    else:
        pr.to_parquet(d / "probability_metrics.parquet", index=False)
    # ---------------- MathVista (VLM)
    d = root / "Qwen2.5-VL-3B-Instruct" / "MathVista"
    d.mkdir(parents=True, exist_ok=True)
    gen, prob, att = [], [], []
    for q in range(Q):
        sid = str(q + 1)
        kind = q % 3
        if kind == 0:
            meta = {"question_type": "multi_choice", "answer_type": "text", "choices": ["red", "blue", "green"],
                    "precision": None}
            gold, wrong = "blue", ["A", "(C) green", "gren", "C"]
            right = ["B", "(B) blue", "blue"]
        elif kind == 1:
            meta = {"question_type": "free_form", "answer_type": "integer", "choices": None, "precision": None}
            gold, wrong, right = "5", ["4", "6", "five?"], ["5", "5.0"]
        else:
            meta = {"question_type": "free_form", "answer_type": "float", "choices": None, "precision": 1.0}
            gold, wrong, right = "1.5", ["1.4", "2"], ["1.5", "1.50", "1.53"]
        p_corr = [0.2, 0.8, 0.5, 1.0][q % 4]
        for r in range(R):
            T = int(rng.integers(*t_range))
            ext = (rng.choice(right) if rng.random() < p_corr else rng.choice(wrong)) if r != 7 else None
            tok = _rollout_tokens(rng, T)
            img = rng.uniform(0.0, 0.3, T)
            prm = rng.uniform(0.2, 0.5, T)
            gn = rng.uniform(0.2, 0.5, T)
            if q == 3 and r == 2 and T > 3:
                img[1] = prm[1] = gn[1] = 0.0          # undefined v at one token
            gen.append({"dataset": "mathvista", "sample_id": sid, "rollout_id": r, "response": f"... {ext}",
                        "token_ids": tok["ids"], "num_tokens": T, "extracted_answer": ext, "gold_answer": gold,
                        "question": f"Q{q}", **meta})
            prob.append({"dataset": "mathvista", "sample_id": sid, "rollout_id": r, "logp": tok["logp"],
                         "kl_u_p": tok["kl"]})
            att.append({"dataset": "mathvista", "sample_id": sid, "rollout_id": r, "image_attention": img.tolist(),
                        "prompt_text_attention": prm.tolist(), "generated_text_attention": gn.tolist()})
    g = pd.DataFrame(gen)
    g["choices"] = g["choices"].apply(lambda c: json.dumps(c) if c is not None else None)
    g.to_parquet(d / "generations.parquet", index=False)
    pd.DataFrame(prob).to_parquet(d / "probability_metrics.parquet", index=False)
    pd.DataFrame(att).sample(frac=1.0, random_state=3).to_parquet(d / "attention_metrics.parquet", index=False)
    (root / "README.MD").write_text("synthetic test data\n")
    return root


@pytest.fixture()
def env(tmp_path, monkeypatch):
    data = make_data(tmp_path / "data")
    out = tmp_path / "out"
    monkeypatch.setenv("PA3_OUTPUT_ROOT", str(out))
    monkeypatch.setenv("PA3_SCRATCH", str(tmp_path / "scratch"))
    monkeypatch.setenv("PA3_CONFIG_OVERRIDES", json.dumps({
        "data_root": str(data), "n_questions_expected": 16, "n_dev": 6, "n_live": 3, "n_boot": 50}))
    monkeypatch.setenv("PA3_LIVE_BACKEND", "fake")
    monkeypatch.delenv("PA3_CPUS", raising=False)
    return {"data": data, "out": out, "tmp": tmp_path}


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/tests/test_discover.py
"""Data discovery is tolerant to the upload layout, and a miss produces a diagnosable error + listing."""
from __future__ import annotations

import json
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pytest

from pa3ttc import stages
from pa3ttc.data.discover import find_tables
from pa3ttc.data.load import Pool


def _std(tmp_path) -> Path:
    from conftest import make_data

    return make_data(tmp_path / "std")


def _load(tmp_path, root, name, monkeypatch):
    monkeypatch.setenv("PA3_OUTPUT_ROOT", str(tmp_path / f"out_{name}"))
    monkeypatch.setenv("PA3_SCRATCH", str(tmp_path / f"scr_{name}"))
    monkeypatch.setenv("PA3_CONFIG_OVERRIDES", json.dumps({"data_root": str(root), "data_fallback_root": str(root),
                                                           "n_questions_expected": 16}))
    assert stages.main("data") == 0
    return Pool("math"), Pool("mathvista")


def _same(a, b):
    for x, y in zip(a, b):
        np.testing.assert_array_equal(x.T, y.T)
        np.testing.assert_array_equal(x.kl, y.kl)
        assert x.qids == y.qids
    assert np.array_equal(a[1].v, b[1].v, equal_nan=True)


def test_layout_variants(tmp_path, monkeypatch):
    std = _std(tmp_path)
    ref = _load(tmp_path, std, "ref", monkeypatch)
    math_dir, vl_dir = std / "Qwen2.5-3B-Instruct" / "MATH", std / "Qwen2.5-VL-3B-Instruct" / "MathVista"

    # (a) both models in one folder, prefixed file names, generations split into two shards
    flat = tmp_path / "flat" / "ATML-PA3-Data"
    flat.mkdir(parents=True)
    g = pq.read_table(math_dir / "generations.parquet")
    pq.write_table(g.slice(0, 200), flat / "math_generations-00000-of-00002.parquet")
    pq.write_table(g.slice(200), flat / "math_generations-00001-of-00002.parquet")
    shutil.copy(math_dir / "probability_metrics.parquet", flat / "math_probability_metrics.parquet")
    for f in ("generations", "probability_metrics", "attention_metrics"):
        shutil.copy(vl_dir / f"{f}.parquet", flat / f"mathvista_{f}.parquet")
    _same(ref, _load(tmp_path, tmp_path / "flat", "flat", monkeypatch))

    # (b) shard folders named after the tables
    sh = tmp_path / "shards"
    for src, name in ((math_dir, "llm"), (vl_dir, "vlm")):
        for f in src.glob("*.parquet"):
            (sh / name / f.stem).mkdir(parents=True, exist_ok=True)
            shutil.copy(f, sh / name / f.stem / "part-0.parquet")
    _same(ref, _load(tmp_path, sh, "shards", monkeypatch))

    # (c) a zipped upload that Kaggle did not unpack
    zroot = tmp_path / "zipped"
    zroot.mkdir()
    with zipfile.ZipFile(zroot / "ATML-PA3-Data.zip", "w") as z:
        for f in std.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(std))
    _same(ref, _load(tmp_path, zroot, "zip", monkeypatch))

    # (d) dataset reached through a symlink
    lroot = tmp_path / "linked"
    lroot.mkdir()
    try:
        os.symlink(std, lroot / "atml-pa3-data")
    except (OSError, NotImplementedError):
        pass
    else:
        _same(ref, _load(tmp_path, lroot, "link", monkeypatch))


def test_not_found_is_diagnosable(tmp_path):
    (tmp_path / "input" / "some-other-dataset").mkdir(parents=True)
    pd.DataFrame({"a": [1]}).to_parquet(tmp_path / "input" / "some-other-dataset" / "x.parquet")
    lp = tmp_path / "listing.txt"
    with pytest.raises(FileNotFoundError, match="Is the dataset attached"):
        find_tables(tmp_path / "input" / "atml-pa3-data", tmp_path / "input", scratch=tmp_path / "s", listing_path=lp)
    txt = lp.read_text()
    assert "does not exist" in txt and "x.parquet" in txt


def test_download_url(tmp_path, monkeypatch):
    std = _std(tmp_path)
    z = tmp_path / "pub" / "ATML-PA3-Data.zip"
    z.parent.mkdir()
    with zipfile.ZipFile(z, "w") as zf:
        for f in std.rglob("*"):
            if f.is_file():
                zf.write(f, Path("ATML-PA3-Data") / f.relative_to(std))
    empty = tmp_path / "input"
    empty.mkdir()
    found = find_tables(empty / "atml-pa3-data", empty, scratch=tmp_path / "scr", url=z.as_uri())
    assert set(found) == {"math", "mathvista"} and "attn" in found["mathvista"]
    # a second call reuses the download
    again = find_tables(empty / "atml-pa3-data", empty, scratch=tmp_path / "scr", url="file:///nonexistent.zip")
    assert again["math"]["gen"] == found["math"]["gen"]


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/tests/test_hf_backend.py
"""The real HF generation path on CPU with a tiny random Qwen2 model + an offline word-level tokenizer:
transformers accepts the sampling kwargs, hub generation defaults are overridden, seeding is reproducible, and
token accounting / truncation / termination flags are consistent with the generated ids."""
from __future__ import annotations

import pytest

torch = pytest.importorskip("torch")
transformers = pytest.importorskip("transformers")

CHAT = ("{% for m in messages %}<|im_start|>{{ m['role'] }}\n{{ m['content'] }}<|im_end|>\n{% endfor %}"
        "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}")
LIVE = {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "repetition_penalty": 1.0, "dtype": "float32"}


@pytest.fixture(scope="module")
def tiny(tmp_path_factory):
    from tokenizers import Tokenizer, models, pre_tokenizers
    from transformers import GenerationConfig, PreTrainedTokenizerFast, Qwen2Config, Qwen2ForCausalLM

    d = tmp_path_factory.mktemp("tiny")
    words = ["<|endoftext|>", "<|im_start|>", "<|im_end|>", "[UNK]", "system", "user", "assistant", "\\boxed{2}",
             "the", "answer", "is", "Hello", "You", "are", "a", "helpful", "assistant.", "Problem", "Think"]
    words += [f"w{i}" for i in range(60)]
    vocab = {w: i for i, w in enumerate(words)}
    tk = Tokenizer(models.WordLevel(vocab, unk_token="[UNK]"))
    tk.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
    tok = PreTrainedTokenizerFast(tokenizer_object=tk, unk_token="[UNK]", eos_token="<|im_end|>",
                                  pad_token="<|endoftext|>", additional_special_tokens=["<|im_start|>"])
    tok.chat_template = CHAT
    tok.save_pretrained(d)
    cfg = Qwen2Config(vocab_size=len(vocab), hidden_size=32, intermediate_size=64, num_hidden_layers=2,
                      num_attention_heads=2, num_key_value_heads=1, max_position_embeddings=256,
                      eos_token_id=[vocab["<|im_end|>"], vocab["<|endoftext|>"]], pad_token_id=vocab["<|endoftext|>"])
    torch.manual_seed(0)
    m = Qwen2ForCausalLM(cfg)
    m.generation_config = GenerationConfig(do_sample=True, temperature=0.3, top_p=0.5, top_k=5,
                                           repetition_penalty=1.05, eos_token_id=cfg.eos_token_id)
    m.save_pretrained(d)
    return d, vocab


def test_hf_backend_generation(tiny):
    from pa3ttc.live.backend import HFBackend

    d, vocab = tiny
    b = HFBackend(str(d), None, LIVE, device="cpu")
    gc = b.model.generation_config
    assert (gc.repetition_penalty, gc.temperature, gc.top_p, gc.top_k, gc.do_sample) == (1.0, 0.7, 0.8, 20, True)
    assert vocab["<|im_end|>"] in b.eos_ids
    msgs = [{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Problem w1 w2\n\nThink"}]
    assert b.render(msgs).endswith("<|im_start|>assistant\n")
    r1 = b.generate(msgs, 12, seed=11)
    r2 = b.generate(msgs, 12, seed=11)
    assert r1["response"] == r2["response"] and r1["output_tokens"] == r2["output_tokens"]
    outs = {b.generate(msgs, 12, seed=s)["response"] for s in range(6)}
    assert len(outs) > 1                                              # seeds matter
    seen_trunc = False
    for s in range(30):
        r = b.generate(msgs, 6, seed=s)
        assert r["input_tokens"] == len(b.tok(r["prompt_text"], add_special_tokens=False)["input_ids"])
        assert 0 <= r["output_tokens_retained"] <= r["output_tokens"] <= 6
        if r["terminated"]:
            assert r["output_tokens_retained"] < r["output_tokens"] and not r["truncated"]
        else:
            seen_trunc |= r["truncated"]
            assert r["output_tokens"] == 6 and r["truncated"] and r["output_tokens_retained"] == 6
        assert r["seconds"] > 0 and r["error"] is None
    assert seen_trunc
    # review prompt: the earlier response is placed as an assistant turn before the review instruction
    from pa3ttc.live.run import messages_review
    lc = {"system": "You are a helpful assistant.", "instruction": "Think", "review": "Review"}
    rv = b.render(messages_review("Problem", "the answer is \\boxed{2}", lc))
    assert "<|im_start|>assistant\nthe answer is \\boxed{2}<|im_end|>\n<|im_start|>user\nReview<|im_end|>" in rv


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/tests/test_orchestrate.py
"""The Kaggle orchestrator on synthetic data: stage subprocesses, background live worker (fake backend), CPU-core
split, markers/skip on rerun, and failure isolation (a failed stage skips only its dependants)."""
from __future__ import annotations

from pa3ttc.io import read_json


def test_stage_auto(env, monkeypatch):
    from pa3ttc import orchestrate as O

    monkeypatch.setenv("PA3_SESSION_START", str(__import__("time").time()))
    O.setup_env(str(env["out"]), budget_hours=1.0)
    O.STATE["gpus"] = 1                                   # pretend a GPU so the live worker is launched
    O.STATE["gen_cores"], O.STATE["off_cores"] = [0], [0]
    res = O.stage_auto()
    assert res["state"] == "done", res
    st = read_json(env["out"] / "status.json")["steps"]
    assert st["live_generation"]["state"] == "done" and st["live_eval"]["state"] == "done"
    assert (env["out"] / "report_assets" / "summary.md").exists()
    # rerun: finished stages are skipped, live calls are not regenerated
    n = len((env["out"] / "live" / "responses.jsonl").read_text().splitlines())
    O.STATE["steps"] = {}
    res = O.stage_auto()
    st = read_json(env["out"] / "status.json")["steps"]
    assert st["offline"].get("note") == "already done" and "live_generation" not in st
    assert len((env["out"] / "live" / "responses.jsonl").read_text().splitlines()) == n


def test_failure_isolation(env, monkeypatch):
    from pa3ttc import orchestrate as O

    O.setup_env(str(env["out"]), budget_hours=1.0)
    O.STATE["gpus"] = 0
    monkeypatch.setenv("PA3_TEST_FAIL_STAGE", "scorer")
    res = O.stage_auto()
    st = read_json(env["out"] / "status.json")["steps"]
    assert st["scorer"]["state"] == "failed" and st["signals"]["state"] == "done"
    assert st["offline"]["state"] == "skipped" and st["analysis"]["state"] == "skipped"
    assert st["live_generation"]["state"] == "skipped"
    assert res["state"] == "failed" and res["failed"] == ["scorer"] and "offline" in res["skipped"]


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/tests/test_pipeline.py
"""End-to-end: every stage on synthetic exports (in-process), live generation with the fake backend, and checks of
the accounting rules (costs, N=1, oracle, consistency between offline SC and the adaptive fallback, live tokens)."""
from __future__ import annotations

import json

import numpy as np
import pandas as pd
import pytest

from pa3ttc import stages
from pa3ttc.data.load import Pool
from pa3ttc.io import read_json, read_jsonl
from pa3ttc.live import run as live_run


def _run_all(skip_live=False):
    for s in ("data", "splits", "scorer", "signals", "offline", "analysis", "adaptive", "compute"):
        assert stages.main(s) == 0
    if not skip_live:
        assert live_run.main() == 0
        assert stages.main("live_eval") == 0
    for s in ("qualitative", "report"):
        assert stages.main(s) == 0


def test_end_to_end(env):
    _run_all()
    out = env["out"]
    # ---- data: join despite shuffled rows, alignment, zero-length rollout, undefined v
    for f in (out / "data").glob("*.json"):
        assert f.stat().st_size < 50_000, f"{f.name} is {f.stat().st_size} bytes (arrays leaked into a report?)"
    ali = read_json(out / "data" / "math_alignment.json")
    assert ali["alignment"]["misaligned_measurements"] == 0 and ali["alignment"]["zero_length_rollouts"] == 1
    assert ali["alignment"]["token_ids_len_minus_T"] == {"1": 16 * 32}
    mv = read_json(out / "data" / "mathvista_alignment.json")
    assert mv["columns"]["attention_metrics"] == {"img": "image_attention", "prm": "prompt_text_attention",
                                                  "gen": "generated_text_attention"}
    assert mv["alignment"]["attention_tokens_undefined_v"] == 1
    pool = Pool("math")
    assert pool.Q == 16 and pool.R == 32 and pool.rollout_ids[0].tolist() == [str(i) for i in range(32)]
    # ---- scorer
    man = read_json(out / "answers" / "scorer_manifest.json")
    assert man["math_verify"] and man["math"]["extractor_check"]["agreement"] == 1.0
    lab = np.load(out / "answers" / "math" / "labels.npz")
    q2 = pool.qids.index("test/algebra/2.json")                                # string order: index != 2
    assert pool.qids[:3] == ["test/algebra/0.json", "test/algebra/1.json", "test/algebra/10.json"]
    assert (lab["group"][q2, :3] == -1).all() and not lab["correct"][q2, :3].any()
    assert man["mathvista"]["metadata_source"] == "supplied files"
    # ---- offline
    po = pd.read_parquet(out / "offline" / "math" / "per_order.parquet")
    pqo = pd.read_parquet(out / "offline" / "math" / "per_question.parquet")
    orders = np.load(out / "splits" / "math_orders.npy")
    T = pool.T
    n1 = po[po["N"] == 1]
    assert n1.groupby(["qi", "order"])["rpos"].nunique().max() == 1          # N=1: all rules agree
    f = po[po["method"] == "first"]
    assert (f["rpos"].to_numpy() == orders[f["qi"], f["order"], 0]).all()
    assert (f["cost"].to_numpy() == T[f["qi"], orders[f["qi"], f["order"], 0]]).all()
    s8 = po[(po["method"] == "sc") & (po["N"] == 8)]
    exp = [T[q, orders[q, k, :8]].sum() for q, k in zip(s8["qi"], s8["order"])]
    assert (s8["cost"].to_numpy() == np.array(exp)).all()
    orc = po[po["method"] == "sc"].pivot_table(index=["qi", "order"], columns="N", values="oracle")
    assert (np.diff(orc.to_numpy().astype(int), axis=1) >= 0).all()             # nested prefixes
    assert ((pqo["acc"] <= pqo["oracle"] + 1e-12)).all()
    assert np.allclose(pqo["missed"] + pqo["no_correct"] + pqo["acc"], 1.0)
    assert set(po["method"].unique()) == {"first", "sc", "logp", "logp_rwc", "selfcert", "selfcert_rwc", "shortest"}
    pv = pd.read_parquet(out / "offline" / "mathvista" / "per_order.parquet")
    assert {"visconf", "visconf_rwc", "vision", "vision_rwc", "tokfuse", "tokfuse_rwc"} <= set(pv["method"].unique())
    bt = pd.read_csv(out / "offline" / "mathvista" / "bootstrap.csv")
    assert {"lo", "hi"} <= set(bt.columns) and (bt["lo"] <= bt["diff"] + 1e-12).all()
    # ---- adaptive: fallback == offline SC at 32; frozen settings meet the dev constraint
    for ds in ("math", "mathvista"):
        fz = read_json(out / "adaptive" / ds / "frozen.json")
        grid = pd.read_csv(out / "adaptive" / ds / "dev_grid.csv")
        for fam in ("agreement", "quality"):
            assert fz[fam]["acc"] >= fz["target_acc"] - 1e-12
            elig = grid[(grid["family"] == fam) & (grid["acc"] >= fz["target_acc"] - 1e-12)]
            assert fz[fam]["cost"] == pytest.approx(elig["cost"].min())
        s = pd.read_csv(out / "offline" / ds / "summary_dev.csv")
        sc32 = s[(s["method"] == "sc") & (s["N"] == 32)]["acc"].item()
        assert fz["sc32_dev_acc"] == pytest.approx(sc32)
        h = read_json(out / "adaptive" / ds / "heldout.json")
        assert sum(h["agreement"]["stop_distribution"].values()) == pytest.approx(1.0)
    # ---- analysis
    cv = read_json(out / "analysis" / "mathvista" / "conf_vision_summary.json")
    assert cv["identity_check"]["max_abs_error"] < 1e-9
    assert (out / "analysis" / "math" / "associations.csv").exists()
    # ---- live accounting
    qs = read_json(out / "live" / "questions.json")
    sp = read_json(out / "splits" / "math.json")
    assert [q["qid"] for q in qs] == sp["eval"][:3]
    rows = read_jsonl(out / "live" / "responses.jsonl")
    assert len(rows) == 15 and [r["slot"] for r in rows[:5]] == [1, 2, 3, 4, 0]
    assert len({r["seed"] for r in rows}) == 15
    by = {(r["qid"], r["slot"]): r for r in rows}
    for q in qs:
        r4 = by[(q["qid"], 4)]
        assert r4["messages"][2] == {"role": "assistant", "content": by[(q["qid"], 1)]["response"]}
        assert r4["messages"][3]["content"].startswith("Review the solution above.")
        assert r4["cap"] == 2048 and by[(q["qid"], 0)]["cap"] == 3072 and by[(q["qid"], 2)]["cap"] == 1024
        assert by[(q["qid"], 1)]["messages"][1]["content"].endswith(
            "Think step by step inside <think> </think> tags, then give your final answer inside \\boxed{}.")
    lq = pd.read_csv(out / "live" / "per_question.csv", dtype={"qid": str}).set_index("qid")
    for q in qs:
        x = lq.loc[q["qid"]]
        assert x["refine_input_tokens"] == by[(q["qid"], 1)]["input_tokens"] + by[(q["qid"], 4)]["input_tokens"]
        assert x["refine_output_tokens"] == by[(q["qid"], 1)]["output_tokens"] + by[(q["qid"], 4)]["output_tokens"]
        assert x["parallel_output_tokens"] == sum(by[(q["qid"], s)]["output_tokens"] for s in (1, 2, 3))
        assert x["single_seconds"] == pytest.approx(by[(q["qid"], 0)]["seconds"])
        if by[(q["qid"], 4)]["extracted"] is None:
            assert not x["refine_correct"] and x["refine_fallback_answer"] == x["initial_answer"]
    ls = read_json(out / "live" / "summary.json")
    t = ls["transitions_vs_initial"]
    assert t["wrong_to_correct"] + t["correct_to_wrong"] + t["correct_to_correct"] + t["wrong_to_wrong"] == 3
    # ---- report
    figs = {p.name for p in (out / "report_assets" / "figures").glob("*.png")}
    assert {"math_accuracy_curves.png", "mathvista_accuracy_curves.png", "live_strategies.png",
            "adaptive_stopping_distribution.png", "mathvista_confidence_vision.png"} <= figs
    assert read_json(out / "_stages" / "report.json")["result"]["failed"] == {}
    assert (out / "qualitative" / "cases.md").exists()


def test_live_resume_and_pause(env, monkeypatch):
    for s in ("data", "splits"):
        stages.main(s)
    out = env["out"]
    monkeypatch.setenv("PA3_DEADLINE_UNIX", "0")                 # budget already used -> pause before any call
    assert live_run.main() == 3
    assert read_json(out / "live" / "status.json")["state"] == "paused"
    monkeypatch.delenv("PA3_DEADLINE_UNIX")
    assert live_run.main() == 0
    rows = read_jsonl(out / "live" / "responses.jsonl")
    # simulate a crash that lost the last two calls, then resume: only those are regenerated, identically
    lines = (out / "live" / "responses.jsonl").read_text().splitlines()
    (out / "live" / "responses.jsonl").write_text("\n".join(lines[:-2]) + "\n")
    assert live_run.main() == 0
    rows2 = read_jsonl(out / "live" / "responses.jsonl")
    assert len(rows2) == len(rows) == 15
    key = lambda r: (r["qid"], r["slot"], r["seed"], r["response"])  # noqa: E731
    assert sorted(map(key, rows)) == sorted(map(key, rows2))


def test_long_format_probability_table(tmp_path, monkeypatch):
    from conftest import make_data

    a = make_data(tmp_path / "a")
    b = make_data(tmp_path / "b", long_format=True)
    res = {}
    for name, root in (("a", a), ("b", b)):
        monkeypatch.setenv("PA3_OUTPUT_ROOT", str(tmp_path / f"out_{name}"))
        monkeypatch.setenv("PA3_SCRATCH", str(tmp_path / f"scr_{name}"))
        monkeypatch.setenv("PA3_CONFIG_OVERRIDES", json.dumps({"data_root": str(root), "n_questions_expected": 16}))
        stages.main("data")
        p = Pool("math")
        res[name] = (p.T.copy(), p.kl.copy(), p.logp.copy())
    for x, y in zip(res["a"], res["b"]):
        np.testing.assert_array_equal(x, y)


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/tests/test_units.py
"""Unit tests: extraction, answer equivalence, splits/orders, selection/tie rules, VisConf (fast == naive,
zero leakage), tokenwise identity, adaptive controller logic."""
from __future__ import annotations

import numpy as np
import pytest

from pa3ttc.adaptive.controllers import evaluate, features, stop_index
from pa3ttc.adaptive.tune import select_setting
from pa3ttc.answers import mathvista as mvn
from pa3ttc.answers.extract import last_boxed
from pa3ttc.answers.math_equiv import NO_ANSWER, group_and_grade
from pa3ttc.data.splits import make_orders, make_split
from pa3ttc.select.rules import avg_rank_desc, rwc, rwc_weights, select, vote, vote_margin
from pa3ttc.signals.visconf import VisConfIndex, naive_scores


# ----------------------------------------------------------------------------- extraction
@pytest.mark.parametrize("text,ans", [
    ("so $\\boxed{\\frac{1}{2}}$.", "\\frac{1}{2}"),
    ("\\boxed{1} then \\boxed{\\sqrt{x^{2}}}", "\\sqrt{x^{2}}"),
    ("\\boxed{a} and finally \\boxed{b", None),            # last box cut off by the cap
    ("answer \\boxed 5 done", "5"),
    ("\\fbox{7}", "7"),
    ("no box here", None),
    ("\\boxed{}", None),
    ("<think>\\boxed{2}</think> \\boxed{ {3} }", "{3}"),
])
def test_last_boxed(text, ans):
    assert last_boxed(text) == ans


# ----------------------------------------------------------------------------- math-verify grouping
def test_math_grouping_and_grading():
    r = group_and_grade(["0.5", "\\frac{1}{2}", None, "1/2", "3", "", "\\dfrac12", "Monday?"], "\\frac{1}{2}")
    g = r["groups"]
    assert g[2] == NO_ANSWER and g[5] == NO_ANSWER
    assert g[0] == g[1] == g[3] == g[6] != g[4]
    assert r["group_correct"][g[0]] and not r["group_correct"][g[4]]
    assert r["indiv_correct"] == [True, True, False, True, False, False, True, False]
    # groups are numbered by first appearance in rollout order
    assert g[0] == 0 and g[4] == 1


def test_math_equation_and_text_gold():
    r = group_and_grade(["x=3", "3", "4"], "3")
    assert r["groups"][0] == r["groups"][1] and r["group_correct"][r["groups"][0]]
    r = group_and_grade(["Monday", "\\text{Monday}", "Tuesday"], "\\text{Monday}")
    assert r["indiv_correct"][:2] == [True, True] and not r["indiv_correct"][2]


# ----------------------------------------------------------------------------- MathVista normalisation
def test_mathvista_rules():
    mc = {"question_type": "multi_choice", "answer_type": "text", "choices": '["red", "blue", "green"]'}
    assert mvn.normalize("B", mc)[0] == "MC:B"
    assert mvn.normalize("(b) blue", mc)[0] == "MC:B"
    assert mvn.normalize("blue", mc) == ("MC:B", "mc_exact")
    assert mvn.normalize("gren", mc) == ("MC:C", "mc_fuzzy")
    assert mvn.normalize("D", mc)[0] == "MC:B" or mvn.normalize("D", mc)[1] == "mc_fuzzy"   # out of range -> fuzzy
    assert mvn.normalize(None, mc) == (None, "na") and mvn.normalize("  ", mc) == (None, "na")
    assert mvn.gold_key("blue", mc) == "MC:B"
    it = {"question_type": "free_form", "answer_type": "integer"}
    assert mvn.normalize("5.0", it)[0] == mvn.normalize("5", it)[0] == "NUM:5"
    assert mvn.normalize("five?", it) == ("STR:five?", "str")
    fl = {"question_type": "free_form", "answer_type": "float", "precision": 1.0}
    assert mvn.normalize("1.53", fl)[0] == mvn.gold_key("1.5", fl) == "NUM:1.5"
    r = mvn.group_and_grade(["B", "blue", None, "A", "(B)"], "blue", mc)
    assert r["groups"] == [0, 0, NO_ANSWER, 1, 0] and r["group_correct"] == [True, False]
    assert mvn.levenshtein("kitten", "sitting") == 3


# ----------------------------------------------------------------------------- splits and orders
def test_split_lexicographic_and_fresh_rng():
    ids = [str(i) for i in range(1, 1001)]
    a = make_split(ids, 6304, 200)
    assert a["sorted"][:3] == ["1", "10", "100"]                    # string order
    exp = np.array(sorted(ids))[np.random.default_rng(6304).permutation(1000)]
    assert a["dev"] == exp[:200].tolist() and a["eval"] == exp[200:].tolist()
    b = make_split([f"x{i:04d}" for i in range(1000)], 6304, 200)
    assert a["permutation"] == b["permutation"]                     # fresh generator per dataset
    assert set(a["dev"]).isdisjoint(a["eval"]) and len(a["eval"]) == 800


def test_orders_stateless():
    o = make_orders(20, 1, 6304, 10, 32)
    o2 = make_orders(5, 1, 6304, 10, 32)
    assert (o[:5] == o2).all()
    assert all(sorted(o[3, k]) == list(range(32)) for k in range(10))
    assert not (make_orders(5, 0, 6304, 10, 32) == o2).all()
    r = np.random.default_rng([6304, 1, 4])
    assert (o[4, 0] == r.permutation(32)).all() and (o[4, 1] == r.permutation(32)).all()


# ----------------------------------------------------------------------------- selection rules
def test_vote_ties_earliest():
    assert vote([5, 7, 7, 5])[:2] == (5, 0)          # 2-2 tie -> answer whose earliest supporter is first
    assert vote([7, 5, 5, 7])[:2] == (7, 0)
    assert vote([1, 2, 3])[:2] == (1, 0)
    assert vote([NO_ANSWER, NO_ANSWER, 4])[0] == NO_ANSWER   # NO_ANSWER votes like any answer


def test_argmax_ties_and_nan():
    assert select("selfcert", [1, 2, 3], [0.5, 0.9, 0.9]) == 1
    assert select("selfcert", [1, 2, 3], [np.nan, -5.0, np.nan]) == 1
    assert select("shortest", [1, 2, 3], None, [30, 10, 10]) == 1
    for m in ("first", "sc", "selfcert", "selfcert_rwc", "shortest", "visconf_rwc"):
        assert select(m, [9], [0.1], [5]) == 0                       # N = 1


def test_rwc_average_ranks():
    s = [0.9, 0.5, 0.5, 0.1]
    assert avg_rank_desc(s).tolist() == [1.0, 2.5, 2.5, 4.0]
    w = rwc_weights(s)
    assert np.allclose(w, [1.0, 0.5, 0.5, 0.0])
    g, idx, tot = rwc([1, 2, 2, 1], s)          # W(1) = 1.0, W(2) = 1.0 -> tie -> earliest supporter (answer 1)
    assert (g, idx) == (1, 0) and tot == {1: 1.0, 2: 1.0}
    g, idx, tot = rwc([2, 1, 1, 2], [0.1, 0.9, 0.8, 0.2])    # ranks 4,1,2,3 -> W(1)=1+2/3, W(2)=1/3
    assert g == 1 and idx == 1
    assert vote_margin(tot) == pytest.approx((5 / 3 - 1 / 3) / 2)
    assert vote_margin({4: 2.0}) == 1.0
    assert np.allclose(rwc_weights([-np.inf, np.nan, 1.0]), [0.25, 0.25, 1.0])


# ----------------------------------------------------------------------------- VisConf
def _random_index(rng, n=12, with_zero=True):
    lens = rng.integers(0, 40, n)
    if with_zero:
        lens[3] = 0
    lens[5] = 1
    off = np.concatenate([[0], np.cumsum(lens)])
    k = rng.uniform(0, 10, off[-1]).astype(np.float32)
    k[off[6]:off[7]] = 4.0                         # constant segment (ties)
    k[: min(5, off[-1])] = np.round(k[: min(5, off[-1])])
    v = rng.uniform(0, 1, off[-1]).astype(np.float32)
    if lens[8] > 2:
        v[off[8] + 1] = np.nan
    return off, k, v


def test_visconf_fast_equals_naive():
    rng = np.random.default_rng(0)
    for trial in range(30):
        off, k, v = _random_index(rng)
        idx = VisConfIndex(off, k, v)
        n = len(off) - 1
        for _ in range(5):
            rows = rng.choice(n, size=int(rng.integers(2, n + 1)), replace=False)
            if np.diff(off)[rows].sum() == 0:
                continue
            segk = [k[off[r]:off[r + 1]].astype(np.float64) for r in rows]
            segv = [v[off[r]:off[r + 1]].astype(np.float64) for r in rows]
            Cn, Zn, Ztn, an, dn = naive_scores(segk, segv)
            V = np.array([np.nanmean(s) if np.isfinite(s).any() else np.nan for s in segv])
            C, Z, Zt, a, d = idx.scores(rows, V)
            assert a == pytest.approx(an) and d == pytest.approx(dn)
            np.testing.assert_allclose(C, Cn, rtol=1e-9, atol=1e-9)
            np.testing.assert_allclose(Z, Zn, rtol=1e-9, atol=1e-9)
            np.testing.assert_allclose(Zt, Ztn, rtol=1e-9, atol=1e-9)


def test_visconf_degenerate_spread():
    off = np.array([0, 4, 8])
    k = np.full(8, 3.0, np.float32)
    k[7] = 3.0 + 1e-9
    v = np.full(8, 0.5, np.float32)
    C, Z, Zt, a, d = VisConfIndex(off, k, v).scores([0, 1], np.array([0.5, 0.5]))
    Cn, Zn, Ztn, an, dn = naive_scores([k[:4].astype(float), k[4:].astype(float)], [v[:4].astype(float), v[4:].astype(float)])
    assert d == 1e-8
    np.testing.assert_allclose(C, Cn, atol=1e-6)


def test_visconf_zero_leakage():
    """Changing rollouts that are not in the candidate prefix never changes the prefix's scores."""
    rng = np.random.default_rng(1)
    off, k, v = _random_index(rng, n=16, with_zero=False)
    rows = np.array([2, 7, 9, 0])
    V = rng.uniform(0, 1, 4)
    base = VisConfIndex(off, k, v).scores(rows, V)
    k2, v2 = k.copy(), v.copy()
    for r in set(range(16)) - set(rows.tolist()):
        k2[off[r]:off[r + 1]] = rng.uniform(-50, 50, off[r + 1] - off[r])
        v2[off[r]:off[r + 1]] = rng.uniform(0, 1, off[r + 1] - off[r])
    other = VisConfIndex(off, k2, v2).scores(rows, V)
    for x, y in zip(base, other):
        np.testing.assert_array_equal(np.asarray(x), np.asarray(y))
    # scores do not depend on the order of the candidate set either
    perm = VisConfIndex(off, k, v).scores(rows[::-1], V[::-1])
    np.testing.assert_array_equal(perm[0][::-1], base[0])


def test_tokenwise_identity():
    rng = np.random.default_rng(2)
    off = np.array([0, 30, 55, 90])
    k = rng.uniform(0, 8, 90)
    v = rng.uniform(0, 1, 90)
    idx = VisConfIndex(off, k, v)
    V = np.array([v[off[i]:off[i + 1]].mean() for i in range(3)])
    C, Z, Zt, a, d = idx.scores([0, 1, 2], V)
    for i in range(3):
        kk, vv = k[off[i]:off[i + 1]], v[off[i]:off[i + 1]]
        c = np.clip((kk - a) / d, 0, 1)
        cov = np.mean(c * vv) - c.mean() * vv.mean()
        assert Zt[i] - Z[i] == pytest.approx(cov, abs=1e-10)


# ----------------------------------------------------------------------------- adaptive controllers
def test_controller_stop_logic():
    # one question, one order: answers revealed in order; quality score = position-based
    R = 32
    group = np.array([[1, 1, 1, 2] + [1] * 4 + [2] * 8 + [3] * 16])
    correct = group == 2
    T = np.full((1, R), 10)
    orders = np.arange(R)[None, None, :]
    f = features(group, correct, T, lambda q, k, o, n: -np.arange(n, dtype=float), orders)
    # n=4: counts {1:3, 2:1} -> lead 0.75 unique; n=8: 7/8; n=16: {1:7,2:9}->2 leads 9/16; n=32: 3 leads 16/32
    assert np.allclose(f["lead_frac"][0, 0], [0.75, 7 / 8, 9 / 16, 0.5])
    st = stop_index(f, 0.75, None, "agreement")
    assert st[0, 0] == 0                                   # stops at n=4 with answer 1 (wrong)
    assert stop_index(f, 0.875, None, "agreement")[0, 0] == 1
    assert stop_index(f, 1.0, None, "agreement")[0, 0] == 3
    assert stop_index(f, None, None, "agreement")[0, 0] == 3
    # quality: RWC with scores favouring early rollouts agrees with SC at n=4; margin check
    assert f["agree_rwc"][0, 0, 0]
    m4 = f["margin"][0, 0, 0]
    assert stop_index(f, 0.75, m4 + 0.01, "quality")[0, 0] != 0
    r = evaluate(f, st)
    assert r["cost"] == 40 and r["rollouts"] == 4 and r["acc"] == 0.0
    # NO_ANSWER leader never stops
    g2 = np.full((1, R), NO_ANSWER)
    f2 = features(g2, g2 == 5, T, lambda q, k, o, n: np.zeros(n), orders)
    assert stop_index(f2, 0.5, None, "agreement")[0, 0] == 3


def test_selection_rule_ties():
    rows = [{"tau": 0.5, "gamma": None, "acc": 0.70, "cost": 100.0}, {"tau": 0.75, "gamma": None, "acc": 0.71, "cost": 100.0},
            {"tau": 0.625, "gamma": None, "acc": 0.71, "cost": 100.0}, {"tau": 1.0, "gamma": None, "acc": 0.60, "cost": 50.0},
            {"tau": None, "gamma": None, "acc": 0.72, "cost": 300.0}]
    assert select_setting(rows, 0.705)["tau"] == 0.625      # cheapest eligible; tie -> higher acc -> smaller tau
    assert select_setting(rows, 0.715)["tau"] is None        # only the fallback qualifies


In [ ]:
%%writefile /kaggle/working/ATML-PA3/task_3/README.md
# PA3 Task 3 — Test-time compute: sampling, selection, refinement and adaptive inference

Offline analysis of the supplied `ATML-PA3-Data` rollouts (32 per question; Qwen2.5-3B-Instruct on a 1,000-question
MATH subset, Qwen2.5-VL-3B-Instruct on MathVista testmini) plus a 50-question live experiment with
`Qwen/Qwen2.5-3B-Instruct` (FP16, batch 1). Approved design: see the Task 3 design summary (2026-10-31).

## Run on Kaggle
Open `notebooks/pa3_task3_kaggle.ipynb` (self-contained: it writes every file of this folder and the needed
`common/` modules with `%%writefile` cells). **Add Input → ATML-PA3-Data** (expected at `/kaggle/input/atml-pa3-data`,
otherwise found anywhere under `/kaggle/input`), **GPU T4 ×2**, **Internet on**, then **Save & Run All (Commit)**.
Only GPU0 is used (live generation, in the background); the offline stages run at the same time on the other CPU
cores. The last cell writes `/kaggle/working/results_task3.zip` and shows a download link, also after a failure.

The **DATA CHECK** cell (runs ~1 min after start) lists everything under `/kaggle/input` and the tables it will use;
if the data is not found it stops there, writes `data/input_listing.txt` into the results zip and the run ends with an
error (it no longer "finishes" with nothing done). Accepted layouts: the task's file names, shard files/folders,
both models in one folder with `math_`/`mathvista_` prefixes, an unextracted .zip/.tar, or a Hugging Face dataset repo
(`data_hf_repo` in the config). Any failed stage makes the final cell raise after the zip is written.

Column names are auto-detected (`outputs/data/<ds>_columns.json` shows what was picked and every column in the
files). If a pick is wrong, set it in `configs/task3.yaml → columns`, e.g. `columns: {math: {text: response}}`.

Regenerate the notebook after code changes: `python task_3/scripts/build_notebook.py`.
CPU tests: `cd task_3 && python -m pytest -q tests` (~20 s; needs torch + transformers for the HF-backend test).

## Pipeline (`src/pa3ttc/orchestrate.py`, one subprocess per stage, `python -m pa3ttc.stages <stage>`)
| stage | spec step | what it does |
|---|---|---|
| data | 1 | find the two model folders, join the 3 parquets on (dataset, sample_id, rollout_id), check token-list alignment, cache ragged token arrays |
| splits | — | sort IDs as strings, `default_rng(6304)` per dataset → 200 dev / 800 held-out; 10 orders per question (`default_rng([6304, d, q])`); first 50 MATH held-out IDs for the live run |
| *live (background, GPU0)* | 4 | 5 seeded calls per question (parallel ×3, revision, single), resumable `responses.jsonl` |
| scorer | 1 | answer groups + correctness (math-verify / MathVista rules), NO_ANSWER handling, frozen `scorer_manifest.json` |
| signals | 2 | L, K, V per rollout; VisConf / tokenwise fusion recalibrated on every candidate prefix |
| offline | 1–2 | every rule × N ∈ {1,2,4,8,16,32} × 10 orders; accuracy, oracle coverage, token cost, distinct answers, failure decomposition, paired bootstrap |
| analysis | 3 | length/score/correctness associations, quartile gaps, confidence–vision correlations, Z_tok − Z = Cov check, trajectories |
| adaptive | 5 | agreement and quality-aware stopping: dev grid → frozen settings → held-out evaluation |
| compute | 6 | marginal value of each extra sample, coverage-vs-selection gap |
| live_eval | 4 | accuracy, tokens, runtime, truncation/extraction, revision transitions |
| qualitative, report | 6 | seeded example cases, figures, tables, `report_assets/summary.md` |

## Layout
```
configs/task3.yaml     seeds, budgets, grids, live settings, optional column overrides
src/pa3ttc/
  data/      discover (folders + README), load (column resolution, Arrow list → ragged arrays, join, alignment), splits
  answers/   extract (balanced-brace last \boxed), math_equiv (math-verify grouping/grading), mathvista (official rules), scorer
  signals/   rollout (L, K, V), visconf (exact sorted-prefix-sum VisConf, no leakage), stage
  select/    rules (FirstAns, SC, argmax, shortest, RWC with average ranks, tie rules, vote margin)
  offline/   runner (methods × budgets × orders), bootstrap (paired, question-level)
  analysis/  signal_assoc, conf_vision, compute_alloc
  adaptive/  controllers (features, stopping, evaluation), tune (dev selection, freeze, held-out)
  live/      backend (HF FP16 / fake), run (worker), evaluate
  report/    figures, qualitative
  stages.py  orchestrate.py
scripts/build_notebook.py
tests/      unit tests, end-to-end pipeline on synthetic exports, live resume, real HF generation with a tiny Qwen2
```

## Outputs (`/kaggle/working/pa3_task3_outputs`)
`data/` README copy, column resolution, alignment reports · `splits/` split IDs, orders, candidate rollout IDs, live IDs ·
`answers/` labels, groups, scorer manifest · `signals/` rollout scores, VisConf per prefix ·
`offline/<ds>/` per_order (selected rollout/answer/cost per question × order × N × method), per_question, summaries,
bootstrap · `analysis/` · `adaptive/<ds>/` dev grid, frozen.json, held-out results, recoverable early stops ·
`compute/` · `live/` questions, responses.jsonl (prompts, responses, seeds, lengths, truncation, time, memory),
per-question table, summary · `qualitative/cases.md` · `report_assets/` figures, tables, summary.md ·
`status.json`, `logs/`.


In [ ]:
# ============================ INTEGRITY CHECK ============================
import hashlib, json
EXPECTED = json.loads("{\"common/pa3common/__init__.py\": \"e66d72d492a8da8c2fed466f233fc02a88fdc69a922463a0cef6fc5a65403a90\", \"common/pa3common/kaggle.py\": \"dafc543c9a557ccebab701927c45a75a53992d49d08966145f1a9249243db520\", \"task_3/configs/task3.yaml\": \"873f1f905f3403b1ee3e774ae829d8fd5521a0fdd762a80f74377c37830afa26\", \"task_3/src/pa3ttc/__init__.py\": \"7dbd07f44315acfd835455ee03eccb96e4bea6d4cb5e65c0bb6189a116b816c9\", \"task_3/src/pa3ttc/adaptive/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_3/src/pa3ttc/adaptive/controllers.py\": \"2f5168655465fc60e4535f9ff3a964d748e9ad318e47ed5ea1e531572b105506\", \"task_3/src/pa3ttc/adaptive/tune.py\": \"6a6405f55bf06190ef3184ab3874115c7c5171a917bb960fe19b25e8c36ab779\", \"task_3/src/pa3ttc/analysis/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_3/src/pa3ttc/analysis/compute_alloc.py\": \"e345887b12475deb82409ca951e4feb9ac03e63dbec750f058614cce1ba72c85\", \"task_3/src/pa3ttc/analysis/conf_vision.py\": \"0c9a3d7d08f28e87f3c5d47104eebc522495d99ca08f8a722b7c522ca4175fb1\", \"task_3/src/pa3ttc/analysis/signal_assoc.py\": \"c423f54054aeea3e56fd72aabf46fac64a0295565d67c10698bdf8074ec8f687\", \"task_3/src/pa3ttc/answers/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_3/src/pa3ttc/answers/extract.py\": \"ceaaf6dba3ea5461957f38772f7fcd28be0f8ca25c657b6eeac38ae700cc6894\", \"task_3/src/pa3ttc/answers/math_equiv.py\": \"b7b29c59c60c3c8d478433490aaa504420d0b4fd5185f4e3b983d31de0bcce63\", \"task_3/src/pa3ttc/answers/mathvista.py\": \"a1efd20088845bf27d8b9a3a54de8216ebe6c92dfb6b5ff901334709d01fe56c\", \"task_3/src/pa3ttc/answers/scorer.py\": \"fa634196f3e3aa928b922f2723f81331c38c280841f34f0a39518fa642063768\", \"task_3/src/pa3ttc/config.py\": \"eb317d8cd33c42ff11054de57ec04697de811cae83eff276cf658170899dfffd\", \"task_3/src/pa3ttc/data/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_3/src/pa3ttc/data/discover.py\": \"dddd3685b68837d6df5e68449d30fa5a83cdf5cea07acba7c4afae05a0dcda94\", \"task_3/src/pa3ttc/data/load.py\": \"82b5792171090c7a4afd45b3438a1c4b380d1405eaaa26c1577360c38366dac4\", \"task_3/src/pa3ttc/data/splits.py\": \"bec5196e56ed23a1ca00c5da2806af9a2761b06f668acc1186aeb6e42995849c\", \"task_3/src/pa3ttc/io.py\": \"30b16c8771abb8dd4579b166d43110d37c1a6ec5d45a15f84b32d53f058bda52\", \"task_3/src/pa3ttc/live/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_3/src/pa3ttc/live/backend.py\": \"80fe7ebed8466734cc9472517b561d7de4f78d3fd409f1f95762eb3563b164f6\", \"task_3/src/pa3ttc/live/evaluate.py\": \"cb10a52d536a2750b27c0ade815bdf7a865eb00fba9298698c360b110ce3a641\", \"task_3/src/pa3ttc/live/run.py\": \"7520b278423e6a5f57573000db55658e72ff21e7532473e6606ea525530c3de7\", \"task_3/src/pa3ttc/offline/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_3/src/pa3ttc/offline/bootstrap.py\": \"6b6a89e6be592ceffbf2d872322b16c2306a595097652e56767346088be0bc45\", \"task_3/src/pa3ttc/offline/runner.py\": \"1ea6a10833efb0e9248426a1c1eb4301b2f32c667b2a5ea2494cb85059a2d26a\", \"task_3/src/pa3ttc/orchestrate.py\": \"bdd6fe356ebb8de6408872abe080f69828929514bde7d5879d8ba68aef060fb8\", \"task_3/src/pa3ttc/report/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_3/src/pa3ttc/report/figures.py\": \"f79300f97f09fb39184d6b82196c4c7aaa57e6919819412b58acf8a4349f1540\", \"task_3/src/pa3ttc/report/qualitative.py\": \"d9ec7467e5451f32c554e6930cdff9e93ae3cfed9d697d8fe469818f679eb014\", \"task_3/src/pa3ttc/select/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_3/src/pa3ttc/select/rules.py\": \"ae39b7d6d2f7b6662522f4426e9e6c9d3b2a9c810167afc3db69fbacc68e44f7\", \"task_3/src/pa3ttc/signals/__init__.py\": \"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\", \"task_3/src/pa3ttc/signals/rollout.py\": \"44c73927cf84be1ae6f05b4b1009def38fb4fe698865d6905833499bd519b6c5\", \"task_3/src/pa3ttc/signals/stage.py\": \"846407ed3b16e0474922d5c5aa5fe93b5d067f049abe45f95f5ba7e6ffde4078\", \"task_3/src/pa3ttc/signals/visconf.py\": \"1f04790328669776bdef8610b536d77dcbda348012ac05bc453c45bb02f87a4d\", \"task_3/src/pa3ttc/stages.py\": \"d0c2a64ab6192b52804fe6315929b6d833f8a69864d2a1ae1a02bb06dec502e6\", \"task_3/scripts/build_notebook.py\": \"6d22cabaf837e06f48317da2cd527ef5ed79f2017fc658785e264b97b6d4bd81\", \"task_3/tests/conftest.py\": \"2e14d7641baa9e5c288fbc6254fd5e163a7acf2a6cc276c13c5e5d4b753dff6d\", \"task_3/tests/test_discover.py\": \"438d8e15a2fdd64af9d6212adae3fb3ed8c68658f30b7d4d7626854a2ab66130\", \"task_3/tests/test_hf_backend.py\": \"c638a0687b2e66109133ef32d736987695a9ace0357dc91419cf0e86d45ee06e\", \"task_3/tests/test_orchestrate.py\": \"77c9b82fe065a8496559fed3007bc3f4797a0ead9f858d08d3d202272b7865e8\", \"task_3/tests/test_pipeline.py\": \"80e408059ec964a87e1bffa471140a1f9b6ec558674638dcaf5f8c8804977d3e\", \"task_3/tests/test_units.py\": \"2c078b70f98bd262c5e854f471949a17b9735b2fb9db7cc1d9e2c6e148fd2996\", \"task_3/README.md\": \"0c494c5c38a9a50ed5af5b851c8d18b2095064793aeb56a28e4f5299d997a408\"}")
bad = [r for r, h in EXPECTED.items()
       if hashlib.sha256(open('/kaggle/working/ATML-PA3/' + r, 'rb').read().rstrip(b'\n')).hexdigest() != h]
if bad:
    SETUP_ERROR = RuntimeError(f'files differ from the embedded sources: {bad}')
    print(SETUP_ERROR)
else:
    print(f'{len(EXPECTED)} files written and verified')

In [ ]:
# ============================ DEPENDENCIES ============================
import sys, subprocess, traceback
sys.path[:0] = [f"{CODE_DIR}/common", f"{CODE_DIR}/task_3/src"]
try:
    from pa3common.kaggle import pip_install
    import torch
    torch_before = torch.__version__
    pip_install(["transformers==5.21.0", "math-verify[antlr4_13_2]==0.9.0", "matplotlib", "pytest", "pyyaml", "scipy"])
    out = subprocess.run([sys.executable, "-c", "import torch, transformers, math_verify, pyarrow; "
                          "import importlib.metadata as m; print(torch.__version__, transformers.__version__, "
                          "m.version('math-verify'), pyarrow.__version__)"], capture_output=True, text=True)
    print("kernel torch:", torch_before, "| subprocess torch/transformers/math-verify/pyarrow:", out.stdout.strip(),
          out.stderr[-400:])
    assert out.stdout.split()[0] == torch_before, "pip changed torch — restart the kernel and rerun"
    print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
          "| CPU cores:", len(os.sched_getaffinity(0)))
except Exception as ex:  # noqa: BLE001
    traceback.print_exc()
    SETUP_ERROR = ex

In [ ]:
# ============================ DATA CHECK ============================
# Shows what is attached under /kaggle/input and which tables the pipeline will use (fails fast if none).
if SETUP_ERROR is None:
    try:
        from pa3ttc.data.discover import find_tables, listing
        print(listing([Path("/kaggle/input")], max_lines=120))
        from pa3common.kaggle import scratch_root
        _t = find_tables(DATA_ROOT, "/kaggle/input", scratch=scratch_root("pa3_task3_scratch"),
                         url=DATA_URL or None, hf_repo=DATA_HF_REPO or None)
        for _k, _v in _t.items():
            print(f"\n{_k}: folder {_v['folder']}")
            for _r in ("gen", "prob", "attn"):
                print(f"   {_r}: {_v.get(_r)}")
    except Exception as ex:  # noqa: BLE001
        print("!! DATA NOT FOUND:", ex)
        SETUP_ERROR = ex
        os.makedirs(f"{OUTPUT_ROOT}/data", exist_ok=True)       # so the results zip shows what was attached
        Path(f"{OUTPUT_ROOT}/data/input_listing.txt").write_text(
            listing([Path("/kaggle/input")], max_lines=3000) + "\n\n" + str(ex))

In [ ]:
# ============================ RUN ============================
# Errors are caught so the RESULTS and DOWNLOAD cells still run (the error is re-raised at the very end).
import traceback
RUN_ERROR = SETUP_ERROR
if RUN_ERROR is None:
    os.chdir(f"{CODE_DIR}/task_3")
    os.environ["PA3_CONFIG_OVERRIDES"] = json.dumps({"data_root": DATA_ROOT, "data_url": DATA_URL or None,
                                                     "data_hf_repo": DATA_HF_REPO or None})
    try:
        from pa3ttc import orchestrate as O
        info = O.setup_env(OUTPUT_ROOT, budget_hours=BUDGET_HOURS)
        result = O.stage_auto(run_cpu_tests=RUN_CPU_TESTS, run_live=RUN_LIVE)
        print(json.dumps(result, indent=1))
        if result["failed"]:      # make a failed stage visible: the DOWNLOAD cell zips, then raises
            RUN_ERROR = RuntimeError(f"stages failed: {result['failed']} (skipped: {result['skipped']}) — "
                                     f"see {OUTPUT_ROOT}/logs/<stage>.log and data/input_listing.txt")
            print("!!", RUN_ERROR)
    except Exception as ex:  # noqa: BLE001
        traceback.print_exc()
        RUN_ERROR = ex
else:
    print("setup failed — skipping the pipeline:", repr(RUN_ERROR))

In [ ]:
# ============================ RESULTS ============================
from pathlib import Path
from IPython.display import Markdown, Image, display
R = Path(OUTPUT_ROOT)
if (R / "status.json").exists():
    print(json.dumps(json.loads((R / "status.json").read_text()), indent=1))
for name in ("data/math_alignment.json", "data/mathvista_alignment.json"):
    if (R / name).exists():
        print(name, json.dumps(json.loads((R / name).read_text()).get("alignment"), indent=1))
T, F = R / "report_assets/tables", R / "report_assets/figures"
for t in sorted(T.glob("*.md")) if T.exists() else []:
    display(Markdown(f"**{t.name}**\n\n" + t.read_text()))
for f in sorted(F.glob("*.png")) if F.exists() else []:
    display(Image(filename=str(f)))

In [ ]:
# ============================ DOWNLOAD RESULTS ============================
# Zips everything in the outputs folder and shows a download link (also after a failure).
import os, zipfile
from pathlib import Path
from IPython.display import FileLink, display
OUT = Path(globals().get("OUTPUT_ROOT", "/kaggle/working/pa3_task3_outputs"))
ZIP = Path("/kaggle/working/results_task3.zip")
n = 0
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.rglob("*")) if OUT.exists() else []:
        if f.is_file() and "_scratch" not in f.parts and f.suffix not in (".tmp", ".part", ".zip"):
            zf.write(f, f.relative_to(OUT.parent))
            n += 1
print(f"{ZIP}: {n} files, {ZIP.stat().st_size / 2**20:.1f} MiB")
os.chdir("/kaggle/working")
display(FileLink(ZIP.name))
if globals().get("RUN_ERROR") is not None:
    raise RuntimeError("the pipeline failed earlier (see the RUN cell); partial results are in the zip") from RUN_ERROR